# Prosper Lending Network: Data Exploration and Validation

## 1. Introduction

This notebook explores the Prosper peer-to-peer lending network dataset for the research project *A Big-Data Pipeline and Explainable Graph Neural Network for Credit Risk Assessment in Uganda's Fintech*.

The dataset contains observed lending relationships between users, recorded loan outcomes, loan amounts, and other financial attributes.

### Objectives

1. Explore the structure and quality of the Prosper dataset.
2. Understand the relationships between lenders and borrowers.
3. Define a suitable credit-default prediction task.
4. Identify potential data leakage.
5. Prepare the data for a fair comparison between Logistic Regression and GraphSAGE.
6. Support future SHAP and GNNExplainer analyses.

**Research context:** The dataset represents historical US peer-to-peer lending. The applicability of the findings to Uganda's fintech sector will be evaluated separately, considering lending policies, regulation, privacy, and financial inclusion.

In [1]:

from pathlib import Path
import zipfile
import pandas as pd

# Find the research project directory
project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

zip_path = project_root / "data" / "raw" / "prosper-loans.zip"

print("Dataset exists:", zip_path.exists())
print("Dataset path:", zip_path)

# Inspect the ZIP archive without extracting it
with zipfile.ZipFile(zip_path, "r") as archive:
    print("\nFiles inside the archive:")
    for file in archive.namelist():
        print(file)


Dataset exists: True
Dataset path: C:\GroupWork\DS_research_project-maria_kisakye-\data\raw\prosper-loans.zip

Files inside the archive:
out.prosper-loans
__MACOSX/
__MACOSX/._out.prosper-loans


In [2]:

# Define provisional column names
columns = [
    "source_id",
    "target_id",
    "edge_weight",
    "timestamp",
    "amount",
    "loan_status",
    "interest_rate_1",
    "interest_rate_2",
    "credit_grade"
]

# Read only the first 10 records
with zipfile.ZipFile(zip_path, "r") as archive:
    with archive.open("out.prosper-loans") as file:
        prosper_sample = pd.read_csv(
            file,
            sep=r"\s+",
            comment="%",
            header=None,
            names=columns,
            nrows=10
        )

print("Dataset sample shape:", prosper_sample.shape)
display(prosper_sample)


Dataset sample shape: (10, 9)


,source_id,target_id,edge_weight,timestamp,amount,loan_status,interest_rate_1,interest_rate_2,credit_grade
0,20504,79275,1,1182816000,100.0,Charge-off,0.1750,0.1950,B
1,6182,59872,1,1213142400,90.0,Paid,0.1355,0.1355,B
2,10118,67702,1,1155600000,1000.0,Defaulted,0.1425,0.1425,A
3,28073,68313,1,1199318400,100.0,Charge-off,0.3299,0.3299,C
4,9896,60769,1,1181606400,50.0,Paid,0.1411,0.1411,C
5,11129,62480,1,1189987200,100.0,Paid,0.1244,0.1244,A
6,24589,72456,1,1206403200,50.0,Paid,0.2125,0.2125,C
7,52137,70310,1,1158710400,150.0,Charge-off,0.1469,0.1469,C
8,28508,69388,1,1172707200,100.0,Paid,0.1700,0.1700,D
9,2022,84015,1,1181606400,100.0,Charge-off,0.1825,0.1900,A


### 1.1 Initial Dataset Inspection

The original Prosper lending network was accessed directly from its compressed ZIP archive.

An initial sample of 10 records was loaded to examine the structure of the data without loading the entire network into memory.

The records contain identifiers representing lending relationships, timestamps, loan amounts, recorded loan statuses, interest-rate fields, and credit grades.

These attributes provide a starting point for investigating whether the dataset can support credit-default prediction using Logistic Regression and GraphSAGE.

The exact meaning and timing of the attributes must be verified before they are used in predictive modelling.

In [3]:

# Load the complete Prosper lending network
with zipfile.ZipFile(zip_path, "r") as archive:
    with archive.open("out.prosper-loans") as file:
        prosper_df = pd.read_csv(
            file,
            sep=r"\s+",
            comment="%",
            header=None,
            names=columns
        )

print("Dataset shape:", prosper_df.shape)

print("\nColumn data types:")
print(prosper_df.dtypes)

print("\nMissing values:")
print(prosper_df.isnull().sum())

print("\nFirst five records:")
display(prosper_df.head())


Dataset shape: (3394979, 9)

Column data types:
source_id            int64
target_id            int64
edge_weight          int64
timestamp            int64
amount             float64
loan_status            str
interest_rate_1    float64
interest_rate_2    float64
credit_grade           str
dtype: object

Missing values:
source_id          0
target_id          0
edge_weight        0
timestamp          0
amount             0
loan_status        0
interest_rate_1    0
interest_rate_2    0
credit_grade       0
dtype: int64

First five records:


,source_id,target_id,edge_weight,timestamp,amount,loan_status,interest_rate_1,interest_rate_2,credit_grade
0,20504,79275,1,1182816000,100.0,Charge-off,0.1750,0.1950,B
1,6182,59872,1,1213142400,90.0,Paid,0.1355,0.1355,B
2,10118,67702,1,1155600000,1000.0,Defaulted,0.1425,0.1425,A
3,28073,68313,1,1199318400,100.0,Charge-off,0.3299,0.3299,C
4,9896,60769,1,1181606400,50.0,Paid,0.1411,0.1411,C


### 1.2 Loading the Complete Prosper Dataset

After inspecting a sample of the Prosper lending network, the complete dataset was loaded into a Pandas DataFrame.

The purpose of this step was to determine the dataset's size, identify the data types of its attributes, and examine missing values.

These checks are important because data quality affects feature engineering, graph construction, and the reliability of predictive models.

The dataset contains lending relationships rather than necessarily unique loans. Further analysis is therefore required to determine the appropriate prediction unit and avoid treating multiple lending relationships associated with the same loan as independent observations.

In [4]:

# Examine the different loan-status categories
status_counts = prosper_df["loan_status"].value_counts()

print("Loan status distribution:")
display(status_counts.to_frame(name="count"))

# Count unique lenders and borrowers
unique_lenders = prosper_df["source_id"].nunique()
unique_borrowers = prosper_df["target_id"].nunique()

# Count all unique users across both columns
unique_users = pd.unique(
    pd.concat([
        prosper_df["source_id"],
        prosper_df["target_id"]
    ], ignore_index=True)
).size

print("\nUnique lenders:", unique_lenders)
print("Unique borrowers:", unique_borrowers)
print("Total unique users:", unique_users)

# Check whether the same user appears as both lender and borrower
lenders = set(prosper_df["source_id"].unique())
borrowers = set(prosper_df["target_id"].unique())

print("Users appearing in both roles:", len(lenders & borrowers))


Loan status distribution:


,count
loan_status,
Paid,1432659
Current,1116402
Charge-off,584023
Defaulted,213080
1,10875
Late,8524
3,8353
2,8213
Payoff,6699



Unique lenders: 59046
Unique borrowers: 35846
Total unique users: 89269
Users appearing in both roles: 5623


### 1.3 Loan Status Distribution and Network Participants

This stage examines the different loan-status categories and the number of unique participants in the Prosper lending network.

The analysis distinguishes between lenders, represented by source identifiers, and borrowers, represented by target identifiers. It also identifies users who appear in both roles.

Understanding the loan-status distribution is necessary before defining a binary credit-default target. In particular, loans marked as paid, defaulted, charged off, current, or late may require different treatment.

The analysis also helps establish the difference between lending records and unique borrowers, which is important for avoiding duplicated observations and designing a fair comparison between Logistic Regression and GraphSAGE.

The final target definition will be determined after examining the recorded statuses and their meanings.

In [5]:

# Check for completely identical records
exact_duplicates = prosper_df.duplicated().sum()

# Count unique lender-borrower pairs
unique_pairs = prosper_df[
    ["source_id", "target_id"]
].drop_duplicates().shape[0]

# Count lending records per borrower
records_per_borrower = prosper_df.groupby(
    "target_id"
).size()

print("Exact duplicate records:", exact_duplicates)
print("Unique lender-borrower pairs:", unique_pairs)

print("\nLending records per borrower:")
display(records_per_borrower.describe())

# Check how many borrowers have more than one recorded status
statuses_per_borrower = prosper_df.groupby(
    "target_id"
)["loan_status"].nunique()

print(
    "\nBorrowers with multiple loan-status categories:",
    (statuses_per_borrower > 1).sum()
)

# Examine repeated lender-borrower pairs
pair_counts = prosper_df.groupby(
    ["source_id", "target_id"]
).size()

print(
    "\nLender-borrower pairs appearing more than once:",
    (pair_counts > 1).sum()
)


Exact duplicate records: 0
Unique lender-borrower pairs: 3330225

Lending records per borrower:


count    35846.000000
mean        94.710121
std         93.915835
min          1.000000
25%         32.000000
50%         64.000000
75%        126.000000
max       1071.000000
dtype: float64


Borrowers with multiple loan-status categories: 3533

Lender-borrower pairs appearing more than once: 55633


### 1.4 Investigating Repeated Lending Relationships

The Prosper dataset contains 3,394,979 lending records involving 35,846 unique borrowers and 59,046 unique lenders. A total of 5,623 users appear in both roles.

The recorded outcomes include Paid, Current, Charge-off, Defaulted, and several other categories. For preliminary analysis, Paid is considered a possible non-default outcome, while Defaulted and Charge-off are possible default outcomes. These definitions require further validation.

This stage investigates exact duplicate records, repeated lender–borrower pairs, the number of lending records associated with each borrower, and whether borrowers have multiple recorded loan statuses.

These checks are necessary because individual lending records may not represent independent loans. Treating multiple relationships associated with the same loan as separate observations could distort model evaluation.

The analysis will help determine whether the dataset supports a reliable borrower-level or loan-level credit-risk prediction task before implementing Logistic Regression and GraphSAGE.

In [6]:

# Investigate whether borrower and timestamp
# could help identify individual borrowing events

grouped = prosper_df.groupby(
    ["target_id", "timestamp"]
)

group_summary = grouped.agg(
    lending_records=("source_id", "size"),
    unique_lenders=("source_id", "nunique"),
    unique_statuses=("loan_status", "nunique"),
    unique_grades=("credit_grade", "nunique"),
    unique_rate_1=("interest_rate_1", "nunique"),
    unique_rate_2=("interest_rate_2", "nunique")
)

print("Potential borrower-date groups:", len(group_summary))

print("\nLending records per borrower-date group:")
display(group_summary["lending_records"].describe())

print(
    "\nGroups containing multiple loan statuses:",
    (group_summary["unique_statuses"] > 1).sum()
)

print(
    "Groups containing multiple credit grades:",
    (group_summary["unique_grades"] > 1).sum()
)

print(
    "Groups containing multiple first interest rates:",
    (group_summary["unique_rate_1"] > 1).sum()
)

print(
    "Groups containing multiple second interest rates:",
    (group_summary["unique_rate_2"] > 1).sum()
)

print("\nExample groups with multiple lending records:")
display(
    group_summary[
        group_summary["lending_records"] > 1
    ].head(10)
)


Potential borrower-date groups: 43572

Lending records per borrower-date group:


count    43572.000000
mean        77.916529
std         75.104958
min          1.000000
25%         28.000000
50%         54.000000
75%        103.000000
max        833.000000
Name: lending_records, dtype: float64


Groups containing multiple loan statuses: 0
Groups containing multiple credit grades: 0
Groups containing multiple first interest rates: 0
Groups containing multiple second interest rates: 0

Example groups with multiple lending records:


lending_records  unique_lenders  unique_statuses  \
target_id timestamp                                                      
4         1209686400              141             140                1   
15        1201564800               36              36                1   
20        1159228800               28              28                1   
          1165881600                6               6                1   
27        1172620800              112             111                1   
30        1216339200              109             109                1   
43        1177632000              126             122                1   
53        1276819200               39              39                1   
59        1166400000               90              86                1   
63        1166054400               26              25                1   

                      unique_grades  unique_rate_1  unique_rate_2  
target_id timestamp                                                
4         1209686400              1              1              1  
15        1201564800              1              1              1  
20        1159228800              1              1              1  
          1165881600              1              1              1  
27        1172620800              1              1              1  
30        1216339200              1              1              1  
43        1177632000              1              1              1  
53        1276819200              1              1              1  
59        1166400000              1              1              1  
63        1166054400              1              1              1

### 1.5 Investigating Potential Loan Groupings

The dataset contains 3,330,225 unique lender–borrower pairs, with 55,633 pairs appearing more than once. No completely identical records were identified.

Borrowers have an average of approximately 95 lending records, and 3,533 borrowers are associated with more than one loan-status category.

These findings indicate that the lending network contains repeated borrowing activity and that individual records should not automatically be treated as independent loans.

To investigate the possibility of identifying individual borrowing events, records are grouped by borrower identifier and timestamp.

Within each group, the analysis examines the number of lending records, unique lenders, loan-status categories, credit grades and interest rates.

The purpose is to assess whether borrower–timestamp combinations provide a reasonable basis for reconstructing borrowing events. However, these combinations are not verified loan identifiers, and any grouping approach must be evaluated carefully before being used for predictive modelling.

In [7]:

# Convert Unix timestamps into readable UTC dates
prosper_df["loan_datetime"] = pd.to_datetime(
    prosper_df["timestamp"],
    unit="s",
    utc=True
)

print("Earliest recorded date:")
print(prosper_df["loan_datetime"].min())

print("\nLatest recorded date:")
print(prosper_df["loan_datetime"].max())

# Examine whether timestamps contain time-of-day information
unique_times = prosper_df[
    "loan_datetime"
].dt.strftime("%H:%M:%S").value_counts()

print("\nMost common recorded times:")
display(unique_times.head(10))

print("\nNumber of distinct times of day:", len(unique_times))

# Count distinct timestamps
print(
    "\nNumber of distinct timestamps:",
    prosper_df["timestamp"].nunique()
)

# Display a sample of readable dates
display(
    prosper_df[
        ["target_id", "timestamp", "loan_datetime"]
    ].head(10)
)


Earliest recorded date:
2005-11-15 00:00:00+00:00

Latest recorded date:
2011-09-27 00:00:00+00:00

Most common recorded times:


loan_datetime
00:00:00    3394979
Name: count, dtype: int64


Number of distinct times of day: 1

Number of distinct timestamps: 1259


,target_id,timestamp,loan_datetime
0,79275,1182816000,2007-06-26 00:00:00+00:00
1,59872,1213142400,2008-06-11 00:00:00+00:00
2,67702,1155600000,2006-08-15 00:00:00+00:00
3,68313,1199318400,2008-01-03 00:00:00+00:00
4,60769,1181606400,2007-06-12 00:00:00+00:00
5,62480,1189987200,2007-09-17 00:00:00+00:00
6,72456,1206403200,2008-03-25 00:00:00+00:00
7,70310,1158710400,2006-09-20 00:00:00+00:00
8,69388,1172707200,2007-03-01 00:00:00+00:00
9,84015,1181606400,2007-06-12 00:00:00+00:00


### 1.6 Evaluating Borrower–Timestamp Grouping

The dataset was grouped by borrower identifier and timestamp to investigate whether these combinations could represent individual borrowing events.

The analysis identified 43,572 potential borrowing-event groups, with an average of approximately 78 lending records per group.

No groups contained conflicting loan statuses, credit grades, or interest-rate values. This consistency supports the hypothesis that multiple lending records within a group may correspond to the same underlying borrowing event.

However, the dataset does not contain an explicit loan identifier. Therefore, borrower–timestamp combinations cannot yet be confirmed as unique loans.

The next analysis examines the precision and distribution of timestamps to determine whether they represent exact transaction times or calendar dates. This distinction is important for constructing reliable prediction units and preventing data leakage.

In [8]:

# Count distinct borrowing dates for each borrower
dates_per_borrower = prosper_df.groupby(
    "target_id"
)["timestamp"].nunique()

print("Borrowing dates per borrower:")
display(dates_per_borrower.describe())

print(
    "\nBorrowers with only one recorded date:",
    (dates_per_borrower == 1).sum()
)

print(
    "Borrowers with multiple recorded dates:",
    (dates_per_borrower > 1).sum()
)

print(
    "Maximum distinct borrowing dates for one borrower:",
    dates_per_borrower.max()
)

# Calculate gaps between consecutive recorded dates
borrower_dates = prosper_df[
    ["target_id", "loan_datetime"]
].drop_duplicates()

borrower_dates = borrower_dates.sort_values(
    ["target_id", "loan_datetime"]
)

borrower_dates["days_since_previous_date"] = (
    borrower_dates.groupby("target_id")[
        "loan_datetime"
    ].diff().dt.days
)

print("\nDays between consecutive borrowing dates:")
display(
    borrower_dates[
        "days_since_previous_date"
    ].dropna().describe()
)

print("\nExamples of borrowers with multiple dates:")
display(
    borrower_dates[
        borrower_dates["target_id"].isin(
            dates_per_borrower[
                dates_per_borrower > 1
            ].head(5).index
        )
    ].head(20)
)


Borrowing dates per borrower:


count    35846.000000
mean         1.215533
std          0.526301
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          8.000000
Name: timestamp, dtype: float64


Borrowers with only one recorded date: 29679
Borrowers with multiple recorded dates: 6167
Maximum distinct borrowing dates for one borrower: 8

Days between consecutive borrowing dates:


count    7726.000000
mean      578.874709
std       355.665241
min        21.000000
25%       306.000000
50%       478.000000
75%       787.750000
max      2012.000000
Name: days_since_previous_date, dtype: float64


Examples of borrowers with multiple dates:


,target_id,loan_datetime,days_since_previous_date
206489,20,2006-09-26 00:00:00+00:00,NaN
565493,20,2006-12-12 00:00:00+00:00,77.0
32851,118,2006-12-13 00:00:00+00:00,NaN
2005266,118,2011-08-03 00:00:00+00:00,1694.0
31560,130,2007-09-11 00:00:00+00:00,NaN
77946,130,2008-04-15 00:00:00+00:00,217.0
18675,157,2007-03-16 00:00:00+00:00,NaN
153208,157,2008-02-11 00:00:00+00:00,332.0
2001501,157,2009-10-15 00:00:00+00:00,612.0
2021800,157,2010-11-16 00:00:00+00:00,397.0


### 1.7 Temporal Structure and Repeat Borrowing

The Prosper dataset contains lending records dated between 15 November 2005 and 27 September 2011.

All timestamps correspond to midnight UTC, indicating that the recorded temporal information has calendar-date precision rather than exact transaction-time precision.

A total of 1,259 distinct timestamps were identified.

The earlier borrower–timestamp grouping produced 43,572 potential borrowing events. However, because the timestamps contain only dates, these groups cannot be treated as verified individual loans.

This stage investigates the number of distinct recorded borrowing dates associated with each borrower and the intervals between consecutive dates.

The analysis helps describe repeat borrowing activity and evaluate the suitability of borrower–date combinations as possible prediction observations.

Further validation is required before defining loan-level labels or constructing a temporal graph. In particular, the recorded date must be confirmed as information available at the intended prediction point.

In [9]:

# Create a temporary table of potential borrowing events
event_df = (
    prosper_df.groupby(
        ["target_id", "timestamp"],
        as_index=False
    )
    .agg(
        lending_records=("source_id", "size"),
        unique_lenders=("source_id", "nunique"),
        total_recorded_amount=("amount", "sum"),
        loan_status=("loan_status", "first"),
        interest_rate_1=("interest_rate_1", "first"),
        interest_rate_2=("interest_rate_2", "first"),
        credit_grade=("credit_grade", "first")
    )
)

print("Potential borrowing events:", len(event_df))

print("\nFinancial attribute summary:")
display(
    event_df[
        [
            "lending_records",
            "unique_lenders",
            "total_recorded_amount",
            "interest_rate_1",
            "interest_rate_2"
        ]
    ].describe()
)

print("\nCredit grade distribution:")
display(
    event_df["credit_grade"].value_counts()
)

print("\nEvent-level loan status distribution:")
display(
    event_df["loan_status"].value_counts()
)

print("\nFirst five potential borrowing events:")
display(event_df.head())


Potential borrowing events: 43572

Financial attribute summary:


,lending_records,unique_lenders,total_recorded_amount,interest_rate_1,interest_rate_2
count,43572.000000,43572.000000,43572.000000,43572.000000,43572.000000
mean,77.916529,76.730102,5932.889441,0.193673,0.195128
std,75.104958,74.525860,5072.803488,0.081768,0.081974
min,1.000000,1.000000,50.000000,0.000000,0.000000
25%,28.000000,27.000000,2511.127500,0.127500,0.129000
50%,54.000000,53.000000,4457.715000,0.180000,0.184500
75%,103.000000,102.000000,7500.990000,0.262000,0.263500
max,833.000000,833.000000,37166.710000,0.497500,0.497500



Credit grade distribution:


credit_grade
D     8796
C     7301
B     5833
E     5673
A     5587
HR    5334
AA    4906
NC     142
Name: count, dtype: int64


Event-level loan status distribution:


loan_status
Paid           20225
Current        11610
Charge-off      7878
Defaulted       3232
1                137
2                111
Late             108
3                107
Payoff            81
Repurchased       64
Cancelled         14
4+                 5
Name: count, dtype: int64


First five potential borrowing events:


,target_id,timestamp,lending_records,unique_lenders,total_recorded_amount,loan_status,interest_rate_1,interest_rate_2,credit_grade
0,4,1209686400,141,140,9918.06,Defaulted,0.1841,0.1841,B
1,15,1201564800,36,36,2988.61,Paid,0.0680,0.0680,AA
2,20,1159228800,28,28,2526.36,Paid,0.0800,0.0800,AA
3,20,1165881600,6,6,11200.00,Paid,0.0799,0.0799,AA
4,27,1172620800,112,111,9715.07,Paid,0.0890,0.0890,AA


### 1.8 Preliminary Borrowing-Event Aggregation

The repeat-borrowing analysis identified 29,679 borrowers with one recorded borrowing date and 6,167 borrowers with multiple recorded dates.

The median interval between consecutive recorded dates was 478 days, suggesting that repeated borrowing activity generally occurred over extended periods.

To investigate the available financial attributes, a temporary event-level dataset was constructed by grouping lending records using borrower identifier and timestamp.

For each group, the analysis calculates the number of lending records, number of unique lenders, and total recorded amount. It also retains the recorded loan status, credit grade and interest-rate fields.

These groups are treated as potential borrowing events rather than verified individual loans.

The resulting table will help assess the available features and outcome distributions before deciding whether a valid credit-default prediction task can be constructed.

Particular attention will be given to whether each attribute was available at the intended prediction time, since using information generated after a loan decision could introduce data leakage.

In [10]:

# Select potential borrowing events with resolved outcomes
resolved_statuses = ["Paid", "Defaulted", "Charge-off"]

resolved_events = event_df[
    event_df["loan_status"].isin(resolved_statuses)
].copy()

# Create a provisional binary target for exploration
resolved_events["default_flag"] = resolved_events[
    "loan_status"
].isin(["Defaulted", "Charge-off"]).astype(int)

print("Resolved potential borrowing events:", len(resolved_events))

print("\nBinary outcome distribution:")
display(
    resolved_events["default_flag"]
    .value_counts()
    .rename(index={0: "Non-default", 1: "Default"})
)

# Examine default proportions by credit grade
grade_summary = (
    resolved_events.groupby("credit_grade")
    .agg(
        events=("default_flag", "size"),
        defaults=("default_flag", "sum"),
        default_rate=("default_flag", "mean")
    )
    .sort_values("default_rate")
)

grade_summary["default_rate"] = (
    grade_summary["default_rate"] * 100
).round(2)

print("\nDefault proportion by credit grade (%):")
display(grade_summary)


Resolved potential borrowing events: 31335

Binary outcome distribution:


default_flag
Non-default    20225
Default        11110
Name: count, dtype: int64


Default proportion by credit grade (%):


,events,defaults,default_rate
credit_grade,,,
AA,3862,554,14.34
A,3721,862,23.17
B,4539,1411,31.09
C,5954,2094,35.17
D,5710,2172,38.04
E,3573,1693,47.38
HR,3836,2226,58.03
NC,140,98,70.00


### 1.9 Preliminary Default Labels and Credit Grades

The event-level dataset contains 43,572 potential borrowing events, with recorded loan statuses and financial attributes.

For preliminary analysis, events marked Paid are treated as non-default outcomes, while events marked Defaulted or Charge-off are treated as default outcomes. Current, Late, and other unresolved or differently defined statuses are excluded from this initial comparison.

A provisional binary target is created to examine the distribution of resolved outcomes across credit-grade categories.

The purpose is to investigate whether recorded credit grades are associated with differences in observed default proportions.

This analysis is exploratory. The borrower–date groups are not yet verified individual loans, and the timing of the credit-grade information has not been confirmed.

Before model development, the study must establish a suitable prediction point, confirm the interpretation of the available attributes, and design a temporal evaluation strategy that avoids using future information.

In [11]:

# Sort potential borrowing events chronologically
event_history = event_df.sort_values(
    ["target_id", "timestamp"]
).copy()

# Count earlier recorded events for each borrower
event_history["previous_event_count"] = (
    event_history.groupby("target_id").cumcount()
)

# Identify whether a borrower has prior recorded activity
event_history["has_previous_event"] = (
    event_history["previous_event_count"] > 0
)

print("Total potential borrowing events:", len(event_history))

print(
    "\nEvents with previous borrower activity:",
    event_history["has_previous_event"].sum()
)

print(
    "Events without previous borrower activity:",
    (~event_history["has_previous_event"]).sum()
)

print("\nPrevious event count distribution:")
display(
    event_history["previous_event_count"].describe()
)

# Examine historical activity among resolved outcomes
resolved_history = event_history[
    event_history["loan_status"].isin(
        ["Paid", "Defaulted", "Charge-off"]
    )
].copy()

print("\nResolved events with previous borrower activity:")
display(
    resolved_history["has_previous_event"].value_counts()
)

print("\nExamples of repeat borrowing histories:")
display(
    event_history[
        event_history["previous_event_count"] > 0
    ][
        [
            "target_id",
            "timestamp",
            "previous_event_count",
            "loan_status",
            "credit_grade"
        ]
    ].head(10)
)


Total potential borrowing events: 43572

Events with previous borrower activity: 7726
Events without previous borrower activity: 35846

Previous event count distribution:


count    43572.000000
mean         0.221702
std          0.532361
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          7.000000
Name: previous_event_count, dtype: float64


Resolved events with previous borrower activity:


has_previous_event
False    27891
True      3444
Name: count, dtype: int64


Examples of repeat borrowing histories:


,target_id,timestamp,previous_event_count,loan_status,credit_grade
3,20,1165881600,1,Paid,AA
13,118,1312329600,1,Current,A
16,130,1208217600,1,Paid,AA
20,157,1202688000,1,Paid,D
21,157,1255564800,2,Current,C
22,157,1289865600,3,Current,E
25,185,1193788800,1,Paid,AA
28,197,1216598400,1,Charge-off,AA
31,206,1261526400,1,Paid,A
37,276,1311724800,1,Current,D


### 1.10 Credit Grades, Default Outcomes and Historical Activity

The preliminary analysis identified 31,335 borrower–date groups with resolved repayment outcomes, comprising 20,225 Paid groups and 11,110 Defaulted or Charge-off groups.

The overall observed default proportion among these groups was approximately 35.46%.

Default proportions increased across the recorded credit-grade categories, from 14.34% for AA to 58.03% for HR. The NC category had a default proportion of 70%, although it contained relatively few observations.

These findings indicate an association between credit grades and recorded repayment outcomes. However, the results are descriptive and do not establish causal relationships or predictive performance.

The next stage investigates historical borrowing activity by counting earlier recorded borrower–date groups.

This analysis will help determine the extent to which past lending relationships may be available for constructing a temporal financial graph.

Historical activity must be distinguished from historical repayment outcomes, because a previously issued loan may not have reached its final repayment status at the time of a subsequent borrowing event.

The eventual model comparison will require a consistent prediction point, a leakage-safe temporal graph, and the same target observations and evaluation split for Logistic Regression and GraphSAGE.

In [12]:

# Count how many distinct lenders have funded
# borrowers on dates earlier than each event

# One record per lender, borrower and date
unique_funding = prosper_df[
    ["source_id", "target_id", "timestamp"]
].drop_duplicates()

# First recorded lending date for each lender
first_lender_date = (
    unique_funding.groupby("source_id")["timestamp"]
    .min()
    .rename("first_lender_date")
)

# Attach each lender's first recorded date
funding_history = unique_funding.join(
    first_lender_date,
    on="source_id"
)

# A lender has earlier recorded activity only if
# their first date precedes the current event date
funding_history["historically_active_lender"] = (
    funding_history["first_lender_date"]
    < funding_history["timestamp"]
)

# Summarize historical lender coverage per event
historical_coverage = (
    funding_history.groupby(["target_id", "timestamp"])
    .agg(
        total_unique_lenders=("source_id", "size"),
        lenders_with_prior_activity=(
            "historically_active_lender", "sum"
        )
    )
    .reset_index()
)

historical_coverage["has_historical_lender"] = (
    historical_coverage["lenders_with_prior_activity"] > 0
)

# Restrict summary to resolved events
resolved_coverage = resolved_events[
    ["target_id", "timestamp"]
].merge(
    historical_coverage,
    on=["target_id", "timestamp"],
    how="left",
    validate="one_to_one"
)

print("Resolved events:", len(resolved_coverage))

print("\nEvents with at least one historically active lender:")
display(
    resolved_coverage[
        "has_historical_lender"
    ].value_counts()
)

print("\nHistorical lender counts per resolved event:")
display(
    resolved_coverage[
        "lenders_with_prior_activity"
    ].describe()
)

print("\nPercentage of resolved events with historical lenders:")
print(
    round(
        resolved_coverage[
            "has_historical_lender"
        ].mean() * 100, 2
    ),
    "%"
)


Resolved events: 31335

Events with at least one historically active lender:


has_historical_lender
True     31231
False      104
Name: count, dtype: int64


Historical lender counts per resolved event:


count    31335.000000
mean        67.647295
std         65.980075
min          0.000000
25%         24.000000
50%         46.000000
75%         90.000000
max        831.000000
Name: lenders_with_prior_activity, dtype: float64


Percentage of resolved events with historical lenders:
99.67 %


### 1.11 Historical Borrowing and Lender Network Coverage

The historical borrowing analysis identified 7,726 potential borrowing events associated with borrowers who had at least one earlier recorded borrowing date.

Among the 31,335 events with resolved outcomes, only 3,444 had earlier recorded borrower activity.

This indicates that most resolved events belong to borrowers without previous borrowing events in the dataset.

However, a lack of previous borrower activity does not necessarily imply an absence of useful relational information.

The Prosper network contains lender–borrower relationships, and lenders may have funded other borrowers before participating in a new borrowing event.

To investigate this possibility, the analysis identifies each lender's earliest recorded lending date and measures whether the lenders associated with each borrowing event had previously appeared in the network.

This provides an initial estimate of historical lender-network coverage.

The analysis does not assume that previous loan outcomes were known at the time of the current event. It also does not establish whether historical lender relationships improve prediction accuracy.

These questions will be addressed through temporal graph construction and a controlled comparison between Logistic Regression and GraphSAGE.

In [13]:

# Convert event timestamps to calendar years
event_df["year"] = pd.to_datetime(
    event_df["timestamp"],
    unit="s",
    utc=True
).dt.year

# Count all potential borrowing events by year
events_by_year = (
    event_df.groupby("year")
    .size()
    .rename("all_events")
)

# Count resolved borrowing events by year
resolved_by_year = (
    event_df[
        event_df["loan_status"].isin(
            ["Paid", "Defaulted", "Charge-off"]
        )
    ]
    .groupby("year")
    .size()
    .rename("resolved_events")
)

# Count unique lenders active each year
prosper_df["year"] = prosper_df[
    "loan_datetime"
].dt.year

lenders_by_year = (
    prosper_df.groupby("year")["source_id"]
    .nunique()
    .rename("unique_lenders")
)

# Combine annual statistics
annual_summary = pd.concat(
    [
        events_by_year,
        resolved_by_year,
        lenders_by_year
    ],
    axis=1
).fillna(0).astype(int)

annual_summary["resolved_percentage"] = (
    annual_summary["resolved_events"]
    / annual_summary["all_events"] * 100
).round(2)

print("Annual lending activity:")
display(annual_summary)


Annual lending activity:


,all_events,resolved_events,unique_lenders,resolved_percentage
year,,,,
2005,22,22,19,100.00
2006,5947,5901,9891,99.23
2007,11479,11460,30440,99.83
2008,11565,11209,35949,96.92
2009,2047,892,9614,43.58
2010,5652,1509,14421,26.70
2011,6860,342,14979,4.99


### 1.12 Historical Lender Coverage and Temporal Distribution

The historical lender analysis identified 31,231 resolved borrowing events containing at least one lender who had appeared in the network on an earlier recorded date.

This represents approximately 99.67% of the 31,335 resolved borrowing events.

The median number of historically active lenders per resolved event was 46, while the mean was approximately 67.65.

These findings demonstrate substantial historical lender participation in the Prosper network.

Although most borrowers had no earlier recorded borrowing event, many were funded by lenders with previous network activity. This provides a possible source of relational information for GraphSAGE.

However, historical lender participation does not establish that the relationships improve default prediction. It also does not confirm that previous repayment outcomes were known at the prediction time.

The next analysis examines lending activity by year, including the number of potential borrowing events, resolved outcomes and active lenders.

This temporal analysis will help assess the feasibility of chronological evaluation and identify possible outcome-selection bias.

Before training the models, the study must also verify the meaning of the recorded timestamps and determine which financial attributes and relationships were available at the intended prediction point.

In [14]:

# Create a monthly period for each potential borrowing event
event_df["month"] = pd.to_datetime(
    event_df["timestamp"],
    unit="s",
    utc=True
).dt.to_period("M")

# Identify whether the recorded outcome is resolved
event_df["is_resolved"] = event_df[
    "loan_status"
].isin(["Paid", "Defaulted", "Charge-off"])

# Summarize monthly outcome coverage
monthly_summary = (
    event_df.groupby("month")
    .agg(
        all_events=("target_id", "size"),
        resolved_events=("is_resolved", "sum")
    )
)

monthly_summary["resolved_percentage"] = (
    monthly_summary["resolved_events"]
    / monthly_summary["all_events"] * 100
).round(2)

print("Monthly outcome coverage during 2008:")
display(
    monthly_summary.loc["2008-01":"2008-12"]
)

print("\nMonthly outcome coverage during 2009:")
display(
    monthly_summary.loc["2009-01":"2009-12"]
)


Monthly outcome coverage during 2008:


C:\Users\YOGA\AppData\Local\Temp\ipykernel_20528\3594487156.py:6: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  ).dt.to_period("M")


,all_events,resolved_events,resolved_percentage
month,,,
2008-01,1050,1050,100.00
2008-02,903,903,100.00
2008-03,1121,1121,100.00
2008-04,1305,1305,100.00
2008-05,1603,1600,99.81
2008-06,1436,1420,98.89
2008-07,1366,1352,98.98
2008-08,1202,1173,97.59
2008-09,1038,906,87.28



Monthly outcome coverage during 2009:


,all_events,resolved_events,resolved_percentage
month,,,
2009-05,13,8,61.54
2009-07,20,11,55.00
2009-08,248,110,44.35
2009-09,317,155,48.90
2009-10,458,192,41.92
2009-11,504,215,42.66
2009-12,487,201,41.27


### 1.13 Annual Lending Activity and Outcome Availability

The annual analysis showed substantial variation in the proportion of borrowing events with resolved repayment outcomes.

Between 2006 and 2008, more than 96% of potential borrowing events were marked Paid, Defaulted or Charge-off.

However, the resolved proportion declined to 43.58% in 2009, 26.70% in 2010 and 4.99% in 2011.

This pattern is consistent with possible outcome censoring, where more recently originated loans have not yet reached a final repayment outcome at the time of data collection. The precise cause cannot be confirmed without additional dataset documentation.

The low resolved-outcome coverage in later years raises concerns about selection bias if only resolved events are used for model training and evaluation.

Consequently, the 2006–2008 period is being investigated as a potential primary modelling window.

A chronological evaluation strategy is preferred over a random split because the research involves time-dependent financial relationships.

However, chronological splitting alone is insufficient to prevent all forms of data leakage. The availability dates of repayment outcomes and the timing of graph relationships must also be considered.

The next analysis examines monthly outcome coverage during 2008 and 2009 to assess whether a suitable modelling cutoff can be identified.

In [15]:

# Examine resolved outcomes by month, using the
# preliminary borrower-date event groups

monthly_default = resolved_events.copy()

monthly_default["month"] = (
    pd.to_datetime(
        monthly_default["timestamp"],
        unit="s",
        utc=True
    )
    .dt.strftime("%Y-%m")
)

monthly_default_summary = (
    monthly_default.groupby("month")
    .agg(
        resolved_events=("default_flag", "size"),
        default_events=("default_flag", "sum"),
        default_rate=("default_flag", "mean")
    )
)

monthly_default_summary["default_rate"] = (
    monthly_default_summary["default_rate"] * 100
).round(2)

print("Monthly default proportions during 2006:")
display(
    monthly_default_summary.loc["2006-01":"2006-12"]
)

print("\nMonthly default proportions during 2007:")
display(
    monthly_default_summary.loc["2007-01":"2007-12"]
)

print("\nMonthly default proportions during 2008:")
display(
    monthly_default_summary.loc["2008-01":"2008-10"]
)


Monthly default proportions during 2006:


,resolved_events,default_events,default_rate
month,,,
2006-01,8,1,12.50
2006-02,73,13,17.81
2006-03,232,70,30.17
2006-04,288,105,36.46
2006-05,446,147,32.96
2006-06,517,189,36.56
2006-07,550,216,39.27
2006-08,769,313,40.70
2006-09,615,240,39.02



Monthly default proportions during 2007:


,resolved_events,default_events,default_rate
month,,,
2007-01,1017,441,43.36
2007-02,902,406,45.01
2007-03,1160,444,38.28
2007-04,1098,445,40.53
2007-05,1070,440,41.12
2007-06,950,380,40.00
2007-07,934,354,37.90
2007-08,979,386,39.43
2007-09,758,302,39.84



Monthly default proportions during 2008:


,resolved_events,default_events,default_rate
month,,,
2008-01,1050,355,33.81
2008-02,903,294,32.56
2008-03,1121,387,34.52
2008-04,1305,444,34.02
2008-05,1600,565,35.31
2008-06,1420,451,31.76
2008-07,1352,425,31.43
2008-08,1173,365,31.12
2008-09,906,299,33.00


### 1.14 Monthly Outcome Coverage and Default Patterns

The monthly analysis revealed high resolved-outcome coverage during the early part of 2008.

From January to April 2008, all recorded borrower–date groups had resolved outcomes. Coverage remained above 97% through August before declining to 87.28% in September and 70.06% in October.

The dataset contained no recorded borrowing events during November and December 2008, with recorded activity resuming in May 2009.

During the available months of 2009, resolved-outcome coverage was substantially lower than in earlier years.

These findings suggest that the 2006 to August 2008 period may be more suitable for a primary retrospective default-classification experiment.

However, the observation period, loan-status definitions, and timing of outcome availability must still be verified before the modelling window is finalized.

The next analysis examines monthly default proportions among resolved borrower–date groups.

This will help identify changes in the observed outcome distribution over time and inform the selection of chronological training, validation and testing periods.

The analysis remains exploratory and does not establish that the proposed prediction task is free from temporal leakage.

In [16]:

# Inspect the original archive without loading it again
import zipfile

with zipfile.ZipFile(zip_path, "r") as archive:
    print("Archive contents:")
    for item in archive.infolist():
        print(item.filename, "-", item.file_size, "bytes")

    print("\nFirst 15 lines of the original data file:")

    with archive.open("out.prosper-loans") as file:
        for _ in range(15):
            line = file.readline()
            if not line:
                break
            print(line.decode("utf-8", errors="replace").rstrip())


Archive contents:
out.prosper-loans - 190005464 bytes
__MACOSX/ - 0 bytes
__MACOSX/._out.prosper-loans - 171 bytes

First 15 lines of the original data file:
% asym positive
20504 79275	1 1182816000 100.00 Charge-off 0.17500 0.19500 B
6182 59872	1 1213142400 90.00 Paid 0.13550 0.13550 B
10118 67702	1 1155600000 1000.00 Defaulted 0.14250 0.14250 A
28073 68313	1 1199318400 100.00 Charge-off 0.32990 0.32990 C
9896 60769	1 1181606400 50.00 Paid 0.14110 0.14110 C
11129 62480	1 1189987200 100.00 Paid 0.12440 0.12440 A
24589 72456	1 1206403200 50.00 Paid 0.21250 0.21250 C
52137 70310	1 1158710400 150.00 Charge-off 0.14690 0.14690 C
28508 69388	1 1172707200 100.00 Paid 0.17000 0.17000 D
2022 84015	1 1181606400 100.00 Charge-off 0.18250 0.19000 A
52062 81732	1 1175472000 50.00 Paid 0.12800 0.12800 B
2261 78049	1 1181174400 300.00 Defaulted 0.19500 0.21000 C
8005 77799	1 1206057600 50.00 Paid 0.12400 0.12400 B
32 83682	1 1172102400 75.00 Charge-off 0.18000 0.18000 B


### 1.15 Temporal Variation in Observed Default Outcomes

Monthly default proportions were examined among borrower–date groups with resolved repayment outcomes between 2006 and 2008.

During 2006, the observed default proportion generally increased, reaching approximately 42% by December. In 2007, default proportions were relatively high during the early months before declining toward the end of the year.

During January–August 2008, observed default proportions ranged from approximately 31% to 35%.

These results indicate that the distribution of recorded repayment outcomes varied over time. Such variation is important when evaluating whether credit-risk models generalize to later borrowing periods.

The analysis supports investigating a chronological model evaluation rather than relying exclusively on a random train–test split.

However, observed default proportions are calculated only among resolved outcomes and may be affected by outcome-selection bias.

Before finalizing the experimental design, the original dataset metadata and field definitions must be examined to establish the meaning of timestamps, financial attributes and repayment statuses.

The study will also distinguish retrospective default classification from a historically deployable prediction system, particularly where the timing of repayment outcomes cannot be established.

In [17]:

# Investigate monetary amounts within borrower-date groups

amount_summary = (
    prosper_df.groupby(["target_id", "timestamp"])
    .agg(
        lending_records=("source_id", "size"),
        unique_lenders=("source_id", "nunique"),
        total_amount=("amount", "sum"),
        largest_recorded_amount=("amount", "max"),
        smallest_recorded_amount=("amount", "min")
    )
    .reset_index()
)

amount_summary["repeated_lender_records"] = (
    amount_summary["lending_records"]
    - amount_summary["unique_lenders"]
)

print("Event-level monetary summary:")
display(
    amount_summary[
        [
            "total_amount",
            "largest_recorded_amount",
            "smallest_recorded_amount",
            "repeated_lender_records"
        ]
    ].describe()
)

print("\nGroups containing repeated lender records:")
print(
    (amount_summary["repeated_lender_records"] > 0).sum()
)

print("\nGroups where one record equals the total amount:")
print(
    (
        amount_summary["largest_recorded_amount"]
        == amount_summary["total_amount"]
    ).sum()
)

print("\nExamples of groups with repeated lender records:")
display(
    amount_summary[
        amount_summary["repeated_lender_records"] > 0
    ].head(10)
)


Event-level monetary summary:


,total_amount,largest_recorded_amount,smallest_recorded_amount,repeated_lender_records
count,43572.000000,43572.000000,43572.000000,43572.000000
mean,5932.889441,885.428023,84.527669,1.186427
std,5072.803488,1449.161320,568.949313,1.947314
min,50.000000,50.000000,0.010000,0.000000
25%,2511.127500,250.000000,25.000000,0.000000
50%,4457.715000,500.000000,50.000000,0.000000
75%,7500.990000,999.990000,50.000000,2.000000
max,37166.710000,25000.000000,25000.000000,26.000000



Groups containing repeated lender records:
21216

Groups where one record equals the total amount:
468

Examples of groups with repeated lender records:


,target_id,timestamp,lending_records,unique_lenders,total_amount,largest_recorded_amount,smallest_recorded_amount,repeated_lender_records
0,4,1209686400,141,140,9918.06,824.0,50.0,1
4,27,1172620800,112,111,9715.07,500.0,50.0,1
6,43,1177632000,126,122,10857.71,700.0,50.0,4
8,59,1166400000,90,86,10237.05,500.0,50.0,4
9,63,1166054400,26,25,2458.26,400.0,50.0,1
10,75,1168992000,40,39,5034.99,500.0,50.0,1
11,105,1142812800,91,85,14483.56,1000.0,50.0,6
12,118,1165968000,39,37,3457.83,400.0,50.0,2
18,138,1156464000,60,59,6021.33,500.0,50.0,1
21,157,1255564800,83,82,5000.46,500.0,25.0,1


### 1.16 Original Network File Inspection and Monetary Relationships

The original Prosper archive was inspected to examine its contents and identify any embedded metadata.

The main data file is named `out.prosper-loans` and begins with the comment `% asym positive`, indicating a directed network with positive edge weights.

The archive does not provide a complete data dictionary explaining the recorded financial attributes, interest-rate fields or timestamp semantics.

Consequently, the current column interpretations remain provisional where they have not been independently verified.

The next analysis investigates the monetary amounts associated with borrower–date groups.

For each group, the total recorded amount, minimum and maximum individual amounts, and number of repeated lender records are calculated.

These checks will help assess whether the records are consistent with multiple lender contributions toward a borrowing event.

However, the sum of recorded amounts will not be treated as a verified original loan principal without supporting documentation.

The findings will inform the eventual construction of a financial graph while preserving the distinction between observed lending relationships and inferred loan identities.

In [18]:

# Examine the distribution of individual funding amounts

print("Total lending records:", len(prosper_df))

print("\nRecords with zero amounts:")
print((prosper_df["amount"] == 0).sum())

print("\nRecords with negative amounts:")
print((prosper_df["amount"] < 0).sum())

print("\nRecords with amounts below 1:")
print((prosper_df["amount"] < 1).sum())

print("\nRecords with amounts below 25:")
print((prosper_df["amount"] < 25).sum())

print("\nMost common individual amounts:")
display(
    prosper_df["amount"].value_counts().head(15)
)

print("\nIndividual amount distribution:")
display(
    prosper_df["amount"].describe(
        percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
    )
)


Total lending records: 3394979

Records with zero amounts:
0

Records with negative amounts:
0

Records with amounts below 1:
126

Records with amounts below 25:
1566

Most common individual amounts:


amount
50.0     1465243
25.0      779262
100.0     385527
200.0      75901
75.0       70343
150.0      38111
250.0      31815
30.0       25922
60.0       22265
300.0      22056
500.0      20812
35.0       12838
40.0       11766
55.0       10414
125.0       8852
Name: count, dtype: int64


Individual amount distribution:


count    3.394979e+06
mean     7.614417e+01
std      2.032849e+02
min      1.000000e-02
1%       2.500000e+01
5%       2.500000e+01
25%      3.124000e+01
50%      5.000000e+01
75%      6.666000e+01
95%      2.000000e+02
99%      5.000000e+02
max      2.500000e+04
Name: amount, dtype: float64

### 1.17 Monetary Contributions and Repeated Lending Records

The monetary analysis examined the distribution of recorded funding amounts within borrower–date groups.

Across 43,572 potential borrowing events, the median total recorded amount was approximately 4,457.72, while the median largest individual recorded amount was 500.

A total of 21,216 groups contained more lending records than unique lenders, indicating repeated records involving the same lender within a borrower–date group.

Only 468 groups had a largest individual recorded amount equal to the total recorded amount.

These patterns are consistent with multiple lenders contributing funds toward borrowing events, although the exact interpretation of the monetary amounts remains subject to source verification.

The findings support retaining the original lender–borrower relationships alongside the aggregated event-level dataset.

The next analysis examines the distribution of individual recorded amounts and identifies potentially unusual values.

No records will be removed solely because their amounts are small. Any data-cleaning decisions will require a clear methodological justification.

The eventual GraphSAGE implementation will also need to distinguish between using the observed network structure and incorporating monetary amounts as edge attributes.

### 2.0 Preliminary Experimental Design and Modelling Assumptions

The Prosper dataset will be investigated as the primary source of observed financial relationships for comparing feature-based credit-risk classification with Graph Neural Networks.

**Prediction task:** The proposed task is retrospective classification of eventual repayment outcomes following loan funding. It is not intended to represent credit approval decisions made before funding.

**Prediction observations:** Borrower–timestamp combinations will be used as provisional borrowing-event identifiers. This grouping is supported by the consistency of recorded loan statuses, credit grades and interest rates within each group. However, the dataset does not provide verified individual loan identifiers.

**Target variable:** Paid will provisionally represent non-default, while Defaulted and Charge-off will represent default. Other statuses will be excluded from the primary binary classification experiment, subject to further validation.

**Modelling period:** The 2006–August 2008 period will be investigated as a primary modelling window because of its relatively high resolved-outcome coverage.

**Baseline model:** Logistic Regression will provide a feature-based benchmark.

**Graph model:** GraphSAGE will be investigated using observed lender–borrower relationships. The representation must account for multiple borrowing events associated with the same borrower and avoid exposing future relationships to earlier predictions.

**Fair comparison:** Both models will be evaluated using identical target observations, outcome definitions and temporal data partitions.

**Explainability:** SHAP will be used to interpret the Logistic Regression model, while GNNExplainer will be investigated for explaining GraphSAGE predictions.

**Responsible use:** The research will consider privacy, fairness, interpretability and the limitations of transferring findings from historical United States peer-to-peer lending data to Uganda's fintech sector.

**Methodological limitations:** The available network extract does not contain explicit loan identifiers, verified repayment-resolution dates or complete definitions for all financial attributes. These limitations prevent claims of fully verified loan reconstruction or historically deployable leakage-free prediction without additional evidence.

In [19]:

# Step 23: Create unique identifiers for provisional borrowing events

# Start from the existing event-level dataset
event_data = event_df.copy()

# Sort events chronologically, then by borrower ID
event_data = event_data.sort_values(
    ["timestamp", "target_id"]
).reset_index(drop=True)

# Assign one unique identifier to each borrower-date group
event_data["event_id"] = range(len(event_data))

# Verify identifier uniqueness
print("Total borrowing events:", len(event_data))

print(
    "Unique event IDs:",
    event_data["event_id"].nunique()
)

print(
    "Duplicate borrower-date combinations:",
    event_data.duplicated(
        subset=["target_id", "timestamp"]
    ).sum()
)

print(
    "Duplicate event IDs:",
    event_data["event_id"].duplicated().sum()
)

# Inspect the resulting table
display(event_data.head(10))


Total borrowing events: 43572
Unique event IDs: 43572
Duplicate borrower-date combinations: 0
Duplicate event IDs: 0


,target_id,timestamp,lending_records,unique_lenders,total_recorded_amount,loan_status,interest_rate_1,interest_rate_2,credit_grade,year,month,is_resolved,event_id
0,4578,1132012800,1,1,1000.0,Paid,0.0400,0.0400,AA,2005,2005-11,True,0
1,60169,1132272000,1,1,1000.0,Paid,0.1200,0.1700,E,2005,2005-11,True,1
2,1663,1132617600,1,1,2500.0,Paid,0.2500,0.2500,HR,2005,2005-11,True,2
3,75787,1132617600,2,2,2000.0,Paid,0.1300,0.1300,AA,2005,2005-11,True,3
4,62475,1132704000,2,2,1250.0,Paid,0.0435,0.0435,AA,2005,2005-11,True,4
5,9015,1132876800,1,1,4800.0,Paid,0.0900,0.0900,NC,2005,2005-11,True,5
6,60541,1132876800,9,6,12900.0,Paid,0.1200,0.1200,C,2005,2005-11,True,6
7,60765,1133136000,5,4,1850.0,Paid,0.1200,0.1200,B,2005,2005-11,True,7
8,67359,1133136000,5,5,4050.0,Paid,0.0800,0.0850,AA,2005,2005-11,True,8
9,74760,1133136000,6,6,5050.0,Paid,0.0800,0.0850,AA,2005,2005-11,True,9


### 2.1 Construction of Provisional Borrowing-Event Identifiers

The first stage of data preparation involved assigning unique identifiers to the borrower–date groups identified during exploratory analysis.

The original dataset contains multiple lending records associated with individual borrowers. A borrower may also appear on several dates, representing potentially different borrowing events.

To distinguish these observations, the event-level dataset was sorted chronologically by timestamp and borrower identifier.

A sequential `event_id` was then assigned to each borrower–date combination.

The identifiers will support subsequent event-level feature preparation, graph construction and model evaluation.

Validation checks were included to confirm that the resulting event identifiers are unique and that no duplicate borrower–date combinations remain.

The generated identifiers are analytical identifiers rather than verified original Prosper loan identifiers. The absence of explicit loan IDs remains a limitation of the dataset.

No original lender–borrower relationships or monetary records were modified during this step.

In [20]:

# Step 24: Link original lending records to event IDs

event_lookup = event_data[
    ["target_id", "timestamp", "event_id"]
].copy()

# Attach the provisional event ID to every original record
lending_edges = prosper_df.merge(
    event_lookup,
    on=["target_id", "timestamp"],
    how="left",
    validate="many_to_one"
)

# Validate that all original records are preserved
print("Original lending records:", len(prosper_df))
print("Linked lending records:", len(lending_edges))

print(
    "Records without event IDs:",
    lending_edges["event_id"].isna().sum()
)

print(
    "Unique linked events:",
    lending_edges["event_id"].nunique()
)

print(
    "All records preserved:",
    len(lending_edges) == len(prosper_df)
)

# Display a sample of the linked records
display(
    lending_edges[
        [
            "source_id",
            "target_id",
            "timestamp",
            "amount",
            "event_id"
        ]
    ].head(10)
)


Original lending records: 3394979
Linked lending records: 3394979
Records without event IDs: 0
Unique linked events: 43572
All records preserved: True


,source_id,target_id,timestamp,amount,event_id
0,20504,79275,1182816000,100.0,12037
1,6182,59872,1213142400,90.0,23985
2,10118,67702,1155600000,1000.0,2519
3,28073,68313,1199318400,100.0,17520
4,9896,60769,1181606400,50.0,11532
5,11129,62480,1189987200,100.0,14449
6,24589,72456,1206403200,50.0,20270
7,52137,70310,1158710400,150.0,3318
8,28508,69388,1172707200,100.0,7917
9,2022,84015,1181606400,100.0,11578


### 2.2 Linking Original Lending Records to Borrowing Events

Following the successful creation of 43,572 unique provisional borrowing-event identifiers, the original lending records were linked to their corresponding events.

The linking operation used the borrower identifier (`target_id`) and recorded timestamp as the matching keys.

A many-to-one merge was applied because multiple lending records may belong to the same borrower–date group, while each group has only one provisional event identifier.

The purpose of this step was to preserve the original observed lender–borrower relationships while establishing a connection to the event-level dataset.

Validation checks were included to compare the number of records before and after the merge, identify unmatched records and confirm the number of linked events.

Repeated lender contributions were retained rather than removed or automatically aggregated.

The resulting linked dataset will support subsequent graph construction and event-level feature preparation.

The event identifiers remain provisional because the original network extract does not contain verified loan identifiers.

In [21]:

# Step 25: Define the provisional modelling population

# Select the candidate modelling period
start_date = pd.Timestamp("2006-01-01", tz="UTC")
end_date = pd.Timestamp("2008-09-01", tz="UTC")

event_dates = pd.to_datetime(
    event_data["timestamp"],
    unit="s",
    utc=True
)

period_mask = (
    (event_dates >= start_date)
    & (event_dates < end_date)
)

candidate_events = event_data.loc[period_mask].copy()

# Keep only the provisional resolved outcome categories
resolved_statuses = ["Paid", "Defaulted", "Charge-off"]

model_events = candidate_events[
    candidate_events["loan_status"].isin(resolved_statuses)
].copy()

# Create the binary target
model_events["default_flag"] = (
    model_events["loan_status"]
    .isin(["Defaulted", "Charge-off"])
    .astype(int)
)

print("Events in candidate period:", len(candidate_events))
print("Resolved modelling events:", len(model_events))

print(
    "Excluded unresolved/other events:",
    len(candidate_events) - len(model_events)
)

print("\nTarget distribution:")
display(
    model_events["default_flag"]
    .value_counts()
    .rename(index={0: "Non-default", 1: "Default"})
)

print("\nTarget percentages:")
display(
    (
        model_events["default_flag"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
        .rename(index={0: "Non-default", 1: "Default"})
    )
)

print("\nSelected period:")
print("Earliest:", event_dates.loc[model_events.index].min())
print("Latest:", event_dates.loc[model_events.index].max())

print("\nUnique modelling event IDs:")
print(model_events["event_id"].nunique())


Events in candidate period: 27412
Resolved modelling events: 27285
Excluded unresolved/other events: 127

Target distribution:


default_flag
Non-default    17151
Default        10134
Name: count, dtype: int64


Target percentages:


default_flag
Non-default    62.86
Default        37.14
Name: proportion, dtype: float64


Selected period:
Earliest: 2006-01-10 00:00:00+00:00
Latest: 2008-08-29 00:00:00+00:00

Unique modelling event IDs:
27285


### 2.3 Definition of the Provisional Modelling Population

The modelling population was provisionally restricted to borrowing events recorded between 1 January 2006 and 31 August 2008.

This period was selected for investigation because earlier exploratory analysis showed relatively high coverage of resolved repayment outcomes.

Events were retained when their recorded repayment status was Paid, Defaulted or Charge-off.

A binary target variable named `default_flag` was defined as follows:

- `0`: Paid
- `1`: Defaulted or Charge-off

Events with other repayment statuses were excluded from the primary binary classification population.

The original lending-record dataset was preserved separately so that historical financial relationships remain available for subsequent graph construction.

The number of eligible events, excluded events and class proportions will be examined before selecting chronological training, validation and testing periods.

This modelling population remains provisional. In particular, the absence of verified repayment-resolution dates limits the ability to reconstruct exactly which outcomes would have been known at a historical prediction date.

The analysis is therefore framed as retrospective repayment-outcome classification rather than a verified historical credit-approval simulation.

In [22]:

# Step 26: Examine provisional chronological partitions

split_data = model_events.copy()

split_data["event_date"] = pd.to_datetime(
    split_data["timestamp"],
    unit="s",
    utc=True
)

train_end = pd.Timestamp("2008-01-01", tz="UTC")
validation_end = pd.Timestamp("2008-05-01", tz="UTC")

split_data["partition"] = "test"

split_data.loc[
    split_data["event_date"] < train_end,
    "partition"
] = "train"

split_data.loc[
    (split_data["event_date"] >= train_end)
    & (split_data["event_date"] < validation_end),
    "partition"
] = "validation"

# Count events and outcomes in each partition
split_summary = (
    split_data.groupby("partition")
    .agg(
        events=("event_id", "size"),
        defaults=("default_flag", "sum"),
        default_rate=("default_flag", "mean"),
        first_date=("event_date", "min"),
        last_date=("event_date", "max")
    )
)

split_summary["non_defaults"] = (
    split_summary["events"] - split_summary["defaults"]
)

split_summary["default_rate"] = (
    split_summary["default_rate"] * 100
).round(2)

split_summary = split_summary[
    [
        "events",
        "non_defaults",
        "defaults",
        "default_rate",
        "first_date",
        "last_date"
    ]
]

display(
    split_summary.reindex(
        ["train", "validation", "test"]
    )
)

print("\nTotal partitioned events:", len(split_data))

print(
    "Unique partitioned event IDs:",
    split_data["event_id"].nunique()
)

print(
    "Missing partition assignments:",
    split_data["partition"].isna().sum()
)


,events,non_defaults,defaults,default_rate,first_date,last_date
partition,,,,,,
train,17361,10513,6848,39.44,2006-01-10 00:00:00+00:00,2007-12-31 00:00:00+00:00
validation,4379,2899,1480,33.80,2008-01-02 00:00:00+00:00,2008-04-30 00:00:00+00:00
test,5545,3739,1806,32.57,2008-05-01 00:00:00+00:00,2008-08-29 00:00:00+00:00



Total partitioned events: 27285
Unique partitioned event IDs: 27285
Missing partition assignments: 0


### 2.4 Provisional Chronological Data Partitioning

The selected modelling population contained 27,285 provisional borrowing events recorded between January 2006 and August 2008.

Of these events, 17,151 were labelled non-default and 10,134 were labelled default, corresponding to approximately 62.86% and 37.14%, respectively.

A provisional chronological partitioning strategy was defined to evaluate model performance on later borrowing events.

The proposed partitions are:

- Training: January 2006 to December 2007.
- Validation: January to April 2008.
- Testing: May to August 2008.

Chronological partitioning was preferred over random splitting because the dataset contains time-dependent financial relationships.

The same partitions will be used for Logistic Regression and GraphSAGE to support a consistent comparison.

The number of observations and class distribution in each partition will be examined before model development.

Although chronological partitioning reduces some forms of temporal leakage, it does not establish that historical repayment outcomes were available at the simulated training cutoff.

Consequently, the experiment will be described as a retrospective temporal evaluation rather than a fully verified historical deployment simulation.

Further controls will be required to ensure that future lending relationships and repayment outcomes are not used as predictive graph features.

In [23]:

# Step 27: Investigate borrower overlap across partitions

# Count unique borrowers in each partition
borrowers_by_partition = (
    split_data.groupby("partition")["target_id"]
    .nunique()
)

print("Unique borrowers by partition:")
display(
    borrowers_by_partition.reindex(
        ["train", "validation", "test"]
    )
)

# Create sets of borrower IDs
train_borrowers = set(
    split_data.loc[
        split_data["partition"] == "train",
        "target_id"
    ]
)

validation_borrowers = set(
    split_data.loc[
        split_data["partition"] == "validation",
        "target_id"
    ]
)

test_borrowers = set(
    split_data.loc[
        split_data["partition"] == "test",
        "target_id"
    ]
)

print("\nBorrower overlap:")
print(
    "Train and validation:",
    len(train_borrowers & validation_borrowers)
)

print(
    "Train and test:",
    len(train_borrowers & test_borrowers)
)

print(
    "Validation and test:",
    len(validation_borrowers & test_borrowers)
)

print(
    "Present in all three:",
    len(
        train_borrowers
        & validation_borrowers
        & test_borrowers
    )
)

# Classify test events by borrower history
test_events = split_data[
    split_data["partition"] == "test"
].copy()

earlier_borrowers = (
    train_borrowers | validation_borrowers
)

test_events["borrower_group"] = (
    test_events["target_id"].isin(earlier_borrowers)
    .map({
        True: "Returning borrower",
        False: "New borrower"
    })
)

print("\nTest events by borrower group:")
display(
    test_events.groupby("borrower_group")
    .agg(
        events=("event_id", "size"),
        defaults=("default_flag", "sum"),
        default_rate=("default_flag", "mean")
    )
    .assign(
        default_rate=lambda df: (
            df["default_rate"] * 100
        ).round(2)
    )
)


Unique borrowers by partition:


partition
train         16606
validation     4375
test           5533
Name: target_id, dtype: int64


Borrower overlap:
Train and validation: 895
Train and test: 578
Validation and test: 57
Present in all three: 11

Test events by borrower group:


,events,defaults,default_rate
borrower_group,,,
New borrower,4920,1638,33.29
Returning borrower,625,168,26.88


### 2.5 Chronological Partition Validation and Borrower Overlap

The provisional modelling population was divided into three chronological partitions.

The training partition contained 17,361 events, the validation partition contained 4,379 events, and the testing partition contained 5,545 events.

The observed default proportions were 39.44%, 33.80% and 32.57%, respectively.

These results indicate a decline in the proportion of default-labelled events across the selected periods.

All 27,285 modelling events received a partition assignment, and the number of unique event identifiers matched the total number of observations.

The next analysis examines borrower overlap between the partitions.

Because individual borrowers may have multiple borrowing events, the same borrower can potentially appear in more than one chronological partition.

Such overlap is not automatically leakage, provided that only information available before the relevant prediction point is used.

Borrower overlap will be measured to distinguish returning borrowers from borrowers without earlier eligible events in the selected modelling population.

This distinction will support later evaluation of model performance across borrower groups.

The analysis will not assume that earlier repayment outcomes were historically available unless their availability can be established.

In [24]:

# Step 28: Inspect candidate modelling features

print("Modelling dataset shape:")
print(split_data.shape)

print("\nAvailable columns:")
print(split_data.columns.tolist())

print("\nMissing values by column:")
display(
    split_data.isna().sum().sort_values(ascending=False)
)

print("\nCandidate numerical feature summary:")
candidate_numeric = [
    "total_recorded_amount",
    "unique_lenders",
    "lending_records",
    "interest_rate_1",
    "interest_rate_2"
]

display(
    split_data[candidate_numeric].describe()
)

print("\nCredit grade distribution:")
display(
    split_data["credit_grade"].value_counts()
)

print("\nCredit grades by partition:")
display(
    pd.crosstab(
        split_data["partition"],
        split_data["credit_grade"]
    ).reindex(
        ["train", "validation", "test"]
    )
)


Modelling dataset shape:
(27285, 16)

Available columns:
['target_id', 'timestamp', 'lending_records', 'unique_lenders', 'total_recorded_amount', 'loan_status', 'interest_rate_1', 'interest_rate_2', 'credit_grade', 'year', 'month', 'is_resolved', 'event_id', 'default_flag', 'event_date', 'partition']

Missing values by column:


target_id                0
timestamp                0
lending_records          0
unique_lenders           0
total_recorded_amount    0
loan_status              0
interest_rate_1          0
interest_rate_2          0
credit_grade             0
year                     0
month                    0
is_resolved              0
event_id                 0
default_flag             0
event_date               0
partition                0
dtype: int64


Candidate numerical feature summary:


,total_recorded_amount,unique_lenders,lending_records,interest_rate_1,interest_rate_2
count,27285.000000,27285.000000,27285.000000,27285.000000,27285.000000
mean,6191.356471,67.212571,68.475829,0.180018,0.182308
std,5537.801553,62.413680,63.238855,0.072614,0.073398
min,50.000000,1.000000,1.000000,0.000000,0.000000
25%,2526.830000,24.000000,25.000000,0.125000,0.126000
50%,4510.820000,47.000000,48.000000,0.169500,0.170000
75%,7882.330000,90.000000,92.000000,0.230000,0.233300
max,35776.270000,458.000000,459.000000,0.497500,0.497500



Credit grade distribution:


credit_grade
C     5317
D     4876
B     4067
HR    3381
AA    3266
E     3160
A     3079
NC     139
Name: count, dtype: int64


Credit grades by partition:


credit_grade,A,AA,B,C,D,E,HR,NC
partition,,,,,,,,
train,1786,1810,2292,3181,3113,2373,2667,139
validation,583,675,824,902,748,360,287,0
test,710,781,951,1234,1015,427,427,0


### 2.6 Borrower Overlap and Candidate Feature Assessment

The borrower-overlap analysis showed that some borrowers appeared in multiple chronological partitions.

There were 895 borrowers shared between training and validation, 578 between training and testing, and 57 between validation and testing. Eleven borrowers appeared in all three partitions.

Among the 5,545 test events, 4,920 were associated with borrowers who had no earlier eligible event in the training or validation partitions. The remaining 625 events belonged to returning borrowers under this definition.

The observed default proportions were 33.29% for new-borrower events and 26.88% for returning-borrower events.

These differences are descriptive and do not establish a causal relationship between previous borrowing and repayment performance.

The findings motivate a secondary evaluation comparing model performance for new and returning borrowers.

The next stage examines the available event-level features, their completeness and their distributions across chronological partitions.

Feature selection will distinguish between identifiers, repayment outcomes, candidate financial predictors and relational information.

Repayment statuses and target labels will not be used as predictive inputs.

Financial variables whose definitions or timing cannot be verified will be excluded from the primary feature set or examined only in clearly labelled sensitivity analyses.

This approach aims to support a fair comparison between Logistic Regression and GraphSAGE while acknowledging the limitations of the historical Prosper network extract.

In [25]:

# Step 29: Prepare the initial shared feature set

numeric_features = [
    "total_recorded_amount",
    "unique_lenders",
    "lending_records"
]

categorical_features = [
    "credit_grade"
]

feature_columns = numeric_features + categorical_features

# Separate the observations by chronological partition
train_data = split_data[
    split_data["partition"] == "train"
].copy()

validation_data = split_data[
    split_data["partition"] == "validation"
].copy()

test_data = split_data[
    split_data["partition"] == "test"
].copy()

# Prepare predictors and target labels
X_train = train_data[feature_columns].copy()
X_validation = validation_data[feature_columns].copy()
X_test = test_data[feature_columns].copy()

y_train = train_data["default_flag"].copy()
y_validation = validation_data["default_flag"].copy()
y_test = test_data["default_flag"].copy()

print("Selected features:", feature_columns)

print("\nFeature matrix shapes:")
print("Training:", X_train.shape)
print("Validation:", X_validation.shape)
print("Testing:", X_test.shape)

print("\nTarget vector shapes:")
print("Training:", y_train.shape)
print("Validation:", y_validation.shape)
print("Testing:", y_test.shape)

print("\nTraining feature sample:")
display(X_train.head())

print("\nTarget distributions:")
print("Training:", y_train.value_counts().to_dict())
print("Validation:", y_validation.value_counts().to_dict())
print("Testing:", y_test.value_counts().to_dict())


Selected features: ['total_recorded_amount', 'unique_lenders', 'lending_records', 'credit_grade']

Feature matrix shapes:
Training: (17361, 4)
Validation: (4379, 4)
Testing: (5545, 4)

Target vector shapes:
Training: (17361,)
Validation: (4379,)
Testing: (5545,)

Training feature sample:


,total_recorded_amount,unique_lenders,lending_records,credit_grade
22,3000.00,1,1,AA
23,2900.00,5,6,AA
24,10225.01,8,11,C
25,2002.00,6,6,AA
26,7950.00,9,11,AA



Target distributions:
Training: {0: 10513, 1: 6848}
Validation: {0: 2899, 1: 1480}
Testing: {0: 3739, 1: 1806}


### 2.7 Initial Feature Selection for the Baseline Model

The feature audit confirmed that all 27,285 provisional modelling observations contained complete values for the available event-level attributes.

Differences in credit-grade composition were observed across the chronological partitions. In particular, the proportion of HR-rated events decreased in the later periods, while several higher credit-grade categories became more common.

These changes suggest temporal variation in the observed borrower population and may contribute to differences in repayment outcomes across the partitions.

For the initial post-funding classification experiment, four candidate predictors were selected:

- Total recorded funding amount
- Number of unique lenders
- Number of lending records
- Recorded credit grade

The two interest-rate fields were excluded pending verification of their definitions and timing.

Repayment statuses, target labels and identifiers were excluded from the predictive feature matrices.

The same initial feature set is intended for both Logistic Regression and GraphSAGE. This will allow the graph model's additional relational information to be evaluated beyond simple event-level funding summaries.

Training, validation and testing feature matrices were prepared using the previously established chronological partitions.

Preprocessing transformations will be fitted exclusively on training observations to reduce information leakage.

The selected variables remain provisional post-funding predictors. Their use does not imply that they would have been available during pre-loan credit approval.

In [26]:

# Step 30: Build the Logistic Regression pipeline

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Numerical preprocessing
numeric_transformer = StandardScaler()

# Categorical preprocessing
categorical_transformer = OneHotEncoder(
    handle_unknown="ignore"
)

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            numeric_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ]
)

# Build the complete model pipeline
lr_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

print("Logistic Regression pipeline created successfully.")

display(lr_pipeline)


Logistic Regression pipeline created successfully.


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, 

### 2.8 Logistic Regression Preprocessing Pipeline

Following the preparation of the modelling feature matrices, a preprocessing pipeline was defined for the Logistic Regression baseline.

The initial feature set consists of three numerical predictors and one categorical predictor.

Numerical variables are standardized using `StandardScaler` to place them on comparable scales.

The categorical credit-grade variable is transformed using `OneHotEncoder`, allowing Logistic Regression to use credit-grade information without imposing an artificial numerical ordering.

A `ColumnTransformer` combines the numerical and categorical preprocessing operations.

These transformations are incorporated into a scikit-learn `Pipeline` alongside the Logistic Regression classifier.

The pipeline will be fitted using only the chronological training partition. This ensures that preprocessing parameters are learned without accessing validation or testing observations.

The initial classifier uses default class weights, a maximum of 1,000 iterations and a fixed random state of 42.

Class weighting and classification-threshold adjustments may be investigated later using the validation partition.

The resulting baseline will provide a reference for evaluating whether GraphSAGE offers additional predictive value from observed lender–borrower relationships.

In [27]:

# Step 31: Train and validate Logistic Regression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

# Fit preprocessing and classifier on training data only
lr_pipeline.fit(X_train, y_train)

# Generate validation predictions
y_val_pred = lr_pipeline.predict(X_validation)

# Probability of the default class (1)
y_val_proba = lr_pipeline.predict_proba(X_validation)[:, 1]

# Calculate validation metrics
validation_metrics = {
    "Accuracy": accuracy_score(y_validation, y_val_pred),
    "Precision (Default)": precision_score(
        y_validation, y_val_pred, zero_division=0
    ),
    "Recall (Default)": recall_score(
        y_validation, y_val_pred, zero_division=0
    ),
    "F1-score (Default)": f1_score(
        y_validation, y_val_pred, zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_validation, y_val_proba
    ),
    "PR-AUC (Average Precision)": average_precision_score(
        y_validation, y_val_proba
    )
}

print("LOGISTIC REGRESSION - VALIDATION RESULTS")
print("-" * 45)

for metric, value in validation_metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_validation, y_val_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_validation,
        y_val_pred,
        target_names=["Non-default", "Default"],
        zero_division=0
    )
)


LOGISTIC REGRESSION - VALIDATION RESULTS
---------------------------------------------
Accuracy: 0.6664
Precision (Default): 0.5168
Recall (Default): 0.1973
F1-score (Default): 0.2856
ROC-AUC: 0.6516
PR-AUC (Average Precision): 0.4729

Confusion Matrix:
[[2626  273]
 [1188  292]]

Classification Report:
              precision    recall  f1-score   support

 Non-default       0.69      0.91      0.78      2899
     Default       0.52      0.20      0.29      1480

    accuracy                           0.67      4379
   macro avg       0.60      0.55      0.53      4379
weighted avg       0.63      0.67      0.61      4379



### 2.9 Logistic Regression Training and Validation

The Logistic Regression baseline was configured using a scikit-learn pipeline containing numerical standardization, categorical encoding and binary classification.

The model is trained using 17,361 borrowing-event observations from the chronological training partition.

Preprocessing parameters are fitted exclusively on the training data to avoid introducing information from later evaluation partitions.

The trained model is evaluated using 4,379 observations from the validation partition.

Predicted class labels and default probabilities are generated for the validation observations.

Performance is assessed using accuracy, default-class precision, recall, F1-score, ROC-AUC and average precision.

A confusion matrix and classification report are also generated to examine the types of classification errors made by the model.

The initial evaluation uses the default classification threshold of 0.5.

The test partition remains untouched during this stage. Model adjustments and threshold selection will be based on validation results before final held-out evaluation.

The experiment is interpreted as retrospective post-funding repayment-outcome classification, subject to the previously documented limitations concerning outcome timing and provisional event reconstruction.

In [28]:

# Step 32: Evaluate classification thresholds

import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

thresholds = np.arange(0.20, 0.61, 0.05)

threshold_results = []

for threshold in thresholds:

    # Convert predicted probabilities into class labels
    predictions = (
        y_val_proba >= threshold
    ).astype(int)

    threshold_results.append({
        "threshold": round(float(threshold), 2),

        "accuracy": accuracy_score(
            y_validation, predictions
        ),

        "precision": precision_score(
            y_validation, predictions,
            zero_division=0
        ),

        "recall": recall_score(
            y_validation, predictions,
            zero_division=0
        ),

        "f1_score": f1_score(
            y_validation, predictions,
            zero_division=0
        )
    })

threshold_df = pd.DataFrame(threshold_results)

print("VALIDATION THRESHOLD COMPARISON")
display(
    threshold_df.round(4)
)

# Select the best threshold by validation F1-score
best_row = threshold_df.loc[
    threshold_df["f1_score"].idxmax()
]

best_threshold = float(best_row["threshold"])

print("\nBest threshold by validation F1-score:")
print(best_threshold)

print("\nMetrics at selected threshold:")
display(best_row.to_frame(name="value"))


VALIDATION THRESHOLD COMPARISON


,threshold,accuracy,precision,recall,f1_score
0,0.20,0.4764,0.3829,0.8980,0.5369
1,0.25,0.5111,0.3938,0.8277,0.5337
2,0.30,0.5497,0.4081,0.7385,0.5257
3,0.35,0.6072,0.4402,0.5966,0.5066
4,0.40,0.6497,0.4815,0.4750,0.4782
5,0.45,0.6634,0.5029,0.3500,0.4127
6,0.50,0.6664,0.5168,0.1973,0.2856
7,0.55,0.6730,0.5574,0.1574,0.2455
8,0.60,0.6721,0.5705,0.1203,0.1987



Best threshold by validation F1-score:
0.2

Metrics at selected threshold:


,value
threshold,0.200000
accuracy,0.476364
precision,0.382887
recall,0.897973
f1_score,0.536861


### 2.10 Logistic Regression Validation Results and Threshold Analysis

The initial Logistic Regression baseline was evaluated on 4,379 validation observations using a classification threshold of 0.5.

The model achieved an accuracy of 66.64%, default-class precision of 51.68%, default recall of 19.73%, and default F1-score of 28.56%.

The ROC-AUC was 0.6516, while average precision was 0.4729.

The confusion matrix showed that the model correctly identified 292 default events but failed to identify 1,188 other default events.

These results indicate that the model has modest discriminatory ability but performs poorly at identifying defaults when using the default classification threshold.

The validation accuracy was also only slightly higher than that of a majority-class classifier that predicts every event as non-default.

To investigate the effect of the classification threshold, additional predictions will be evaluated using thresholds between 0.20 and 0.60.

For each threshold, accuracy, default precision, recall and F1-score will be calculated.

The threshold producing the highest validation F1-score among the evaluated candidates will be selected provisionally.

Threshold selection will use only validation observations. The held-out testing partition will remain untouched until the modelling and selection procedures have been finalized.

This analysis will help distinguish changes in classification performance from changes in the underlying discriminatory ability of the model.

In [29]:

# Step 33: Evaluate the selected validation threshold

from sklearn.metrics import (
    confusion_matrix,
    classification_report
)

# Apply the selected threshold
y_val_selected = (
    y_val_proba >= best_threshold
).astype(int)

# Confusion matrix
cm_selected = confusion_matrix(
    y_validation,
    y_val_selected,
    labels=[0, 1]
)

tn, fp, fn, tp = cm_selected.ravel()

print(f"Selected threshold: {best_threshold:.2f}")

print("\nConfusion matrix:")
print(cm_selected)

print("\nClassification counts:")
print("True negatives:", tn)
print("False positives:", fp)
print("False negatives:", fn)
print("True positives:", tp)

print("\nClassification report:")
print(
    classification_report(
        y_validation,
        y_val_selected,
        target_names=["Non-default", "Default"],
        zero_division=0
    )
)

print("\nPercentage of events flagged as default:")
print(
    f"{y_val_selected.mean() * 100:.2f}%"
)


Selected threshold: 0.20

Confusion matrix:
[[ 757 2142]
 [ 151 1329]]

Classification counts:
True negatives: 757
False positives: 2142
False negatives: 151
True positives: 1329

Classification report:
              precision    recall  f1-score   support

 Non-default       0.83      0.26      0.40      2899
     Default       0.38      0.90      0.54      1480

    accuracy                           0.48      4379
   macro avg       0.61      0.58      0.47      4379
weighted avg       0.68      0.48      0.44      4379


Percentage of events flagged as default:
79.26%


### 2.11 Classification Threshold Selection

The Logistic Regression baseline was evaluated using classification thresholds ranging from 0.20 to 0.60.

The results demonstrated a substantial trade-off between default-class precision and recall.

At the default threshold of 0.50, the model achieved 51.68% precision, 19.73% recall and an F1-score of 28.56%.

Reducing the threshold to 0.20 increased default recall to 89.80% and the F1-score to 53.69%, while precision decreased to 38.29%.

The overall accuracy also decreased from 66.64% to 47.64%.

The threshold of 0.20 produced the highest validation F1-score among the evaluated candidate thresholds and was therefore selected provisionally.

This threshold should not be interpreted as an economically optimal lending threshold because the analysis does not include the financial costs of classification errors.

The ROC-AUC and average precision remain unchanged by threshold adjustments because they are calculated from predicted probabilities rather than thresholded class labels.

The selected threshold will be used to examine the confusion matrix and classification errors on the validation partition.

The held-out test partition remains unused during threshold selection.

For the eventual GraphSAGE comparison, the same validation-based threshold-selection procedure will be applied to both models.

In [30]:

# Step 34: Prepare unique lender-event graph connections

# Preserve the original funding records
graph_edges = (
    lending_edges[
        ["source_id", "event_id", "timestamp"]
    ]
    .drop_duplicates(
        subset=["source_id", "event_id"]
    )
    .copy()
)

# Rename columns for clarity
graph_edges = graph_edges.rename(
    columns={
        "source_id": "lender_id"
    }
)

print("Original lending records:", len(lending_edges))
print("Unique lender-event edges:", len(graph_edges))

print(
    "Repeated records represented by existing edges:",
    len(lending_edges) - len(graph_edges)
)

print(
    "\nDuplicate lender-event pairs:",
    graph_edges.duplicated(
        subset=["lender_id", "event_id"]
    ).sum()
)

print(
    "\nUnique lenders:",
    graph_edges["lender_id"].nunique()
)

print(
    "Unique connected events:",
    graph_edges["event_id"].nunique()
)

print("\nSample graph edges:")
display(graph_edges.head(10))


Original lending records: 3394979
Unique lender-event edges: 3343284
Repeated records represented by existing edges: 51695

Duplicate lender-event pairs: 0

Unique lenders: 59046
Unique connected events: 43572

Sample graph edges:


,lender_id,event_id,timestamp
0,20504,12037,1182816000
1,6182,23985,1213142400
2,10118,2519,1155600000
3,28073,17520,1199318400
4,9896,11532,1181606400
5,11129,14449,1189987200
6,24589,20270,1206403200
7,52137,3318,1158710400
8,28508,7917,1172707200
9,2022,11578,1181606400


### 3.0 Graph Construction and GraphSAGE Preparation

The next stage of the research prepares the observed financial relationships for Graph Neural Network modelling.

An event-aware bipartite graph is proposed because the same borrower may have multiple borrowing events with different repayment outcomes.

The graph will contain lender nodes and provisional borrowing-event nodes.

Each observed lender–event connection will be derived from an original Prosper lending record.

For the initial unweighted graph, repeated records involving the same lender and borrowing event will be represented by a single connection.

The original lending-record dataset will remain unchanged so that monetary contributions and repeated records are preserved for potential additional experiments.

The resulting graph structure will allow GraphSAGE to investigate relationships between borrowing events through shared lenders.

Repayment statuses and default labels will not be used as graph input features.

Future lending relationships must be excluded from earlier prediction snapshots to prevent temporal information leakage.

The initial implementation will use unweighted connections. Monetary edge attributes may be investigated separately.

The graph representation and prediction timing remain subject to the previously documented limitations of the original Prosper dataset.

In [31]:

# Step 35: Create consecutive graph node indices

import pandas as pd

# Create lender node mapping
unique_lenders = sorted(
    graph_edges["lender_id"].unique()
)

lender_mapping = pd.DataFrame({
    "lender_id": unique_lenders,
    "lender_node_index": range(len(unique_lenders))
})

# Create event node mapping
event_mapping = event_data[
    ["event_id", "target_id", "timestamp"]
].copy()

event_mapping["event_node_index"] = (
    event_mapping["event_id"]
)

# Attach lender node indices to graph edges
indexed_graph_edges = graph_edges.merge(
    lender_mapping,
    on="lender_id",
    how="left",
    validate="many_to_one"
)

# Attach event node indices
indexed_graph_edges = indexed_graph_edges.merge(
    event_mapping[
        ["event_id", "event_node_index"]
    ],
    on="event_id",
    how="left",
    validate="many_to_one"
)

print("Total lender nodes:", len(lender_mapping))
print("Total event nodes:", len(event_mapping))

print("\nIndexed graph edges:", len(indexed_graph_edges))

print(
    "Missing lender indices:",
    indexed_graph_edges["lender_node_index"].isna().sum()
)

print(
    "Missing event indices:",
    indexed_graph_edges["event_node_index"].isna().sum()
)

print(
    "Unique lender indices:",
    lender_mapping["lender_node_index"].nunique()
)

print(
    "Unique event indices:",
    event_mapping["event_node_index"].nunique()
)

print("\nSample indexed graph edges:")
display(
    indexed_graph_edges[
        [
            "lender_id",
            "lender_node_index",
            "event_id",
            "event_node_index",
            "timestamp"
        ]
    ].head(10)
)


Total lender nodes: 59046
Total event nodes: 43572

Indexed graph edges: 3343284
Missing lender indices: 0
Missing event indices: 0
Unique lender indices: 59046
Unique event indices: 43572

Sample indexed graph edges:


,lender_id,lender_node_index,event_id,event_node_index,timestamp
0,20504,20504,12037,12037,1182816000
1,6182,6182,23985,23985,1213142400
2,10118,10118,2519,2519,1155600000
3,28073,28073,17520,17520,1199318400
4,9896,9896,11532,11532,1181606400
5,11129,11129,14449,14449,1189987200
6,24589,24589,20270,20270,1206403200
7,52137,52137,3318,3318,1158710400
8,28508,28508,7917,7917,1172707200
9,2022,2022,11578,11578,1181606400


### 3.1 Construction of Graph Node Identifiers

The graph-preparation stage produced 3,343,284 unique lender–event connections from 3,394,979 original lending records.

Repeated lender–event records were represented by single connections for the initial unweighted graph, while the original transaction records were preserved separately.

The resulting network contains 59,046 unique lenders and 43,572 provisional borrowing events.

To prepare the network for Graph Neural Network implementation, consecutive integer node indices were assigned to lenders and borrowing events.

Original lender identifiers were mapped to consecutive indices, while the existing provisional event identifiers were retained as event-node indices.

Separate node-index spaces were maintained because lenders and borrowing events represent different node types.

The node mappings were merged with the graph-edge table to create an indexed representation of the observed financial relationships.

Validation checks were included to identify missing node indices and confirm the number of nodes and edges.

The indexed graph will support subsequent construction of a bipartite graph representation.

The full graph will not automatically be used for historical predictions. Temporal restrictions will be required to prevent future lending relationships from influencing earlier borrowing-event predictions.

The eventual GraphSAGE implementation must also account for differences between lender and event node attributes.

In [32]:

# Step 36: Examine graph connections by chronological period

graph_timeline = indexed_graph_edges.copy()

graph_timeline["edge_date"] = pd.to_datetime(
    graph_timeline["timestamp"],
    unit="s",
    utc=True
)

def assign_graph_period(date):
    if date < pd.Timestamp("2006-01-01", tz="UTC"):
        return "pre_2006"
    elif date < pd.Timestamp("2008-01-01", tz="UTC"):
        return "train_period"
    elif date < pd.Timestamp("2008-05-01", tz="UTC"):
        return "validation_period"
    elif date < pd.Timestamp("2008-09-01", tz="UTC"):
        return "test_period"
    else:
        return "post_test"

graph_timeline["graph_period"] = (
    graph_timeline["edge_date"].apply(assign_graph_period)
)

graph_period_summary = (
    graph_timeline.groupby("graph_period")
    .agg(
        edges=("event_id", "size"),
        unique_lenders=("lender_id", "nunique"),
        unique_events=("event_id", "nunique"),
        first_date=("edge_date", "min"),
        last_date=("edge_date", "max")
    )
)

period_order = [
    "pre_2006",
    "train_period",
    "validation_period",
    "test_period",
    "post_test"
]

print("GRAPH CONNECTIONS BY PERIOD")
display(
    graph_period_summary.reindex(period_order)
)

print("\nTotal graph edges:")
print(len(graph_timeline))

print("\nUnique connected events:")
print(graph_timeline["event_id"].nunique())


GRAPH CONNECTIONS BY PERIOD


,edges,unique_lenders,unique_events,first_date,last_date
graph_period,,,,,
pre_2006,96,19,22,2005-11-15 00:00:00+00:00,2005-12-30 00:00:00+00:00
train_period,1062511,32764,17426,2006-01-10 00:00:00+00:00,2007-12-31 00:00:00+00:00
validation_period,372125,24459,4379,2008-01-02 00:00:00+00:00,2008-04-30 00:00:00+00:00
test_period,406972,26723,5607,2008-05-01 00:00:00+00:00,2008-08-29 00:00:00+00:00
post_test,1501580,31579,16138,2008-09-02 00:00:00+00:00,2011-09-27 00:00:00+00:00



Total graph edges:
3343284

Unique connected events:
43572


### 3.2 Temporal Distribution of Graph Connections

The indexed Prosper network contains 59,046 lender nodes, 43,572 provisional borrowing-event nodes and 3,343,284 unique lender–event connections.

All connections were successfully mapped to valid lender and event node indices.

The next stage examines the temporal distribution of the graph connections.

Each connection is assigned to a chronological period based on its recorded timestamp.

The periods include events before 2006, the training period, the validation period, the testing period and events occurring after the testing window.

This analysis will establish the number of observed financial relationships available during each period.

The purpose is to support the construction of time-restricted graph representations for subsequent GraphSAGE experiments.

Connections recorded after a prediction date must not influence predictions for earlier borrowing events.

Furthermore, repayment outcomes and future event attributes must not be propagated through shared lender nodes.

The temporal analysis is exploratory and does not yet establish that the resulting graph snapshots are fully free from information leakage.

The final graph-construction procedure will need to define the prediction point and permissible information for each observation.

In [33]:

# Step 37: Check graph coverage of labelled modelling events

# Count graph connections for each event
event_graph_coverage = (
    indexed_graph_edges
    .groupby("event_id")
    .agg(
        graph_edge_count=("lender_node_index", "size"),
        graph_unique_lenders=("lender_node_index", "nunique")
    )
    .reset_index()
)

# Match graph coverage to our existing modelling partitions
coverage_check = split_data[
    ["event_id", "partition", "default_flag", "unique_lenders"]
].merge(
    event_graph_coverage,
    on="event_id",
    how="left",
    validate="one_to_one"
)

# Check for events without graph connections
coverage_check["has_graph_edges"] = (
    coverage_check["graph_edge_count"].fillna(0) > 0
)

print("TOTAL MODELLING EVENTS:", len(coverage_check))

print("\nGRAPH COVERAGE BY PARTITION")
coverage_summary = (
    coverage_check.groupby("partition")
    .agg(
        labelled_events=("event_id", "size"),
        events_with_edges=("has_graph_edges", "sum"),
        minimum_edges=("graph_edge_count", "min"),
        median_edges=("graph_edge_count", "median"),
        maximum_edges=("graph_edge_count", "max")
    )
)

display(
    coverage_summary.reindex(
        ["train", "validation", "test"]
    )
)

print("\nEvents without graph connections:")
print((~coverage_check["has_graph_edges"]).sum())

print("\nMismatch with original unique lender counts:")
print(
    (
        coverage_check["unique_lenders"]
        != coverage_check["graph_unique_lenders"]
    ).sum()
)

print("\nDuplicate modelling event IDs:")
print(coverage_check["event_id"].duplicated().sum())


TOTAL MODELLING EVENTS: 27285

GRAPH COVERAGE BY PARTITION


,labelled_events,events_with_edges,minimum_edges,median_edges,maximum_edges
partition,,,,,
train,17361,17361,1,43.0,449
validation,4379,4379,1,65.0,458
test,5545,5545,1,51.0,433



Events without graph connections:
0

Mismatch with original unique lender counts:
0

Duplicate modelling event IDs:
0


### 3.3 Temporal Graph Distribution and Labelled Event Coverage

The temporal analysis identified 3,343,284 unique lender–event connections across 43,572 provisional borrowing events.

The training period contained 1,062,511 connections, the validation period contained 372,125 connections, and the testing period contained 406,972 connections.

A further 1,501,580 connections occurred after the testing period and must not be used to predict earlier borrowing events.

The graph also contains borrowing events that are excluded from the supervised modelling population because they do not meet the selected repayment-status and date criteria.

These events may provide additional network context, but their repayment outcomes will not be used as supervised labels.

The next validation step checks whether all 27,285 labelled modelling events have corresponding observed lender connections.

Graph-derived lender counts will also be compared against the previously calculated event-level unique lender counts.

This ensures that the graph representation and Logistic Regression baseline cover the same labelled observations.

For the subsequent GraphSAGE experiment, the graph structure must respect the chronological prediction setting. Future financial relationships and repayment outcomes must not influence earlier predictions.

Because verified repayment-resolution dates are unavailable, the experiment will be interpreted as retrospective post-funding default prediction rather than a fully reconstructed historical lending-decision system.

In [34]:

# Step 38: Prepare event-node features using training-fitted preprocessing

import numpy as np
import pandas as pd

# Use the existing Logistic Regression preprocessing pipeline
# It was already fitted on X_train only.

event_feature_columns = (
    numeric_features + categorical_features
)

# Keep modelling events in event_id order
graph_model_events = (
    split_data
    .sort_values("event_id")
    .reset_index(drop=True)
    .copy()
)

# Transform event features using the fitted training preprocessor
event_features = lr_pipeline.named_steps[
    "preprocessor"
].transform(
    graph_model_events[event_feature_columns]
)

# Convert sparse output to a dense array if necessary
if hasattr(event_features, "toarray"):
    event_features = event_features.toarray()

event_features = np.asarray(
    event_features,
    dtype=np.float32
)

# Create an event_id-to-feature-row mapping
event_feature_mapping = pd.DataFrame({
    "event_id": graph_model_events["event_id"],
    "feature_row_index": np.arange(
        len(graph_model_events)
    )
})

print("EVENT FEATURE PREPARATION")
print("-------------------------")

print("Number of labelled events:", len(graph_model_events))

print("Event feature matrix shape:", event_features.shape)

print("Missing feature values:", np.isnan(event_features).sum())

print("Infinite feature values:", np.isinf(event_features).sum())

print(
    "Duplicate event IDs:",
    event_feature_mapping["event_id"].duplicated().sum()
)

print("\nFirst five feature vectors:")
display(
    pd.DataFrame(event_features).head()
)

print("\nEvent feature mapping:")
display(event_feature_mapping.head())


EVENT FEATURE PREPARATION
-------------------------
Number of labelled events: 27285
Event feature matrix shape: (27285, 11)
Missing feature values: 0
Infinite feature values: 0
Duplicate event IDs: 0

First five feature vectors:


,0,1,2,3,4,5,6,7,8,9,10
0,-0.574673,-1.068492,-1.074304,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,-0.592577,-0.997221,-0.986889,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.718897,-0.943768,-0.899475,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,-0.753356,-0.979403,-0.986889,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.311578,-0.925950,-0.899475,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0



Event feature mapping:


,event_id,feature_row_index
0,22,0
1,23,1
2,24,2
3,25,3
4,26,4


### 3.4 Preparation of Borrowing-Event Node Features

The graph coverage analysis confirmed that all 27,285 labelled borrowing events have at least one observed lender connection.

No modelling events were missing from the graph, and the graph-derived unique lender counts matched the original event-level lender counts.

The next stage prepares numerical feature vectors for borrowing-event nodes.

To maintain comparability with the Logistic Regression baseline, the initial GraphSAGE experiment uses the same four financial features: total recorded amount, number of unique lenders, number of lending records and credit grade.

The preprocessing transformations fitted on the Logistic Regression training partition are reused to standardize numerical variables and encode the categorical credit-grade variable.

This ensures that feature preprocessing is based on training observations rather than information from validation or testing observations.

The resulting feature matrix contains one row per labelled borrowing event.

A separate mapping links each original event identifier to its position in the feature matrix.

Validation checks examine the number of feature rows, missing values, infinite values and duplicate event identifiers.

The prepared feature matrix will subsequently be integrated into the graph representation.

The final GraphSAGE data-loading procedure must prevent future events and future financial relationships from influencing earlier predictions.

The same labelled observations and chronological partitions will be retained for comparison with the Logistic Regression baseline.

In [35]:

# Step 39: Prepare initial lender-node features

import numpy as np

# Number of lender nodes
num_lenders = len(lender_mapping)

# Assign one constant feature to every lender
lender_features = np.ones(
    (num_lenders, 1),
    dtype=np.float32
)

# Verify that graph node indices are consecutive
expected_indices = np.arange(num_lenders)

actual_indices = (
    lender_mapping["lender_node_index"]
    .to_numpy()
)

indices_valid = np.array_equal(
    actual_indices,
    expected_indices
)

print("LENDER FEATURE PREPARATION")
print("--------------------------")

print("Number of lenders:", num_lenders)

print(
    "Lender feature matrix shape:",
    lender_features.shape
)

print(
    "Consecutive lender indices:",
    indices_valid
)

print(
    "Missing lender feature values:",
    np.isnan(lender_features).sum()
)

print(
    "Infinite lender feature values:",
    np.isinf(lender_features).sum()
)

print("\nFirst five lender feature vectors:")
print(lender_features[:5])

print("\nFirst five lender mappings:")
display(lender_mapping.head())


LENDER FEATURE PREPARATION
--------------------------
Number of lenders: 59046
Lender feature matrix shape: (59046, 1)
Consecutive lender indices: True
Missing lender feature values: 0
Infinite lender feature values: 0

First five lender feature vectors:
[[1.]
 [1.]
 [1.]
 [1.]
 [1.]]

First five lender mappings:


,lender_id,lender_node_index
0,0,0
1,1,1
2,2,2
3,3,3
4,4,4


### 3.5 Preparation of Lender-Node Features

The borrowing-event feature preparation produced a matrix containing 27,285 rows and 11 numerical columns.

The features consist of three standardized numerical variables and eight one-hot-encoded credit-grade categories.

No missing or infinite values were detected, and every labelled event was assigned a unique feature-row mapping.

The next stage prepares the initial lender-node representations.

The Prosper extract does not provide a separate set of lender financial characteristics. Therefore, the initial graph experiment assigns a constant feature value of 1.0 to every lender.

This produces a lender-feature matrix with one row for each of the 59,046 lender nodes.

The constant feature does not represent lender income, creditworthiness or any other financial characteristic. It is used only as an initial input for graph message passing.

A heterogeneous GraphSAGE architecture will subsequently be considered to support information exchange between lender nodes and borrowing-event nodes.

The ability of this representation to capture useful relational information will be assessed empirically.

The graph construction and training procedure must prevent future events, future funding relationships and repayment outcomes from entering earlier predictions.

This approach preserves the observed network structure without inventing lender financial attributes.

In [36]:

# Step 40: Check the GraphSAGE software environment

import sys
import importlib.util

print("GRAPH NEURAL NETWORK ENVIRONMENT")
print("--------------------------------")

print("Python version:", sys.version.split()[0])

# Check package availability without installing anything
packages = {
    "torch": "PyTorch",
    "torch_geometric": "PyTorch Geometric"
}

for package, name in packages.items():
    installed = importlib.util.find_spec(package) is not None
    print(f"{name} installed: {installed}")

# Check PyTorch and available hardware
if importlib.util.find_spec("torch") is not None:
    import torch

    print("\nPyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    print("Selected device:", device)

# Check PyTorch Geometric
if importlib.util.find_spec("torch_geometric") is not None:
    import torch_geometric
    from torch_geometric.nn import SAGEConv
    from torch_geometric.data import HeteroData

    print(
        "\nPyTorch Geometric version:",
        torch_geometric.__version__
    )

    print("SAGEConv import: Successful")
    print("HeteroData import: Successful")


GRAPH NEURAL NETWORK ENVIRONMENT
--------------------------------
Python version: 3.13.5
PyTorch installed: False
PyTorch Geometric installed: False


### 3.6 Graph Neural Network Environment Preparation

The lender-node feature preparation was completed successfully.

A total of 59,046 lender nodes were assigned constant initial feature vectors containing the value 1.0.

All lender indices were consecutive, and no missing or infinite feature values were detected.

The borrowing-event feature matrix contains 27,285 labelled events with 11 numerical features per event.

Before constructing the graph tensors, the software environment will be checked for PyTorch and PyTorch Geometric.

PyTorch will provide the neural-network training functionality, while PyTorch Geometric will support graph data structures and GraphSAGE convolution operations.

The environment check will also identify whether CUDA acceleration is available.

The eventual graph implementation must preserve the distinction between lender and event nodes, correctly map event identifiers to feature rows, and prevent future network information from influencing earlier predictions.

The initial constant lender features are a modelling choice whose effectiveness must be evaluated rather than assumed.

In [37]:

# Step 41: Install Graph Neural Network libraries
# Install into the Python environment used by this notebook

%pip install torch torch-geometric


  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
   ---------------------------------------- 0.0/126.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/126.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/126.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/126.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/126.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/126.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/126.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/126.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/126.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/126.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/126.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/126.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/126.7 MB ? eta -:--:--
   -----------

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\YOGA\\AppData\\Local\\Temp\\pip-unpack-ivaickmw\\torch-2.15.0-cp313-cp313-win_amd64.whl'
Check the permissions.



### 3.7 Installation of Graph Neural Network Libraries

The software environment check identified Python version 3.13.5.

Neither PyTorch nor PyTorch Geometric was initially installed in the active Jupyter environment.

These libraries are required for the planned GraphSAGE implementation.

PyTorch provides tensor computation, neural-network operations and automatic differentiation.

PyTorch Geometric provides graph-learning components, including graph data structures and GraphSAGE convolution layers.

The libraries will be installed in the existing Jupyter environment before proceeding with graph construction and model training.

Successful installation and package imports will be verified in the next step.

### 3.8 Verification of the Graph Neural Network Environment (also at 3.11)

Following installation, the Python environment was checked for compatibility with PyTorch and PyTorch Geometric.

The verification checks whether both libraries can be imported and whether the GraphSAGE convolution layer (`SAGEConv`) is available.

GPU availability is also checked using PyTorch's CUDA interface. A CUDA-enabled GPU is optional because GraphSAGE can also be trained using CPU computation.

Successful verification confirms that the environment is ready for the next stage of graph data preparation.

In [39]:

# Step 42A: Install PyTorch in the current Jupyter environment

import sys
import subprocess

print("Python executable:", sys.executable)

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "torch"
])

print("\nPyTorch installation completed!")


Python executable: c:\GroupWork\DS_research_project-maria_kisakye-\.venv\Scripts\python.exe

PyTorch installation completed!


In [40]:

# Step 42B: Verify PyTorch installation

import sys
import torch

print("PYTORCH VERIFICATION")
print("--------------------")

print("Python executable:", sys.executable)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Selected device:", device)

# Test a simple tensor operation
test_tensor = torch.tensor([1.0, 2.0, 3.0])

print("\nTest tensor:", test_tensor)
print("Tensor sum:", test_tensor.sum().item())

print("\nPyTorch verification successful!")


PYTORCH VERIFICATION
--------------------
Python executable: c:\GroupWork\DS_research_project-maria_kisakye-\.venv\Scripts\python.exe
PyTorch version: 2.15.0+cpu
CUDA available: False
Selected device: cpu

Test tensor: tensor([1., 2., 3.])
Tensor sum: 6.0

PyTorch verification successful!


### 3.9 PyTorch Installation and Verification

The initial attempt to import PyTorch resulted in a `ModuleNotFoundError`, indicating that the package was unavailable to the active Jupyter kernel.

To resolve this, PyTorch was installed using the Python executable associated with the project's virtual environment.

The installation command completed successfully.

The next step verifies the installed PyTorch version, checks CUDA availability and performs a basic tensor calculation to confirm that the library functions correctly.

Successful verification will allow the project to proceed with installing and testing PyTorch Geometric for GraphSAGE implementation.

In [41]:

# Step 42C: Install PyTorch Geometric

import sys
import subprocess

print("Installing PyTorch Geometric...")
print("Python executable:", sys.executable)

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "torch-geometric"
])

print("\nPyTorch Geometric installation completed!")


Installing PyTorch Geometric...
Python executable: c:\GroupWork\DS_research_project-maria_kisakye-\.venv\Scripts\python.exe

PyTorch Geometric installation completed!


### 3.10 PyTorch Verification and PyTorch Geometric Installation

PyTorch was successfully installed and verified in the project's Python 3.13.5 virtual environment.

The installed PyTorch version was 2.15.0+cpu

In [42]:

# Step 42D: Verify PyTorch Geometric and GraphSAGE

import torch
import torch_geometric

from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv

print("GRAPH NEURAL NETWORK LIBRARY VERIFICATION")
print("-----------------------------------------")
print("PyTorch version:", torch.__version__)
print("PyTorch Geometric version:", torch_geometric.__version__)
print("Selected device:", "cuda" if torch.cuda.is_available() else "cpu")

# Test GraphSAGE on a small example graph
x = torch.tensor([
    [1.0, 2.0],
    [2.0, 3.0],
    [3.0, 4.0]
])

edge_index = torch.tensor([
    [0, 1, 2, 1],
    [1, 0, 1, 2]
], dtype=torch.long)

sage_layer = SAGEConv(in_channels=2, out_channels=4)

with torch.no_grad():
    output = sage_layer(x, edge_index)

print("\nGraphSAGE test output shape:", tuple(output.shape))
print("Expected output shape: (3, 4)")
print("HeteroData available:", HeteroData is not None)

print("\nGraphSAGE environment verification successful!")


c:\GroupWork\DS_research_project-maria_kisakye-\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\GroupWork\DS_research_project-maria_kisakye-\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


GRAPH NEURAL NETWORK LIBRARY VERIFICATION
-----------------------------------------
PyTorch version: 2.15.0+cpu
PyTorch Geometric version: 2.8.1
Selected device: cpu

GraphSAGE test output shape: (3, 4)
Expected output shape: (3, 4)
HeteroData available: True

GraphSAGE environment verification successful!


### 3.11 Verification of PyTorch Geometric and GraphSAGE

PyTorch Geometric was installed in the project's existing Python virtual environment.

To verify compatibility, a small graph containing three nodes was created and processed using a GraphSAGE convolution layer.

The test checks whether the GraphSAGE implementation can aggregate information from connected nodes and generate output embeddings.

The `HeteroData` class is also imported to confirm that the environment supports heterogeneous graph representations.

Once verification is successful, the next stage will involve converting the prepared Prosper lender–borrowing-event relationships into graph tensors.

The graph implementation will preserve the chronological training, validation and testing partitions to reduce the risk of information leakage.

In [43]:

# Step 43: Prepare features for all borrowing-event nodes

import numpy as np
import pandas as pd

# All borrowing events, ordered by their existing graph node indices
all_graph_events = (
    event_data
    .sort_values("event_id")
    .reset_index(drop=True)
    .copy()
)

# Use the preprocessor fitted on the training partition only
graph_preprocessor = lr_pipeline.named_steps["preprocessor"]

all_event_features = graph_preprocessor.transform(
    all_graph_events[feature_columns]
)

# Convert to a dense NumPy array if necessary
if hasattr(all_event_features, "toarray"):
    all_event_features = all_event_features.toarray()

all_event_features = np.asarray(
    all_event_features,
    dtype=np.float32
)

# Verify that event IDs correspond to graph node indices
event_indices_valid = np.array_equal(
    all_graph_events["event_id"].to_numpy(),
    np.arange(len(all_graph_events))
)

print("COMPLETE GRAPH EVENT FEATURES")
print("-----------------------------")
print("Total event nodes:", len(all_graph_events))
print("Feature matrix shape:", all_event_features.shape)
print("Feature columns:", feature_columns)
print("Event indices aligned:", event_indices_valid)
print("Missing feature values:", np.isnan(all_event_features).sum())
print("Infinite feature values:", np.isinf(all_event_features).sum())

assert event_indices_valid, "Event node indices are not aligned!"
assert np.isfinite(all_event_features).all(), "Invalid feature values found!"

print("\nComplete event feature preparation successful!")


COMPLETE GRAPH EVENT FEATURES
-----------------------------
Total event nodes: 43572
Feature matrix shape: (43572, 11)
Feature columns: ['total_recorded_amount', 'unique_lenders', 'lending_records', 'credit_grade']
Event indices aligned: True
Missing feature values: 0
Infinite feature values: 0

Complete event feature preparation successful!


### 3.12 Preparation of Complete Borrowing-Event Node Features

The initial graph feature matrix contained only the 27,285 borrowing events selected for supervised credit-risk modelling. However, the complete Prosper network contains 43,572 borrowing events.

To support graph representation, features were prepared for all borrowing-event nodes, including events excluded from supervised evaluation because of their status or observation period.

The same numerical standardisation and categorical encoding fitted on the Logistic Regression training partition were reused. This avoids fitting preprocessing transformations on validation or test data.

The resulting feature matrix was checked for missing values, infinite values and alignment between event identifiers and graph node indices.

The inclusion of an event in the feature matrix does not mean that it is eligible for training. Supervised labels and graph relationships will be restricted according to the research design and chronological evaluation protocol.

In [44]:

# Step 44: Convert graph node features into PyTorch tensors

import torch

# Borrowing-event node features
event_x = torch.from_numpy(
    all_event_features
).float()

# Lender node features
lender_x = torch.from_numpy(
    lender_features
).float()

print("GRAPH NODE FEATURE TENSORS")
print("--------------------------")

print("Borrowing-event tensor shape:", tuple(event_x.shape))
print("Lender tensor shape:", tuple(lender_x.shape))

print("\nEvent tensor dtype:", event_x.dtype)
print("Lender tensor dtype:", lender_x.dtype)

print("\nEvent tensor device:", event_x.device)
print("Lender tensor device:", lender_x.device)

print("\nEvent tensor finite:", torch.isfinite(event_x).all().item())
print("Lender tensor finite:", torch.isfinite(lender_x).all().item())

assert event_x.shape == (43572, 11)
assert lender_x.shape == (59046, 1)
assert torch.isfinite(event_x).all()
assert torch.isfinite(lender_x).all()

print("\nGraph node tensor preparation successful!")


GRAPH NODE FEATURE TENSORS
--------------------------
Borrowing-event tensor shape: (43572, 11)
Lender tensor shape: (59046, 1)

Event tensor dtype: torch.float32
Lender tensor dtype: torch.float32

Event tensor device: cpu
Lender tensor device: cpu

Event tensor finite: True
Lender tensor finite: True

Graph node tensor preparation successful!


### 3.13 Conversion of Node Features into PyTorch Tensors

The prepared node feature matrices were converted from NumPy arrays into PyTorch tensors with 32-bit floating-point precision.

The borrowing-event feature tensor represents 43,572 events, each described by 11 processed features. The lender feature tensor represents 59,046 lenders, each initially assigned a constant feature value of 1.0.

The tensors were checked for the expected dimensions, numerical validity and computational device.

This conversion prepares the two node types for use in a heterogeneous GraphSAGE model. Graph connections and supervised labels will be incorporated separately, with attention to chronological information leakage.

In [45]:

# Step 45: Convert lender-event relationships into edge tensors

import numpy as np
import torch

# Extract indexed lender-event relationships
edge_array = indexed_graph_edges[
    ["lender_node_index", "event_node_index"]
].to_numpy(dtype=np.int64)

# Shape: [2, number_of_edges]
lender_to_event_edge_index = torch.from_numpy(
    edge_array.T.copy()
).long()

# Reverse direction for heterogeneous message passing
event_to_lender_edge_index = (
    lender_to_event_edge_index.flip(0).contiguous()
)

print("GRAPH EDGE TENSORS")
print("------------------")

print(
    "Lender-to-event shape:",
    tuple(lender_to_event_edge_index.shape)
)

print(
    "Event-to-lender shape:",
    tuple(event_to_lender_edge_index.shape)
)

print("\nLender index range:",
      lender_to_event_edge_index[0].min().item(),
      "to",
      lender_to_event_edge_index[0].max().item())

print("Event index range:",
      lender_to_event_edge_index[1].min().item(),
      "to",
      lender_to_event_edge_index[1].max().item())

# Validate index boundaries
assert lender_to_event_edge_index.shape[0] == 2
assert lender_to_event_edge_index.shape[1] == len(indexed_graph_edges)

assert lender_to_event_edge_index[0].min() >= 0
assert lender_to_event_edge_index[0].max() < lender_x.shape[0]

assert lender_to_event_edge_index[1].min() >= 0
assert lender_to_event_edge_index[1].max() < event_x.shape[0]

assert torch.equal(
    event_to_lender_edge_index,
    lender_to_event_edge_index.flip(0)
)

print("\nGraph edge tensor preparation successful!")


GRAPH EDGE TENSORS
------------------
Lender-to-event shape: (2, 3343284)
Event-to-lender shape: (2, 3343284)

Lender index range: 0 to 59045
Event index range: 0 to 43571

Graph edge tensor preparation successful!


### 3.14 Construction of Lender–Borrowing-Event Edge Tensors

The observed Prosper funding relationships were converted into PyTorch Geometric-compatible edge index tensors.

The graph contains 3,343,284 unique relationships connecting lender nodes to borrowing-event nodes. Each relationship is represented by the corresponding lender and event node indices.

Two directed edge representations were prepared: lender-to-event and event-to-lender. The reverse representation enables message passing in both directions within the heterogeneous graph.

The edge tensors were checked for their expected dimensions, valid node indices and consistency between forward and reverse relationships.

These tensors represent the complete observed network and are not yet used for model training. Chronological restrictions will be applied before graph learning to prevent future funding relationships from entering earlier prediction contexts.

In [46]:

# Step 46: Create chronological graph edge masks

import numpy as np
import pandas as pd
import torch

# Convert each edge timestamp to a UTC datetime
edge_dates = pd.to_datetime(
    indexed_graph_edges["timestamp"],
    unit="s",
    utc=True
)

train_cutoff = pd.Timestamp("2008-01-01", tz="UTC")
validation_cutoff = pd.Timestamp("2008-05-01", tz="UTC")
test_cutoff = pd.Timestamp("2008-09-01", tz="UTC")

# Cumulative graph snapshots
train_edge_mask = torch.from_numpy(
    np.asarray(edge_dates < train_cutoff)
)

validation_edge_mask = torch.from_numpy(
    np.asarray(edge_dates < validation_cutoff)
)

test_edge_mask = torch.from_numpy(
    np.asarray(edge_dates < test_cutoff)
)

# Verify chronological nesting
assert torch.all(train_edge_mask <= validation_edge_mask)
assert torch.all(validation_edge_mask <= test_edge_mask)

# Count edges in each snapshot
print("CHRONOLOGICAL GRAPH EDGE MASKS")
print("------------------------------")

print("Training snapshot edges:",
      train_edge_mask.sum().item())

print("Validation snapshot edges:",
      validation_edge_mask.sum().item())

print("Testing snapshot edges:",
      test_edge_mask.sum().item())

print("\nExcluded post-test edges:",
      (~test_edge_mask).sum().item())

print("\nTotal original graph edges:",
      len(indexed_graph_edges))

assert len(train_edge_mask) == lender_to_event_edge_index.shape[1]
assert len(validation_edge_mask) == lender_to_event_edge_index.shape[1]
assert len(test_edge_mask) == lender_to_event_edge_index.shape[1]

print("\nChronological edge masks prepared successfully!")


CHRONOLOGICAL GRAPH EDGE MASKS
------------------------------
Training snapshot edges: 1062607
Validation snapshot edges: 1434732
Testing snapshot edges: 1841704

Excluded post-test edges: 1501580

Total original graph edges: 3343284

Chronological edge masks prepared successfully!


C:\Users\YOGA\AppData\Local\Temp\ipykernel_20528\1702461254.py:19: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:214.)
  train_edge_mask = torch.from_numpy(


### 3.15 Preparation of Chronological Graph Edge Masks

Chronological masks were constructed for the observed Prosper lender–borrowing-event relationships using their recorded timestamps.

Three cumulative graph snapshots were defined:

- **Training snapshot:** Relationships recorded before 1 January 2008.
- **Validation snapshot:** Relationships recorded before 1 May 2008.
- **Testing snapshot:** Relationships recorded before 1 September 2008.

Each successive snapshot contains the relationships from the preceding period, preserving historical network information.

Relationships recorded after August 2008 are excluded from the three modelling snapshots.

The masks were checked to ensure that they contain the expected number of edges and follow the correct chronological order.

These snapshots establish preliminary temporal boundaries. They do not, by themselves, guarantee that each prediction uses only information available at its event date. Additional temporal controls will be necessary during graph construction and message passing.

The objective is to evaluate whether observed lender–borrowing-event relationships provide predictive information beyond the feature-based Logistic Regression baseline without introducing future network information into earlier predictions.

In [48]:

# Step 47: Prepare supervised labels and partition masks

import numpy as np
import torch

num_events = len(all_graph_events)

# -1 means the event has no eligible supervised label
event_y = torch.full(
    (num_events,),
    -1,
    dtype=torch.long
)

train_node_mask = torch.zeros(num_events, dtype=torch.bool)
validation_node_mask = torch.zeros(num_events, dtype=torch.bool)
test_node_mask = torch.zeros(num_events, dtype=torch.bool)

# Assign labels and masks using the established event IDs
event_ids = torch.tensor(
    split_data["event_id"].to_numpy().copy(),
    dtype=torch.long
)

labels = torch.tensor(
    split_data["default_flag"].to_numpy().copy(),
    dtype=torch.long
)

event_y[event_ids] = labels

train_ids = torch.tensor(
    split_data.loc[
        split_data["partition"] == "train", "event_id"
    ].to_numpy().copy(),
    dtype=torch.long
)

validation_ids = torch.tensor(
    split_data.loc[
        split_data["partition"] == "validation", "event_id"
    ].to_numpy().copy(),
    dtype=torch.long
)

test_ids = torch.tensor(
    split_data.loc[
        split_data["partition"] == "test", "event_id"
    ].to_numpy().copy(),
    dtype=torch.long
)

train_node_mask[train_ids] = True
validation_node_mask[validation_ids] = True
test_node_mask[test_ids] = True

# Validate label and partition consistency
supervised_mask = (
    train_node_mask |
    validation_node_mask |
    test_node_mask
)

assert supervised_mask.sum().item() == len(split_data)
assert torch.all(event_y[supervised_mask] >= 0)
assert torch.all(event_y[~supervised_mask] == -1)

assert not torch.any(train_node_mask & validation_node_mask)
assert not torch.any(train_node_mask & test_node_mask)
assert not torch.any(validation_node_mask & test_node_mask)

print("SUPERVISED GRAPH LABELS AND MASKS")
print("---------------------------------")
print("Total graph events:", num_events)
print("Supervised events:", supervised_mask.sum().item())
print("Unlabelled/excluded events:", (~supervised_mask).sum().item())

print("\nPartition counts:")
print("Training:", train_node_mask.sum().item())
print("Validation:", validation_node_mask.sum().item())
print("Testing:", test_node_mask.sum().item())

print("\nDefault counts:")
print("Training:", event_y[train_node_mask].sum().item())
print("Validation:", event_y[validation_node_mask].sum().item())
print("Testing:", event_y[test_node_mask].sum().item())

print("\nSupervised label preparation successful!")


SUPERVISED GRAPH LABELS AND MASKS
---------------------------------
Total graph events: 43572
Supervised events: 27285
Unlabelled/excluded events: 16287

Partition counts:
Training: 17361
Validation: 4379
Testing: 5545

Default counts:
Training: 6848
Validation: 1480
Testing: 1806

Supervised label preparation successful!


### 3.16 Preparation of Supervised Graph Labels and Partition Masks

A supervised label tensor was created for the 43,572 borrowing-event nodes in the Prosper graph.

Eligible borrowing events were assigned binary credit-risk labels, where 0 represents Paid and 1 represents Defaulted or Charge-off. Events excluded from supervised evaluation were assigned a placeholder value of -1.

Boolean masks were constructed to identify the training, validation and testing events according to the chronological partitions established for the Logistic Regression baseline.

The masks were checked to confirm that every eligible event belongs to exactly one partition and that excluded events do not receive supervised labels.

The labels remain separate from the graph input features. During GraphSAGE training, the training mask will control which labels contribute to the optimisation objective, while validation and test labels will be reserved for their respective evaluation stages.

Further temporal restrictions will be applied to graph message passing before model training.

In [49]:

# Step 48: Examine lender relationships across modelling periods

import pandas as pd

# Link funding relationships to supervised event partitions
edge_partition_check = indexed_graph_edges[
    ["lender_node_index", "event_node_index"]
].merge(
    split_data[["event_id", "partition"]],
    left_on="event_node_index",
    right_on="event_id",
    how="inner",
    validate="many_to_one"
)

# Count unique lenders participating in each partition
lenders_by_partition = {
    period: set(
        edge_partition_check.loc[
            edge_partition_check["partition"] == period,
            "lender_node_index"
        ].unique()
    )
    for period in ["train", "validation", "test"]
}

train_lenders = lenders_by_partition["train"]
validation_lenders = lenders_by_partition["validation"]
test_lenders = lenders_by_partition["test"]

print("TEMPORAL LENDER OVERLAP")
print("-----------------------")

print("Training lenders:", len(train_lenders))
print("Validation lenders:", len(validation_lenders))
print("Testing lenders:", len(test_lenders))

print("\nLenders shared between periods:")
print("Train and validation:",
      len(train_lenders & validation_lenders))

print("Train and test:",
      len(train_lenders & test_lenders))

print("Validation and test:",
      len(validation_lenders & test_lenders))

print("\nNew lenders in later partitions:")
print("Validation lenders not in training:",
      len(validation_lenders - train_lenders))

print("Test lenders not in training:",
      len(test_lenders - train_lenders))

print("\nFunding edges linked to supervised events:",
      len(edge_partition_check))

print("\nTemporal lender overlap analysis completed!")


TEMPORAL LENDER OVERLAP
-----------------------
Training lenders: 32747
Validation lenders: 24459
Testing lenders: 26674

Lenders shared between periods:
Train and validation: 16174
Train and test: 13835
Validation and test: 17288

New lenders in later partitions:
Validation lenders not in training: 8285
Test lenders not in training: 12839

Funding edges linked to supervised events: 1833895

Temporal lender overlap analysis completed!


### 3.17 Analysis of Lender Overlap Across Chronological Partitions

An analysis was prepared to examine how lenders participate across the training, validation and testing periods.

The purpose is to identify lenders that connect borrowing events occurring in different chronological partitions.

Such relationships are important because they provide the relational structure through which GraphSAGE may learn additional information beyond conventional borrower-event features.

However, shared lenders also create a potential pathway for temporal information leakage if future borrowing events are included when predicting earlier events.

The analysis measures the number of unique lenders in each partition, the overlap between partitions and the number of lenders appearing in later partitions but not in the training partition.

<!-- These results will inform the design of temporally restricted graph message passing. They will not, by themselves, establish that the model is free from leakage. -->

In [50]:

# Step 49: Measure historical lender coverage at each event timestamp

import numpy as np
import pandas as pd

# Keep only the information needed for historical coverage
history_edges = indexed_graph_edges[
    ["lender_node_index", "event_node_index", "timestamp"]
].copy()

# Identify each lender's first observed funding timestamp
first_lender_timestamp = (
    history_edges
    .groupby("lender_node_index")["timestamp"]
    .min()
)

# Map first participation dates back to individual edges
history_edges["lender_first_timestamp"] = (
    history_edges["lender_node_index"]
    .map(first_lender_timestamp)
)

# Strictly earlier means earlier than the current event timestamp
history_edges["has_prior_lender_history"] = (
    history_edges["lender_first_timestamp"]
    < history_edges["timestamp"]
)

# Summarise historical lender coverage for each borrowing event
event_history = (
    history_edges
    .groupby("event_node_index")
    .agg(
        total_lenders=("lender_node_index", "size"),
        lenders_with_history=("has_prior_lender_history", "sum")
    )
    .reset_index()
    .rename(columns={"event_node_index": "event_id"})
)

event_history["historical_lender_share"] = (
    event_history["lenders_with_history"]
    / event_history["total_lenders"]
)

# Attach existing modelling partitions
history_summary = split_data[
    ["event_id", "partition"]
].merge(
    event_history,
    on="event_id",
    how="left",
    validate="one_to_one"
)

summary = (
    history_summary
    .groupby("partition")
    .agg(
        events=("event_id", "size"),
        events_with_historical_lenders=(
            "lenders_with_history",
            lambda x: (x > 0).sum()
        ),
        median_historical_lender_share=(
            "historical_lender_share", "median"
        ),
        mean_historical_lender_share=(
            "historical_lender_share", "mean"
        )
    )
    .reindex(["train", "validation", "test"])
)

print("HISTORICAL LENDER COVERAGE")
print("--------------------------")
print(summary.round(4).to_string())

print("\nEvents missing historical coverage information:",
      history_summary["total_lenders"].isna().sum())

assert history_summary["total_lenders"].notna().all()

print("\nHistorical lender coverage analysis successful!")


HISTORICAL LENDER COVERAGE
--------------------------
            events  events_with_historical_lenders  median_historical_lender_share  mean_historical_lender_share
partition                                                                                                       
train        17361                           17291                          0.9688                        0.9434
validation    4379                            4364                          0.9806                        0.9653
test          5545                            5532                          0.9888                        0.9724

Events missing historical coverage information: 0

Historical lender coverage analysis successful!


### 3.18 Historical Lender Coverage Analysis

Historical lender coverage was examined to determine whether borrowing events are connected to lenders with previously observed funding activity.

For each lender, the earliest recorded funding timestamp was identified. A lender was considered to have prior observed history for a borrowing event only when their first recorded funding timestamp preceded the event's timestamp.

For each borrowing event, the number and proportion of participating lenders with prior observed history were calculated. The results were then summarised separately for the training, validation and testing partitions.

This analysis helps assess whether historical lender relationships are sufficiently common to support further investigation using GraphSAGE.

The measure does not establish whether prior event features or repayment outcomes were available at the prediction date. Those limitations will be considered when designing and interpreting the graph-learning experiment.

In [51]:

# Step 50: Prepare time-ordered lender-event adjacency

import numpy as np
import pandas as pd

# Extract the necessary graph relationship information
temporal_adjacency = indexed_graph_edges[
    ["lender_node_index", "event_node_index", "timestamp"]
].copy()

# Sort by lender, then chronological event timestamp
temporal_adjacency = (
    temporal_adjacency
    .sort_values(
        ["lender_node_index", "timestamp", "event_node_index"]
    )
    .reset_index(drop=True)
)

# Identify where each lender's adjacency list begins and ends
lender_ids = temporal_adjacency[
    "lender_node_index"
].to_numpy(dtype=np.int64)

lender_event_ids = temporal_adjacency[
    "event_node_index"
].to_numpy(dtype=np.int64)

lender_event_times = temporal_adjacency[
    "timestamp"
].to_numpy(dtype=np.int64)

# CSR-style offsets for fast lender-neighbour lookup
lender_edge_counts = np.bincount(
    lender_ids,
    minlength=len(lender_mapping)
)

lender_offsets = np.zeros(
    len(lender_mapping) + 1,
    dtype=np.int64
)

lender_offsets[1:] = np.cumsum(lender_edge_counts)

# Validate adjacency structure
assert lender_offsets[-1] == len(temporal_adjacency)
assert len(lender_event_ids) == len(indexed_graph_edges)
assert np.all(np.diff(lender_offsets) >= 0)

# Check chronological ordering within lender groups
same_lender = lender_ids[1:] == lender_ids[:-1]

assert np.all(
    lender_event_times[1:][same_lender]
    >= lender_event_times[:-1][same_lender]
)

print("TEMPORAL LENDER ADJACENCY")
print("-------------------------")
print("Lenders:", len(lender_mapping))
print("Unique funding relationships:", len(temporal_adjacency))
print("Lender offsets:", len(lender_offsets))
print("Event indices:", len(lender_event_ids))
print("Timestamps:", len(lender_event_times))

print("\nFirst five lender edge counts:")
print(lender_edge_counts[:5])

print("\nChronological ordering verified:", True)
print("Adjacency preparation successful!")


TEMPORAL LENDER ADJACENCY
-------------------------
Lenders: 59046
Unique funding relationships: 3343284
Lender offsets: 59047
Event indices: 3343284
Timestamps: 3343284

First five lender edge counts:
[539 310 117 146 136]

Chronological ordering verified: True
Adjacency preparation successful!


### 3.19 Construction of Time-Ordered Lender Adjacency

A time-ordered adjacency structure was prepared from the observed Prosper lender–borrowing-event relationships.

The relationships were sorted by lender identifier and event timestamp. For each lender, the corresponding borrowing-event indices and timestamps were stored in chronological order.

A compressed sparse row (CSR)-style offset array was constructed to identify the beginning and end of each lender's adjacency list. This allows historical connections to be retrieved efficiently without repeatedly scanning the complete network.

The adjacency structure was validated by checking the total relationship count, offset consistency and chronological ordering within each lender's records.

This structure will support the development of temporally restricted graph neighbourhoods for GraphSAGE.

Only relationships preceding a target event's timestamp should be considered historical neighbours. Additional care is required to prevent future event information from influencing earlier predictions.

The resulting experiment will be interpreted as retrospective post-funding credit-risk classification, subject to the limitations of the available timestamps and repayment outcome information.

In [52]:

# Step 51: Test time-restricted historical-neighbour retrieval

import numpy as np

def get_prior_lender_events(lender_index, target_timestamp):
    """
    Return event indices connected to a lender strictly
    before the target timestamp.
    """
    start = lender_offsets[lender_index]
    end = lender_offsets[lender_index + 1]

    times = lender_event_times[start:end]
    events = lender_event_ids[start:end]

    # Find the first position at or after the target timestamp
    cutoff = np.searchsorted(
        times,
        target_timestamp,
        side="left"
    )

    return events[:cutoff]


# Select the earliest supervised validation event
sample_event = (
    split_data.loc[split_data["partition"] == "validation"]
    .sort_values(["timestamp", "event_id"])
    .iloc[0]
)

sample_event_id = int(sample_event["event_id"])
sample_timestamp = int(sample_event["timestamp"])

# Identify lenders funding the selected event
sample_lenders = temporal_adjacency.loc[
    temporal_adjacency["event_node_index"] == sample_event_id,
    "lender_node_index"
].to_numpy(dtype=np.int64)

# Test the first five participating lenders
selected_lenders = sample_lenders[:5]

print("HISTORICAL NEIGHBOUR RETRIEVAL")
print("------------------------------")
print("Sample event ID:", sample_event_id)
print(
    "Sample event date:",
    pd.to_datetime(sample_timestamp, unit="s", utc=True)
)
print("Total lenders funding event:", len(sample_lenders))

print("\nHistorical events for first five lenders:")

for lender in selected_lenders:
    prior_events = get_prior_lender_events(
        int(lender),
        sample_timestamp
    )

    print(
        f"Lender {lender}: "
        f"{len(prior_events)} earlier events"
    )

    # Verify that every retrieved event is strictly earlier
    if len(prior_events) > 0:
        prior_timestamps = all_graph_events.loc[
            prior_events, "timestamp"
        ].to_numpy()

        assert np.all(
            prior_timestamps < sample_timestamp
        )

        assert sample_event_id not in prior_events

print("\nHistorical neighbour retrieval successful!")


HISTORICAL NEIGHBOUR RETRIEVAL
------------------------------
Sample event ID: 17448
Sample event date: 2008-01-02 00:00:00+00:00
Total lenders funding event: 10

Historical events for first five lenders:
Lender 2257: 277 earlier events
Lender 2466: 526 earlier events
Lender 3117: 25 earlier events
Lender 9812: 2 earlier events
Lender 9939: 671 earlier events

Historical neighbour retrieval successful!


### 3.20 Verification of Historical-Neighbour Retrieval

A historical-neighbour retrieval function was implemented using the time-ordered lender adjacency structure.

For a given lender and target event timestamp, the function uses binary search to retrieve only borrowing events recorded strictly before the target timestamp.

Events with equal or later timestamps are excluded to prevent future relationships from entering the historical neighbourhood.

The function was tested using a borrowing event from the validation partition. Participating lenders were identified, and their earlier connected events were retrieved.

The resulting event timestamps were checked to ensure that all retrieved relationships preceded the target event and that the target event itself was excluded.

This provides a preliminary verification of chronological neighbour retrieval for the planned GraphSAGE architecture.

The check concerns the timing of observed funding relationships. It does not establish the availability dates of repayment outcomes or all recorded features, which remain limitations of the retrospective research design.

In [53]:

# Step 52: Examine historical neighbourhood sizes

import numpy as np
import pandas as pd

# For each funding edge, count how many earlier event
# connections its lender had at that timestamp.
# Equal timestamps are excluded from the historical count.

historical_connection_counts = np.empty(
    len(temporal_adjacency),
    dtype=np.int64
)

for lender_index in range(len(lender_mapping)):
    start = lender_offsets[lender_index]
    end = lender_offsets[lender_index + 1]

    if start == end:
        continue

    times = lender_event_times[start:end]

    historical_connection_counts[start:end] = (
        np.searchsorted(times, times, side="left")
    )

# Summarise historical connections per lender-event relationship
history_stats = pd.Series(
    historical_connection_counts,
    name="historical_connections"
)

print("HISTORICAL NEIGHBOURHOOD SIZE ANALYSIS")
print("--------------------------------------")

print("Total lender-event relationships:", len(history_stats))

print("\nHistorical connection distribution:")
print(
    history_stats.describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    ).round(2).to_string()
)

print("\nRelationships with no earlier lender connections:",
      (history_stats == 0).sum())

print("Relationships with more than 100 earlier connections:",
      (history_stats > 100).sum())

print("Relationships with more than 500 earlier connections:",
      (history_stats > 500).sum())

assert len(history_stats) == len(indexed_graph_edges)
assert (historical_connection_counts >= 0).all()

print("\nHistorical neighbourhood analysis successful!")


HISTORICAL NEIGHBOURHOOD SIZE ANALYSIS
--------------------------------------
Total lender-event relationships: 3343284

Historical connection distribution:
count    3343284.00
mean         235.95
std          457.84
min            0.00
25%           21.00
50%           74.00
75%          235.00
90%          623.00
95%         1051.00
99%         2293.00
max         6968.00

Relationships with no earlier lender connections: 82388
Relationships with more than 100 earlier connections: 1443247
Relationships with more than 500 earlier connections: 423014

Historical neighbourhood analysis successful!


### 3.21 Analysis of Historical Neighbourhood Sizes

The number of earlier borrowing-event connections was calculated for each observed lender–event relationship.

For every lender, the time-ordered adjacency structure was used to count connections recorded strictly before the current event timestamp. Connections with equal timestamps were excluded.

Descriptive statistics were calculated to examine the distribution of historical neighbourhood sizes, including the median and upper percentiles.

The analysis was conducted to inform the design of an efficient temporal GraphSAGE implementation. Large historical neighbourhoods can increase memory consumption and computation time, particularly when training on a CPU.

The resulting distribution will help determine whether neighbour sampling is necessary and guide the selection of a sampling limit.

Any sampling strategy will be applied without including relationships recorded at or after the target event timestamp. Its effect on predictive performance and computational efficiency will be considered during model evaluation.

In [54]:

# Step 53: Implement time-restricted historical-neighbour sampling

import numpy as np

MAX_HISTORICAL_NEIGHBOURS = 32
SAMPLING_SEED = 42

def sample_prior_lender_events(
    lender_index,
    target_timestamp,
    max_neighbours=MAX_HISTORICAL_NEIGHBOURS,
    seed=SAMPLING_SEED
):
    """
    Sample earlier borrowing events connected to a lender.

    Only events strictly before target_timestamp are eligible.
    Sampling is deterministic for the same inputs.
    """
    prior_events = get_prior_lender_events(
        lender_index,
        target_timestamp
    )

    if len(prior_events) <= max_neighbours:
        return prior_events.copy()

    # Reproducible random sampling without replacement
    seed_sequence = np.random.SeedSequence([
        seed,
        int(lender_index),
        int(target_timestamp)
    ])

    rng = np.random.default_rng(seed_sequence)

    sampled = rng.choice(
        prior_events,
        size=max_neighbours,
        replace=False
    )

    return np.sort(sampled)


# Test sampling using the validation event from Step 51
print("HISTORICAL NEIGHBOUR SAMPLING")
print("-----------------------------")
print("Target event ID:", sample_event_id)
print("Maximum historical neighbours:", MAX_HISTORICAL_NEIGHBOURS)

for lender in selected_lenders:
    sampled_events = sample_prior_lender_events(
        int(lender),
        sample_timestamp
    )

    # Repeat to verify reproducibility
    repeated_sample = sample_prior_lender_events(
        int(lender),
        sample_timestamp
    )

    assert np.array_equal(
        sampled_events,
        repeated_sample
    )

    assert len(sampled_events) <= MAX_HISTORICAL_NEIGHBOURS
    assert len(np.unique(sampled_events)) == len(sampled_events)

    if len(sampled_events) > 0:
        sampled_timestamps = all_graph_events.loc[
            sampled_events, "timestamp"
        ].to_numpy()

        assert np.all(sampled_timestamps < sample_timestamp)
        assert sample_event_id not in sampled_events

    print(
        f"Lender {lender}: "
        f"{len(sampled_events)} sampled historical events"
    )

print("\nHistorical neighbour sampling successful!")


HISTORICAL NEIGHBOUR SAMPLING
-----------------------------
Target event ID: 17448
Maximum historical neighbours: 32
Lender 2257: 32 sampled historical events
Lender 2466: 32 sampled historical events
Lender 3117: 25 sampled historical events
Lender 9812: 2 sampled historical events
Lender 9939: 32 sampled historical events

Historical neighbour sampling successful!


### 3.22 Reproducible Historical-Neighbour Sampling

The historical neighbourhood analysis showed that some lenders were connected to thousands of earlier borrowing events, making unrestricted graph aggregation potentially expensive.

A preliminary neighbour-sampling strategy was therefore implemented to select up to 32 historical borrowing events per lender.

Sampling was restricted to events recorded strictly before the target borrowing event's timestamp. A fixed random seed was used to make the selection reproducible.

The sampling function was tested for chronological validity, maximum neighbourhood size, uniqueness of selected events and reproducibility.

The value of 32 was selected as an initial computational setting rather than an empirically established optimum. Alternative sampling limits may be assessed during model validation.

The sampled historical relationships will provide inputs for subsequent graph representation learning. Repayment outcomes will not be included as node features.

In [55]:

# Step 54: Build a sampled historical neighbourhood for one event

import numpy as np

def build_event_neighbourhood(
    target_event_id,
    max_neighbours=MAX_HISTORICAL_NEIGHBOURS
):
    """
    Construct a two-hop historical neighbourhood:
    target event -> participating lenders -> earlier events.
    """
    target_row = all_graph_events.iloc[int(target_event_id)]
    target_timestamp = int(target_row["timestamp"])

    # Retrieve lenders funding the target event
    target_lenders = temporal_adjacency.loc[
        temporal_adjacency["event_node_index"] == target_event_id,
        "lender_node_index"
    ].to_numpy(dtype=np.int64)

    # Store lender-to-event relationships
    historical_edges = []

    for lender in target_lenders:
        earlier_events = sample_prior_lender_events(
            int(lender),
            target_timestamp,
            max_neighbours=max_neighbours
        )

        for earlier_event in earlier_events:
            historical_edges.append(
                (int(lender), int(earlier_event))
            )

    historical_edges = np.asarray(
        historical_edges,
        dtype=np.int64
    ).reshape(-1, 2)

    return {
        "target_event_id": int(target_event_id),
        "target_timestamp": target_timestamp,
        "target_lenders": np.unique(target_lenders),
        "historical_edges": historical_edges,
        "historical_event_ids": np.unique(
            historical_edges[:, 1]
        )
    }


sample_neighbourhood = build_event_neighbourhood(
    sample_event_id
)

print("SAMPLED HISTORICAL EVENT NEIGHBOURHOOD")
print("--------------------------------------")
print("Target event:", sample_neighbourhood["target_event_id"])
print("Participating lenders:",
      len(sample_neighbourhood["target_lenders"]))
print("Sampled historical edges:",
      len(sample_neighbourhood["historical_edges"]))
print("Unique historical events:",
      len(sample_neighbourhood["historical_event_ids"]))

# Verify that every historical event is earlier than the target
historical_ids = sample_neighbourhood["historical_event_ids"]

if len(historical_ids) > 0:
    historical_times = all_graph_events.iloc[
        historical_ids
    ]["timestamp"].to_numpy()

    assert np.all(
        historical_times < sample_neighbourhood["target_timestamp"]
    )

    assert sample_event_id not in historical_ids

assert len(sample_neighbourhood["target_lenders"]) == len(sample_lenders)

assert len(sample_neighbourhood["historical_edges"]) <= (
    len(sample_lenders) * MAX_HISTORICAL_NEIGHBOURS
)

print("\nHistorical neighbourhood construction successful!")


SAMPLED HISTORICAL EVENT NEIGHBOURHOOD
--------------------------------------
Target event: 17448
Participating lenders: 10
Sampled historical edges: 240
Unique historical events: 237

Historical neighbourhood construction successful!


### 3.23 Construction of Sampled Historical Event Neighbourhoods

A function was developed to construct a time-restricted, two-hop neighbourhood for an individual borrowing event.

The neighbourhood contains the target borrowing event, its observed participating lenders and a sample of earlier borrowing events connected to those lenders.

For each lender, up to 32 historical events are selected using the reproducible sampling procedure developed previously. Events occurring at or after the target timestamp are excluded.

The resulting structure records the target event, its participating lenders, sampled historical lender–event relationships and unique historical borrowing events.

Validation checks are used to confirm chronological ordering, exclusion of the target event from its historical neighbourhood and compliance with the maximum sampling limit.

This structure provides the basis for developing a temporal GraphSAGE model that aggregates information from observed lender relationships.

The implementation remains a retrospective post-funding classification experiment. Historical funding timestamps do not establish when all event attributes or repayment outcomes became available.

In [56]:

# Step 55: Convert a sampled neighbourhood into HeteroData

import numpy as np
import torch
from torch_geometric.data import HeteroData

def neighbourhood_to_heterodata(neighbourhood):
    target_id = neighbourhood["target_event_id"]
    target_lenders = neighbourhood["target_lenders"]
    historical_edges = neighbourhood["historical_edges"]
    historical_ids = neighbourhood["historical_event_ids"]

    # Local event nodes: target first, then unique historical events
    local_event_ids = np.concatenate([
        np.array([target_id], dtype=np.int64),
        historical_ids.astype(np.int64)
    ])

    local_lender_ids = np.unique(target_lenders)

    # Map global node IDs to positions in this small graph
    event_lookup = {
        int(node_id): i
        for i, node_id in enumerate(local_event_ids)
    }

    lender_lookup = {
        int(node_id): i
        for i, node_id in enumerate(local_lender_ids)
    }

    # Include current target funding relationships
    current_edges = [
        (int(lender), int(target_id))
        for lender in target_lenders
    ]

    # Include sampled historical relationships
    all_edges = current_edges + [
        (int(lender), int(event))
        for lender, event in historical_edges
    ]

    local_edges = torch.tensor(
        [
            [
                lender_lookup[lender],
                event_lookup[event]
            ]
            for lender, event in all_edges
        ],
        dtype=torch.long
    ).T.contiguous()

    graph = HeteroData()

    graph["lender"].x = lender_x[
        torch.tensor(local_lender_ids, dtype=torch.long)
    ]

    graph["event"].x = event_x[
        torch.tensor(local_event_ids, dtype=torch.long)
    ]

    graph["lender", "funds", "event"].edge_index = local_edges

    graph["event", "funded_by", "lender"].edge_index = (
        local_edges.flip(0).contiguous()
    )

    # The first local event node is the target
    graph["event"].target_mask = torch.zeros(
        len(local_event_ids),
        dtype=torch.bool
    )
    graph["event"].target_mask[0] = True

    return graph


sample_hetero_graph = neighbourhood_to_heterodata(
    sample_neighbourhood
)

print("SAMPLED HETEROGENEOUS GRAPH")
print("---------------------------")
print(sample_hetero_graph)

print("\nNode counts:")
print("Lenders:", sample_hetero_graph["lender"].num_nodes)
print("Events:", sample_hetero_graph["event"].num_nodes)

print("\nDirected edge counts:")
print(
    "Lender to event:",
    sample_hetero_graph["lender", "funds", "event"].num_edges
)
print(
    "Event to lender:",
    sample_hetero_graph["event", "funded_by", "lender"].num_edges
)

print("\nTarget event nodes:",
      sample_hetero_graph["event"].target_mask.sum().item())

assert sample_hetero_graph.validate(raise_on_error=True)
assert sample_hetero_graph["event"].target_mask.sum() == 1
assert sample_hetero_graph["event"].num_nodes == 238
assert sample_hetero_graph["lender"].num_nodes == 10

print("\nHeterogeneous graph construction successful!")


SAMPLED HETEROGENEOUS GRAPH
---------------------------
HeteroData(
  lender={ x=[10, 1] },
  event={
    x=[238, 11],
    target_mask=[238],
  },
  (lender, funds, event)={ edge_index=[2, 250] },
  (event, funded_by, lender)={ edge_index=[2, 250] }
)

Node counts:
Lenders: 10
Events: 238

Directed edge counts:
Lender to event: 250
Event to lender: 250

Target event nodes: 1

Heterogeneous graph construction successful!


### 3.24 Construction of a Sampled Heterogeneous Graph

The sampled historical neighbourhood of a borrowing event was converted into a PyTorch Geometric `HeteroData` object.

Two node types were represented: lenders and borrowing events. The event nodes included the target borrowing event and its sampled historical borrowing events.

Observed funding relationships were represented using two directed edge types: lender-to-event and event-to-lender. Local node indices were created to map the original graph identifiers into the smaller sampled graph.

The target event was identified using a Boolean mask, allowing it to be distinguished from historical context nodes.

The resulting heterogeneous graph was validated for node counts, edge counts, index consistency and structural correctness.

This implementation provides a preliminary graph representation for the temporal GraphSAGE architecture. It does not yet constitute a trained model or establish leakage-free evaluation across multiple target events.

In [57]:

# Step 56: Test heterogeneous GraphSAGE on Prosper data

import torch
import torch.nn.functional as F

from torch import nn
from torch_geometric.nn import HeteroConv, SAGEConv

torch.manual_seed(42)

class ProsperGraphSAGE(nn.Module):
    def __init__(self, hidden_channels=32):
        super().__init__()

        # Project different node feature dimensions
        # into a common hidden representation
        self.lender_encoder = nn.Linear(1, hidden_channels)
        self.event_encoder = nn.Linear(11, hidden_channels)

        # First message-passing layer
        self.conv1 = HeteroConv({
            ("lender", "funds", "event"):
                SAGEConv((-1, -1), hidden_channels),
            ("event", "funded_by", "lender"):
                SAGEConv((-1, -1), hidden_channels)
        }, aggr="sum")

        # Second message-passing layer
        self.conv2 = HeteroConv({
            ("lender", "funds", "event"):
                SAGEConv((-1, -1), hidden_channels),
            ("event", "funded_by", "lender"):
                SAGEConv((-1, -1), hidden_channels)
        }, aggr="sum")

        # Binary classification output
        self.classifier = nn.Linear(hidden_channels, 1)

    def forward(self, data):
        x_dict = {
            "lender": F.relu(
                self.lender_encoder(data["lender"].x)
            ),
            "event": F.relu(
                self.event_encoder(data["event"].x)
            )
        }

        x_dict = self.conv1(
            x_dict,
            data.edge_index_dict
        )

        x_dict = {
            key: F.relu(value)
            for key, value in x_dict.items()
        }

        x_dict = self.conv2(
            x_dict,
            data.edge_index_dict
        )

        # Only classify the target borrowing event
        target_embeddings = x_dict["event"][
            data["event"].target_mask
        ]

        logits = self.classifier(
            target_embeddings
        ).squeeze(-1)

        return logits


# Initialise the model
gnn_model = ProsperGraphSAGE(hidden_channels=32)

# Test without training
gnn_model.eval()

with torch.no_grad():
    sample_logits = gnn_model(sample_hetero_graph)
    sample_probabilities = torch.sigmoid(sample_logits)

print("PROSPER GRAPHSAGE FORWARD PASS")
print("------------------------------")
print("Output shape:", tuple(sample_logits.shape))
print("Raw prediction (logit):", sample_logits.item())
print("Predicted default probability:",
      sample_probabilities.item())

assert sample_logits.shape == (1,)
assert torch.isfinite(sample_logits).all()
assert 0 <= sample_probabilities.item() <= 1

print("\nGraphSAGE forward pass successful!")


PROSPER GRAPHSAGE FORWARD PASS
------------------------------
Output shape: (1,)
Raw prediction (logit): 0.10779798775911331
Predicted default probability: 0.526923418045044

GraphSAGE forward pass successful!


### 3.25 Initial Heterogeneous GraphSAGE Architecture

An initial two-layer heterogeneous GraphSAGE model was implemented using PyTorch and PyTorch Geometric.

The model accepts two node types: lenders and borrowing events. Separate linear transformations project their input features into a common hidden representation.

Two heterogeneous GraphSAGE convolution layers perform message passing across the observed lender–borrowing-event relationships.

The first layer allows event information to contribute to lender representations. The second layer allows information from connected lenders to contribute to borrowing-event representations.

A linear classification layer produces a single logit for the target borrowing event. Applying the sigmoid function converts this logit into a value between zero and one.

The architecture was designed to investigate whether relational information provides additional predictive value beyond the feature-based Logistic Regression baseline.

The initial forward-pass test checks computational compatibility and output validity only. The model remains untrained, and its predictions should not yet be interpreted as meaningful estimates of default risk.

Further work will involve constructing temporally restricted training batches, optimising model parameters and evaluating predictive performance using the established chronological partitions.

In [58]:

# Step 57: Test sensitivity to historical relational information

import torch
from torch_geometric.data import HeteroData

# Original graph with historical borrowing events
full_graph = sample_hetero_graph

# Create a graph with the same target and lenders,
# but without historical borrowing-event connections
direct_graph = HeteroData()

direct_graph["lender"].x = full_graph["lender"].x.clone()

# The target is local event node 0
direct_graph["event"].x = full_graph["event"].x[:1].clone()

direct_graph["event"].target_mask = torch.tensor(
    [True],
    dtype=torch.bool
)

# Keep only edges connecting lenders to the target event
full_edges = full_graph[
    "lender", "funds", "event"
].edge_index

target_edges = full_edges[
    :,
    full_edges[1] == 0
].clone()

direct_graph[
    "lender", "funds", "event"
].edge_index = target_edges

direct_graph[
    "event", "funded_by", "lender"
].edge_index = target_edges.flip(0).contiguous()

assert direct_graph.validate(raise_on_error=True)

# Compare outputs using identical model parameters
gnn_model.eval()

with torch.no_grad():
    full_logit = gnn_model(full_graph)
    direct_logit = gnn_model(direct_graph)

    full_probability = torch.sigmoid(full_logit).item()
    direct_probability = torch.sigmoid(direct_logit).item()

difference = abs(full_probability - direct_probability)

print("GRAPHSAGE RELATIONAL SENSITIVITY TEST")
print("-------------------------------------")

print("Original historical graph:")
print("  Event nodes:", full_graph["event"].num_nodes)
print("  Funding edges:", full_graph["lender", "funds", "event"].num_edges)
print("  Output:", round(full_probability, 6))

print("\nDirect-only graph:")
print("  Event nodes:", direct_graph["event"].num_nodes)
print("  Funding edges:", direct_graph["lender", "funds", "event"].num_edges)
print("  Output:", round(direct_probability, 6))

print("\nAbsolute output difference:", round(difference, 6))

assert torch.isfinite(full_logit).all()
assert torch.isfinite(direct_logit).all()

print("\nStructural sensitivity test completed!")


GRAPHSAGE RELATIONAL SENSITIVITY TEST
-------------------------------------
Original historical graph:
  Event nodes: 238
  Funding edges: 250
  Output: 0.526923

Direct-only graph:
  Event nodes: 1
  Funding edges: 10
  Output: 0.539493

Absolute output difference: 0.01257

Structural sensitivity test completed!


### 3.26 Structural Sensitivity of the Initial GraphSAGE Model

Following the successful GraphSAGE forward pass, a structural sensitivity test was prepared to examine whether historical borrowing-event relationships influence the model's output.

Two graph configurations were compared using the same randomly initialised model parameters.

The first configuration contained the target borrowing event, its participating lenders and sampled historical borrowing events.

The second configuration retained the same target event and participating lenders but removed the historical borrowing-event nodes and their connections.

The model outputs were compared to determine whether the presence of historical relational information affected the calculated score.

This experiment is an architectural verification rather than a predictive performance evaluation. A difference between the outputs does not demonstrate that relational information improves credit-risk assessment.

The predictive contribution of observed lender relationships will be assessed through model training, validation and comparison with the Logistic Regression baseline.

In [60]:

# Step 58: Verify GraphSAGE gradient computation
# CORRECTED VERSION — initialise lazy parameters first

import torch
import torch.nn.functional as F

torch.manual_seed(42)

# Create a separate disposable model
gradient_test_model = ProsperGraphSAGE(
    hidden_channels=32
)

# IMPORTANT: Initialise lazy GraphSAGE parameters
# before copying them or creating the optimiser
gradient_test_model.eval()

with torch.no_grad():
    _ = gradient_test_model(sample_hetero_graph)

# Now all parameters have been initialised
optimizer = torch.optim.Adam(
    gradient_test_model.parameters(),
    lr=0.001
)

# Obtain the actual label for the sample event
sample_label = event_y[
    sample_event_id
].float().reshape(1)

assert sample_label.item() in (0.0, 1.0)

# Switch to training mode
gradient_test_model.train()

# Save initial parameters
parameters_before = [
    p.detach().clone()
    for p in gradient_test_model.parameters()
]

# Perform one optimisation step
optimizer.zero_grad()

logits = gradient_test_model(
    sample_hetero_graph
)

loss = F.binary_cross_entropy_with_logits(
    logits,
    sample_label
)

loss.backward()

# Check gradient computation
gradient_count = sum(
    p.grad is not None
    for p in gradient_test_model.parameters()
)

# Update model parameters
optimizer.step()

# Verify that parameters changed
parameters_changed = any(
    not torch.equal(before, after)
    for before, after in zip(
        parameters_before,
        gradient_test_model.parameters()
    )
)

print("GRAPHSAGE GRADIENT TEST")
print("-----------------------")
print("Sample event ID:", sample_event_id)
print("Sample label:", int(sample_label.item()))
print("Initial training loss:", round(loss.item(), 6))
print("Parameters with gradients:", gradient_count)
print("Model parameters updated:", parameters_changed)

assert torch.isfinite(loss).item()
assert gradient_count > 0
assert parameters_changed

print("\nGraphSAGE gradient verification successful!")
print("This model will NOT be used for final evaluation.")


GRAPHSAGE GRADIENT TEST
-----------------------
Sample event ID: 17448
Sample label: 0
Initial training loss: 0.748498
Parameters with gradients: 15
Model parameters updated: True

GraphSAGE gradient verification successful!
This model will NOT be used for final evaluation.


### 3.27 Verification of GraphSAGE Gradient Computation

A preliminary training-step test was designed to verify that the heterogeneous GraphSAGE architecture supports gradient-based optimisation.

A separate, disposable GraphSAGE model was initialised for this purpose. The model processed a sampled historical neighbourhood and produced a prediction for one borrowing event.

Binary cross-entropy loss was calculated using the event's observed default label. Backpropagation was then performed to compute parameter gradients, followed by a single Adam optimiser update.

The test checks whether the loss is finite, gradients are produced and model parameters change after optimisation.

Because the selected example belongs to the validation partition, this procedure is treated strictly as a computational verification. The resulting model parameters will not be reused in the actual training, validation or testing experiments.

The final GraphSAGE model will be freshly initialised and trained exclusively using eligible training labels.

In [61]:

# Step 59: Prepare chronological event IDs for GraphSAGE

import numpy as np
import torch

# Retrieve event IDs using the existing chronological masks
train_event_ids = torch.where(
    train_node_mask
)[0].cpu().numpy().astype(np.int64)

validation_event_ids = torch.where(
    validation_node_mask
)[0].cpu().numpy().astype(np.int64)

test_event_ids = torch.where(
    test_node_mask
)[0].cpu().numpy().astype(np.int64)

# Check that the expected events were selected
assert len(train_event_ids) == 17361
assert len(validation_event_ids) == 4379
assert len(test_event_ids) == 5545

# Confirm that partitions do not overlap
assert len(
    np.intersect1d(train_event_ids, validation_event_ids)
) == 0

assert len(
    np.intersect1d(train_event_ids, test_event_ids)
) == 0

assert len(
    np.intersect1d(validation_event_ids, test_event_ids)
) == 0

# Confirm all selected events have valid labels
for partition_ids in [
    train_event_ids,
    validation_event_ids,
    test_event_ids
]:
    labels = event_y[
        torch.from_numpy(partition_ids)
    ]

    assert torch.all(
        (labels == 0) | (labels == 1)
    )

# Verify chronological boundaries
train_times = all_graph_events.iloc[
    train_event_ids
]["timestamp"].to_numpy()

validation_times = all_graph_events.iloc[
    validation_event_ids
]["timestamp"].to_numpy()

test_times = all_graph_events.iloc[
    test_event_ids
]["timestamp"].to_numpy()

assert train_times.max() < validation_times.min()
assert validation_times.max() < test_times.min()

print("GRAPHSAGE CHRONOLOGICAL TRAINING PARTITIONS")
print("------------------------------------------")
print("Training events:", len(train_event_ids))
print("Validation events:", len(validation_event_ids))
print("Test events:", len(test_event_ids))

print("\nChronological separation:")
print("Train before validation:",
      train_times.max() < validation_times.min())
print("Validation before test:",
      validation_times.max() < test_times.min())

print("\nPartition overlap: None")
print("All selected events have valid binary labels.")

print("\nGraphSAGE training partition preparation successful!")


GRAPHSAGE CHRONOLOGICAL TRAINING PARTITIONS
------------------------------------------
Training events: 17361
Validation events: 4379
Test events: 5545

Chronological separation:
Train before validation: True
Validation before test: True

Partition overlap: None
All selected events have valid binary labels.

GraphSAGE training partition preparation successful!


### 3.28 Preparation of Chronological GraphSAGE Training Partitions

Following successful verification of gradient computation, the supervised borrowing events were organised into chronological training, validation and test partitions.

The partitions were obtained from the Boolean node masks created during dataset preparation.

The training partition contains 17,361 borrowing events, the validation partition contains 4,379 events, and the test partition contains 5,545 events.

Checks were performed to confirm that the partitions do not overlap, all selected events have valid binary default labels, and the timestamps follow the required chronological ordering.

The training partition will be used to optimise GraphSAGE parameters. The validation partition will support model selection and threshold tuning, while the test partition will be reserved for final evaluation against the Logistic Regression baseline.

Historical graph context will be restricted to borrowing events occurring before each target event's timestamp. Events outside the supervised partitions may contribute historical features and relationships where chronologically eligible, but their labels will not be used for supervised training.

This procedure supports chronological evaluation, although the absence of verified repayment outcome availability dates remains a limitation for claims about real-time deployment.

In [62]:

# Step 60: Build an efficient event-to-lender lookup

import numpy as np

# Use the existing distinct lender-event relationships
event_lender_pairs = indexed_graph_edges[
    ["event_node_index", "lender_node_index"]
].to_numpy(dtype=np.int64)

# Sort relationships by event ID, then lender ID
sort_order = np.lexsort((
    event_lender_pairs[:, 1],
    event_lender_pairs[:, 0]
))

sorted_event_lender_pairs = event_lender_pairs[sort_order]

event_lookup_ids = sorted_event_lender_pairs[:, 0]
event_lookup_lenders = sorted_event_lender_pairs[:, 1]

# Count participating lenders for each borrowing event
event_lender_counts = np.bincount(
    event_lookup_ids,
    minlength=len(all_graph_events)
)

# Build offsets to locate each event's lenders
event_lender_offsets = np.zeros(
    len(all_graph_events) + 1,
    dtype=np.int64
)

event_lender_offsets[1:] = np.cumsum(
    event_lender_counts
)

def get_event_lenders(event_id):
    """
    Return the participating lender indices for an event
    using the precomputed lookup.
    """
    start = event_lender_offsets[int(event_id)]
    end = event_lender_offsets[int(event_id) + 1]

    return event_lookup_lenders[start:end]


# Validate against the previously constructed neighbourhood
retrieved_lenders = get_event_lenders(sample_event_id)

assert np.array_equal(
    retrieved_lenders,
    np.sort(sample_neighbourhood["target_lenders"])
)

assert event_lender_offsets[-1] == len(indexed_graph_edges)

assert len(event_lender_counts) == len(all_graph_events)

# Verify that all supervised events have participating lenders
supervised_ids = np.concatenate([
    train_event_ids,
    validation_event_ids,
    test_event_ids
])

assert np.all(event_lender_counts[supervised_ids] > 0)

print("FAST EVENT-TO-LENDER LOOKUP")
print("---------------------------")
print("Total borrowing events:", len(all_graph_events))
print("Indexed funding relationships:", len(event_lookup_lenders))
print("Lookup offset entries:", len(event_lender_offsets))

print("\nSample event:", sample_event_id)
print("Participating lenders:", len(retrieved_lenders))
print("Matches previous neighbourhood:",
      np.array_equal(
          retrieved_lenders,
          np.sort(sample_neighbourhood["target_lenders"])
      ))

print("\nLenders per supervised event:")
print(
    np.percentile(
        event_lender_counts[supervised_ids],
        [0, 25, 50, 75, 100]
    )
)

print("\nFast event-to-lender lookup successful!")


FAST EVENT-TO-LENDER LOOKUP
---------------------------
Total borrowing events: 43572
Indexed funding relationships: 3343284
Lookup offset entries: 43573

Sample event: 17448
Participating lenders: 10
Matches previous neighbourhood: True

Lenders per supervised event:
[  1.  24.  47.  90. 458.]

Fast event-to-lender lookup successful!


### 3.29 Efficient Event-to-Lender Indexing

An indexed lookup structure was created to improve the efficiency of retrieving lender relationships for individual borrowing events.

The distinct lender–event relationships were sorted by borrowing-event identifier and lender identifier. The number of participating lenders was calculated for each event, and cumulative offsets were generated to identify the corresponding positions in the sorted relationship array.

A lookup function was implemented to retrieve the participating lenders for a specified borrowing event without repeatedly filtering the full relationship dataset.

The lookup was validated against the previously constructed sample neighbourhood. Additional checks confirmed that the indexed relationship count matched the original graph and that all supervised borrowing events had participating lenders.

This indexing procedure is intended to reduce repeated data-processing overhead during mini-batch construction for the temporal GraphSAGE model.

The procedure changes only how relationships are retrieved. It does not modify the underlying observations, supervised labels or chronological data partitions.

In [63]:

# Step 61: Bounded temporal neighbourhood sampling

import numpy as np

MAX_TARGET_LENDERS = 16
MAX_HISTORICAL_NEIGHBOURS = 32
SAMPLING_SEED = 42

def build_bounded_event_neighbourhood(
    target_event_id,
    max_lenders=MAX_TARGET_LENDERS,
    max_historical=MAX_HISTORICAL_NEIGHBOURS,
    seed=SAMPLING_SEED
):
    """
    Build a bounded historical neighbourhood for one target event.

    All historical events must occur strictly before the
    target event's timestamp.
    """

    target_event_id = int(target_event_id)

    target_timestamp = int(
        all_graph_events.iloc[target_event_id]["timestamp"]
    )

    # Fast lookup of current participating lenders
    participating_lenders = get_event_lenders(
        target_event_id
    )

    # Reproducibly sample participating lenders if necessary
    if len(participating_lenders) > max_lenders:
        rng = np.random.default_rng(
            np.random.SeedSequence([
                seed,
                target_event_id
            ])
        )

        selected_lenders = np.sort(
            rng.choice(
                participating_lenders,
                size=max_lenders,
                replace=False
            )
        )
    else:
        selected_lenders = participating_lenders.copy()

    # Retrieve earlier events for each selected lender
    historical_edges = []

    for lender in selected_lenders:
        prior_events = sample_prior_lender_events(
            lender_index=int(lender),
            target_timestamp=target_timestamp,
            max_neighbours=max_historical,
            seed=seed
        )

        for prior_event in prior_events:
            historical_edges.append(
                (int(lender), int(prior_event))
            )

    historical_edges = np.asarray(
        historical_edges,
        dtype=np.int64
    ).reshape(-1, 2)

    historical_event_ids = np.unique(
        historical_edges[:, 1]
    )

    return {
        "target_event_id": target_event_id,
        "target_timestamp": target_timestamp,
        "target_lenders": selected_lenders,
        "historical_edges": historical_edges,
        "historical_event_ids": historical_event_ids
    }


# Test on an event with many participating lenders
largest_event_id = int(
    supervised_ids[
        np.argmax(event_lender_counts[supervised_ids])
    ]
)

large_neighbourhood = build_bounded_event_neighbourhood(
    largest_event_id
)

# Rebuild to verify reproducibility
repeated_neighbourhood = build_bounded_event_neighbourhood(
    largest_event_id
)

assert np.array_equal(
    large_neighbourhood["target_lenders"],
    repeated_neighbourhood["target_lenders"]
)

assert np.array_equal(
    large_neighbourhood["historical_edges"],
    repeated_neighbourhood["historical_edges"]
)

assert len(large_neighbourhood["target_lenders"]) <= 16

assert len(large_neighbourhood["historical_edges"]) <= 512

# Check chronological restrictions
historical_ids = large_neighbourhood["historical_event_ids"]

if len(historical_ids) > 0:
    historical_timestamps = all_graph_events.iloc[
        historical_ids
    ]["timestamp"].to_numpy()

    assert np.all(
        historical_timestamps <
        large_neighbourhood["target_timestamp"]
    )

    assert largest_event_id not in historical_ids

print("BOUNDED TEMPORAL NEIGHBOURHOOD SAMPLING")
print("---------------------------------------")
print("Target event ID:", largest_event_id)
print("Original participating lenders:",
      event_lender_counts[largest_event_id])
print("Sampled participating lenders:",
      len(large_neighbourhood["target_lenders"]))
print("Sampled historical edges:",
      len(large_neighbourhood["historical_edges"]))
print("Unique historical events:",
      len(large_neighbourhood["historical_event_ids"]))

print("\nMaximum target lenders:", MAX_TARGET_LENDERS)
print("Maximum history per lender:", MAX_HISTORICAL_NEIGHBOURS)
print("Maximum historical edges:",
      MAX_TARGET_LENDERS * MAX_HISTORICAL_NEIGHBOURS)

print("\nReproducibility and chronological checks passed!")


BOUNDED TEMPORAL NEIGHBOURHOOD SAMPLING
---------------------------------------
Target event ID: 18807
Original participating lenders: 458
Sampled participating lenders: 16
Sampled historical edges: 320
Unique historical events: 266

Maximum target lenders: 16
Maximum history per lender: 32
Maximum historical edges: 512

Reproducibility and chronological checks passed!


### 3.30 Bounded Temporal Neighbourhood Sampling

A bounded neighbourhood sampling procedure was introduced to support computationally feasible training of the GraphSAGE model on a CPU.

The procedure first retrieves the observed participating lenders for a target borrowing event. Where more than 16 lenders are present, a reproducible random sample of 16 lenders is selected.

For each selected lender, up to 32 historical borrowing events are sampled from relationships occurring strictly before the target event's timestamp.

This produces a maximum of 512 sampled historical lender–event relationships per target borrowing event, excluding the target's own funding relationships.

A fixed random seed is used to make neighbourhood construction reproducible. Validation checks confirm the sampling limits, reproducibility and chronological restrictions.

The sampling configuration represents an initial computational design choice rather than an optimised setting. Its potential effect on predictive performance and representation of high-degree borrowing events will be considered during evaluation.

The resulting neighbourhoods will be processed separately for each target event to preserve target-specific historical cutoffs and prevent cross-target message passing from introducing later relational information.

In [64]:

# Step 62: Construct a temporally isolated GraphSAGE mini-batch

import numpy as np
import torch
from torch_geometric.data import Batch

def build_graphsage_batch(event_ids):
    """
    Construct a batch of independent, time-restricted
    heterogeneous neighbourhoods.
    """

    graphs = []

    for event_id in event_ids:
        neighbourhood = build_bounded_event_neighbourhood(
            int(event_id)
        )

        graph = neighbourhood_to_heterodata(
            neighbourhood
        )

        # Store the target event's supervised label
        # separately from the input node features
        graph["event"].target_y = event_y[
            int(event_id)
        ].reshape(1).float()

        graphs.append(graph)

    # PyG offsets local indices and keeps graphs disconnected
    batch = Batch.from_data_list(graphs)

    return batch


# Select a small training-only batch
sample_train_ids = train_event_ids[:4]

sample_train_batch = build_graphsage_batch(
    sample_train_ids
)

print("GRAPHSAGE MINI-BATCH CONSTRUCTION")
print("---------------------------------")
print("Training event IDs:", sample_train_ids.tolist())
print("Number of graphs:", sample_train_batch.num_graphs)

print("\nBatched graph:")
print(sample_train_batch)

print("\nTotal lender nodes:",
      sample_train_batch["lender"].num_nodes)

print("Total event nodes:",
      sample_train_batch["event"].num_nodes)

print("Total target nodes:",
      sample_train_batch["event"].target_mask.sum().item())

print("Target labels:",
      sample_train_batch["event"].target_y.tolist())

# Validation checks
assert sample_train_batch.num_graphs == 4

assert (
    sample_train_batch["event"].target_mask.sum().item()
    == 4
)

assert sample_train_batch["event"].target_y.numel() == 4

assert sample_train_batch.validate(raise_on_error=True)

# Verify graph isolation: every edge must connect nodes
# belonging to the same sampled neighbourhood
for edge_type in sample_train_batch.edge_types:
    source_type, _, destination_type = edge_type

    edge_index = sample_train_batch[edge_type].edge_index

    source_graph_ids = sample_train_batch[
        source_type
    ].batch[edge_index[0]]

    destination_graph_ids = sample_train_batch[
        destination_type
    ].batch[edge_index[1]]

    assert torch.equal(
        source_graph_ids,
        destination_graph_ids
    ), f"Cross-graph connection detected: {edge_type}"

# Verify that all four events belong to training
assert np.all(
    np.isin(sample_train_ids, train_event_ids)
)

print("\nAll neighbourhoods are isolated.")
print("All target labels belong to training.")
print("\nGraphSAGE mini-batch construction successful!")


GRAPHSAGE MINI-BATCH CONSTRUCTION
---------------------------------
Training event IDs: [22, 23, 24, 25]
Number of graphs: 4

Batched graph:
HeteroDataBatch(
  lender={
    x=[20, 1],
    batch=[20],
    ptr=[5],
  },
  event={
    x=[95, 11],
    target_mask=[95],
    target_y=[4],
    batch=[95],
    ptr=[5],
  },
  (lender, funds, event)={ edge_index=[2, 198] },
  (event, funded_by, lender)={ edge_index=[2, 198] }
)

Total lender nodes: 20
Total event nodes: 95
Total target nodes: 4
Target labels: [0.0, 0.0, 0.0, 0.0]

All neighbourhoods are isolated.
All target labels belong to training.

GraphSAGE mini-batch construction successful!


### 3.31 Construction of Temporally Isolated GraphSAGE Mini-Batches

A mini-batch construction function was implemented to prepare multiple borrowing events for GraphSAGE training.

For each target borrowing event, a bounded historical neighbourhood is constructed using the event's own timestamp as the chronological cutoff. The neighbourhood is then converted into a heterogeneous PyTorch Geometric graph containing lender nodes, borrowing-event nodes and observed funding relationships.

Each graph includes a Boolean mask identifying its target borrowing event. The corresponding binary default label is stored separately from the node input features.

Multiple graphs are combined using the PyTorch Geometric `Batch` class. Local node indices are adjusted automatically, while the individual neighbourhoods remain disconnected.

This design prevents graph message passing between target events with different historical cutoffs, even when their underlying lender identities overlap.

Validation checks confirm the number of graphs, target-node count, label count, graph structure, absence of cross-graph edges and membership of the sampled events in the training partition.

The batching procedure supports controlled temporal message passing and computationally manageable model training. However, it does not resolve limitations associated with unavailable repayment outcome observation dates.

In [65]:

# Step 63: Test GraphSAGE on a mixed-label training batch

import numpy as np
import torch

# Identify training events belonging to each class
train_labels = event_y[
    torch.from_numpy(train_event_ids)
].cpu().numpy()

nondefault_train_ids = train_event_ids[
    train_labels == 0
]

default_train_ids = train_event_ids[
    train_labels == 1
]

# Select two events from each class
mixed_train_ids = np.array([
    nondefault_train_ids[0],
    default_train_ids[0],
    nondefault_train_ids[1],
    default_train_ids[1]
], dtype=np.int64)

# Construct independent historical neighbourhoods
mixed_train_batch = build_graphsage_batch(
    mixed_train_ids
)

# Initialise a fresh model
torch.manual_seed(42)

batch_test_model = ProsperGraphSAGE(
    hidden_channels=32
)

batch_test_model.eval()

with torch.no_grad():
    batch_logits = batch_test_model(
        mixed_train_batch
    )

    batch_probabilities = torch.sigmoid(
        batch_logits
    )

batch_labels = mixed_train_batch[
    "event"
].target_y

print("GRAPHSAGE MIXED-LABEL BATCH FORWARD PASS")
print("----------------------------------------")

print("Number of graphs:", mixed_train_batch.num_graphs)
print("Output shape:", tuple(batch_logits.shape))

print("\nIndividual predictions:")

for event_id, label, probability in zip(
    mixed_train_ids,
    batch_labels,
    batch_probabilities
):
    print(
        f"Event {event_id} | "
        f"Actual label: {int(label.item())} | "
        f"Untrained output: {probability.item():.6f}"
    )

# Validation checks
assert mixed_train_batch.num_graphs == 4
assert batch_logits.shape == (4,)
assert batch_labels.shape == (4,)

assert torch.isfinite(batch_logits).all()
assert torch.all(
    (batch_probabilities >= 0) &
    (batch_probabilities <= 1)
)

assert sorted(batch_labels.tolist()) == [
    0.0, 0.0, 1.0, 1.0
]

print("\nMixed-label batch forward pass successful!")


GRAPHSAGE MIXED-LABEL BATCH FORWARD PASS
----------------------------------------
Number of graphs: 4
Output shape: (4,)

Individual predictions:
Event 22 | Actual label: 0 | Untrained output: 0.538761
Event 28 | Actual label: 1 | Untrained output: 0.543596
Event 23 | Actual label: 0 | Untrained output: 0.541277
Event 32 | Actual label: 1 | Untrained output: 0.551455

Mixed-label batch forward pass successful!


### 3.32 Verification of Mixed-Label GraphSAGE Mini-Batch Processing

A mixed-label mini-batch was prepared to verify that the heterogeneous GraphSAGE architecture could process multiple supervised borrowing events simultaneously.

Four borrowing events were selected from the chronological training partition, comprising two non-default events and two default events.

A separate time-restricted historical neighbourhood was constructed for each target event. The neighbourhoods were combined into a disconnected PyTorch Geometric mini-batch.

A freshly initialised GraphSAGE model performed a forward pass and generated one prediction logit for each target borrowing event. Sigmoid activation was applied to obtain numerical outputs between zero and one.

Validation checks confirmed the number of graphs, output dimensions, finite predictions and presence of both target classes.

This test verifies compatibility between the mini-batch construction procedure and the GraphSAGE architecture. Because the model was not trained, the outputs do not represent validated estimates of default risk or predictive performance.

In [66]:

# Step 64: Initialise the final-experiment GraphSAGE model

import torch
import torch.nn.functional as F

from torch import nn

# Reproducibility
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)

# Initial training configuration
GNN_HIDDEN_CHANNELS = 32
GNN_LEARNING_RATE = 0.001
GNN_BATCH_SIZE = 16
GNN_MAX_EPOCHS = 10

# CPU execution
gnn_device = torch.device("cpu")

# Create a completely fresh model
training_gnn_model = ProsperGraphSAGE(
    hidden_channels=GNN_HIDDEN_CHANNELS
).to(gnn_device)

# Initialise lazy parameters using TRAINING data only
training_gnn_model.eval()

with torch.no_grad():
    _ = training_gnn_model(
        mixed_train_batch.to(gnn_device)
    )

# Create optimiser after parameter initialisation
gnn_optimizer = torch.optim.Adam(
    training_gnn_model.parameters(),
    lr=GNN_LEARNING_RATE
)

# Define binary classification loss
gnn_loss_function = nn.BCEWithLogitsLoss()

# Count trainable parameters
trainable_parameters = sum(
    parameter.numel()
    for parameter in training_gnn_model.parameters()
    if parameter.requires_grad
)

print("GRAPHSAGE TRAINING CONFIGURATION")
print("--------------------------------")
print("Device:", gnn_device)
print("Hidden dimensions:", GNN_HIDDEN_CHANNELS)
print("Learning rate:", GNN_LEARNING_RATE)
print("Batch size:", GNN_BATCH_SIZE)
print("Maximum epochs:", GNN_MAX_EPOCHS)
print("Optimiser: Adam")
print("Loss function: BCEWithLogitsLoss")
print("Random seed:", RANDOM_SEED)
print("Trainable parameters:", trainable_parameters)

print("\nSupervised event counts:")
print("Training:", len(train_event_ids))
print("Validation:", len(validation_event_ids))
print("Test:", len(test_event_ids))

assert trainable_parameters > 0
assert len(train_event_ids) == 17361
assert len(validation_event_ids) == 4379
assert len(test_event_ids) == 5545

print("\nFresh GraphSAGE training model initialised successfully!")


GRAPHSAGE TRAINING CONFIGURATION
--------------------------------
Device: cpu
Hidden dimensions: 32
Learning rate: 0.001
Batch size: 16
Maximum epochs: 10
Optimiser: Adam
Loss function: BCEWithLogitsLoss
Random seed: 42
Trainable parameters: 8801

Supervised event counts:
Training: 17361
Validation: 4379
Test: 5545

Fresh GraphSAGE training model initialised successfully!


### 3.33 Initialisation of the GraphSAGE Training Configuration

Following successful mixed-label mini-batch verification, a fresh heterogeneous GraphSAGE model was initialised for the supervised training experiment.

The initial architecture uses a hidden dimension of 32. The Adam optimiser was selected with a learning rate of 0.001, together with binary cross-entropy loss calculated from prediction logits.

An initial mini-batch size of 16 and a maximum of 10 training epochs were selected to support CPU-based experimentation. A random seed of 42 was specified to improve reproducibility.

Because the GraphSAGE convolution layers use lazy parameter initialisation, a forward pass was performed using training-partition data before constructing the optimiser.

The new model was initialised independently of the earlier architectural and gradient tests. No validation or test labels were used to optimise its parameters during initialisation.

The selected hyperparameters are preliminary and will be assessed using validation results. The test partition will remain reserved for final model evaluation.

In [67]:

# Step 65: Test GraphSAGE training on 64 training events

import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(42)

# Reproducible training-only subset
rng = np.random.default_rng(42)

small_train_ids = rng.choice(
    train_event_ids,
    size=64,
    replace=False
)

# Create a disposable model for the training test
small_test_model = ProsperGraphSAGE(
    hidden_channels=GNN_HIDDEN_CHANNELS
).to(gnn_device)

# Initialise lazy parameters using training data
small_test_model.eval()

with torch.no_grad():
    _ = small_test_model(
        mixed_train_batch.to(gnn_device)
    )

small_optimizer = torch.optim.Adam(
    small_test_model.parameters(),
    lr=GNN_LEARNING_RATE
)

small_loss_function = nn.BCEWithLogitsLoss()

# Train for one diagnostic epoch
small_test_model.train()

batch_losses = []

for start in range(0, len(small_train_ids), GNN_BATCH_SIZE):

    batch_ids = small_train_ids[
        start:start + GNN_BATCH_SIZE
    ]

    batch = build_graphsage_batch(
        batch_ids
    ).to(gnn_device)

    labels = batch["event"].target_y

    small_optimizer.zero_grad()

    logits = small_test_model(batch)

    loss = small_loss_function(
        logits,
        labels
    )

    loss.backward()
    small_optimizer.step()

    batch_losses.append(loss.item())

    print(
        f"Batch {len(batch_losses)} | "
        f"Events: {len(batch_ids)} | "
        f"Loss: {loss.item():.6f}"
    )

average_loss = float(np.mean(batch_losses))

print("\nSMALL-SCALE GRAPHSAGE TRAINING TEST")
print("----------------------------------")
print("Training events:", len(small_train_ids))
print("Mini-batches:", len(batch_losses))
print("Average training loss:", round(average_loss, 6))

assert len(batch_losses) == 4
assert np.all(np.isfinite(batch_losses))
assert np.isfinite(average_loss)

print("\nSmall-scale GraphSAGE training successful!")
print("The full training model remains untouched.")


Batch 1 | Events: 16 | Loss: 0.705372
Batch 2 | Events: 16 | Loss: 0.708027
Batch 3 | Events: 16 | Loss: 0.697010
Batch 4 | Events: 16 | Loss: 0.681056

SMALL-SCALE GRAPHSAGE TRAINING TEST
----------------------------------
Training events: 64
Mini-batches: 4
Average training loss: 0.697866

Small-scale GraphSAGE training successful!
The full training model remains untouched.


### 3.34 Small-Scale Verification of GraphSAGE Training

A preliminary training experiment was designed to verify the end-to-end GraphSAGE optimisation procedure before full-scale training.

A reproducible random sample of 64 borrowing events was selected exclusively from the chronological training partition.

The selected events were processed in four mini-batches of 16. Each mini-batch contained independently constructed, time-restricted historical neighbourhoods.

A separate disposable GraphSAGE model was initialised and trained for one diagnostic epoch using the Adam optimiser and binary cross-entropy loss.

For each mini-batch, the procedure performed a forward pass, calculated the supervised loss, computed gradients through backpropagation and updated the model parameters.

Batch-level and average training losses were recorded to verify numerical stability and successful execution.

This diagnostic experiment is not used to report final predictive performance. The full GraphSAGE training model remains separate and will be trained using the complete training partition after the training procedure has been validated.

In [68]:

# Step 66: Define a reusable GraphSAGE training function

import numpy as np
import torch

def train_graphsage_epoch(
    model,
    optimizer,
    event_ids,
    batch_size=GNN_BATCH_SIZE,
    seed=42
):
    """
    Train GraphSAGE for one epoch using training events.

    Each event is processed once per epoch.
    Historical neighbourhoods are independently constructed
    with target-specific chronological cutoffs.
    """

    event_ids = np.asarray(event_ids, dtype=np.int64)

    # Prevent accidental training on validation or test labels
    assert np.all(
        np.isin(event_ids, train_event_ids)
    ), "Training function received non-training event IDs."

    assert len(event_ids) > 0
    assert len(np.unique(event_ids)) == len(event_ids)
    assert batch_size > 0

    # Shuffle training events reproducibly
    rng = np.random.default_rng(seed)
    shuffled_ids = rng.permutation(event_ids)

    model.train()

    total_loss = 0.0
    total_events = 0
    batch_count = 0

    for start in range(0, len(shuffled_ids), batch_size):

        batch_ids = shuffled_ids[
            start:start + batch_size
        ]

        batch = build_graphsage_batch(
            batch_ids
        ).to(gnn_device)

        labels = batch["event"].target_y

        optimizer.zero_grad()

        logits = model(batch)

        assert logits.shape == labels.shape

        loss = gnn_loss_function(
            logits,
            labels
        )

        if not torch.isfinite(loss).item():
            raise ValueError(
                "Non-finite training loss detected."
            )

        loss.backward()

        optimizer.step()

        # Weight loss by the number of target events
        total_loss += loss.item() * len(batch_ids)
        total_events += len(batch_ids)
        batch_count += 1

    average_loss = total_loss / total_events

    return {
        "average_loss": float(average_loss),
        "events_processed": int(total_events),
        "batches_processed": int(batch_count)
    }


print("REUSABLE GRAPHSAGE TRAINING FUNCTION")
print("------------------------------------")
print("Function defined:", callable(train_graphsage_epoch))
print("Default batch size:", GNN_BATCH_SIZE)
print("Training partition size:", len(train_event_ids))

assert callable(train_graphsage_epoch)

print("\nGraphSAGE epoch-training function ready!")
print("No training has been executed in this step.")


REUSABLE GRAPHSAGE TRAINING FUNCTION
------------------------------------
Function defined: True
Default batch size: 16
Training partition size: 17361

GraphSAGE epoch-training function ready!
No training has been executed in this step.


### 3.35 Reusable GraphSAGE Epoch-Training Function

A reusable training function was developed to perform one epoch of supervised GraphSAGE optimisation.

The function accepts a model, optimiser, training-event identifiers, mini-batch size and random seed.

Before training, it verifies that all supplied borrowing events belong to the established chronological training partition. It also checks for duplicate event identifiers and invalid batch sizes.

Training events are shuffled reproducibly and processed in mini-batches. Each mini-batch is constructed from independent, time-restricted historical neighbourhoods.

For each mini-batch, the model produces prediction logits, binary cross-entropy loss is calculated, gradients are computed and the Adam optimiser updates the model parameters.

The function records the total number of processed events and mini-batches and calculates an event-weighted average training loss.

The function was defined without executing a training epoch. Subsequent steps will implement validation monitoring and test the complete training workflow before full-scale experimentation.

In [69]:

# Step 67: Define a reusable GraphSAGE validation function

import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

def evaluate_graphsage(
    model,
    event_ids,
    batch_size=GNN_BATCH_SIZE,
    threshold=0.5
):
    """
    Evaluate GraphSAGE without updating model parameters.

    Uses independent, time-restricted neighbourhoods.
    Returns loss, classification metrics and predictions.
    """

    event_ids = np.asarray(event_ids, dtype=np.int64)

    assert len(event_ids) > 0
    assert len(np.unique(event_ids)) == len(event_ids)
    assert batch_size > 0
    assert 0 <= threshold <= 1

    # Permit training or validation evaluation only.
    # Keep the test partition untouched until final evaluation.
    permitted_ids = np.concatenate([
        train_event_ids,
        validation_event_ids
    ])

    assert np.all(
        np.isin(event_ids, permitted_ids)
    ), "Evaluation contains events outside train/validation."

    model.eval()

    all_labels = []
    all_probabilities = []

    total_loss = 0.0
    total_events = 0

    with torch.no_grad():

        for start in range(0, len(event_ids), batch_size):

            batch_ids = event_ids[
                start:start + batch_size
            ]

            batch = build_graphsage_batch(
                batch_ids
            ).to(gnn_device)

            labels = batch["event"].target_y

            logits = model(batch)

            assert logits.shape == labels.shape

            loss = gnn_loss_function(
                logits,
                labels
            )

            probabilities = torch.sigmoid(
                logits
            )

            total_loss += loss.item() * len(batch_ids)
            total_events += len(batch_ids)

            all_labels.extend(
                labels.cpu().numpy().tolist()
            )

            all_probabilities.extend(
                probabilities.cpu().numpy().tolist()
            )

    y_true = np.asarray(all_labels, dtype=np.int64)

    y_prob = np.asarray(
        all_probabilities,
        dtype=np.float64
    )

    y_pred = (y_prob >= threshold).astype(np.int64)

    # ROC-AUC requires both classes
    roc_auc = (
        roc_auc_score(y_true, y_prob)
        if len(np.unique(y_true)) == 2
        else np.nan
    )

    average_precision = (
        average_precision_score(y_true, y_prob)
        if np.any(y_true == 1)
        else np.nan
    )

    return {
        "loss": total_loss / total_events,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "f1": f1_score(
            y_true, y_pred, zero_division=0
        ),
        "roc_auc": roc_auc,
        "average_precision": average_precision,
        "y_true": y_true,
        "y_prob": y_prob,
        "y_pred": y_pred
    }


print("GRAPHSAGE VALIDATION FUNCTION")
print("-----------------------------")
print("Function defined:", callable(evaluate_graphsage))
print("Default batch size:", GNN_BATCH_SIZE)
print("Default threshold:", 0.5)
print("Validation events available:", len(validation_event_ids))

assert callable(evaluate_graphsage)

print("\nGraphSAGE validation function ready!")
print("No validation has been executed yet.")


GRAPHSAGE VALIDATION FUNCTION
-----------------------------
Function defined: True
Default batch size: 16
Default threshold: 0.5
Validation events available: 4379

GraphSAGE validation function ready!
No validation has been executed yet.


### 3.36 Reusable GraphSAGE Validation Function

A reusable evaluation function was developed to assess GraphSAGE performance without updating model parameters.

The function processes borrowing events in mini-batches using independently constructed, time-restricted historical neighbourhoods.

During evaluation, the model is placed in evaluation mode and gradient computation is disabled.

For each mini-batch, prediction logits are generated and converted into probability-like scores using the sigmoid function. Binary cross-entropy loss is calculated, and predictions are compared with the observed default labels.

The function returns validation loss, accuracy, precision, recall, F1-score, ROC-AUC and average precision. It also preserves the true labels, predicted scores and binary predictions for further analysis.

A default classification threshold of 0.5 is used initially. Threshold selection will be performed later using validation data.

The evaluation function is restricted to training and validation events, preserving the test partition for final model assessment.

This step defines the evaluation procedure only; no validation experiment has yet been executed.

In [70]:

# Step 68: End-to-end GraphSAGE training and validation test

import numpy as np
import torch

torch.manual_seed(42)

# Select reproducible subsets from the correct partitions
diagnostic_rng = np.random.default_rng(42)

diagnostic_train_ids = diagnostic_rng.choice(
    train_event_ids,
    size=128,
    replace=False
)

diagnostic_validation_ids = diagnostic_rng.choice(
    validation_event_ids,
    size=64,
    replace=False
)

# Create a disposable model
diagnostic_model = ProsperGraphSAGE(
    hidden_channels=GNN_HIDDEN_CHANNELS
).to(gnn_device)

# Initialise lazy parameters with training data only
diagnostic_model.eval()

with torch.no_grad():
    _ = diagnostic_model(
        mixed_train_batch.to(gnn_device)
    )

diagnostic_optimizer = torch.optim.Adam(
    diagnostic_model.parameters(),
    lr=GNN_LEARNING_RATE
)

# Train for one diagnostic epoch
training_results = train_graphsage_epoch(
    model=diagnostic_model,
    optimizer=diagnostic_optimizer,
    event_ids=diagnostic_train_ids,
    batch_size=GNN_BATCH_SIZE,
    seed=42
)

# Save parameters immediately before validation
parameters_before_validation = [
    parameter.detach().clone()
    for parameter in diagnostic_model.parameters()
]

# Evaluate on later validation events
validation_results = evaluate_graphsage(
    model=diagnostic_model,
    event_ids=diagnostic_validation_ids,
    batch_size=GNN_BATCH_SIZE,
    threshold=0.5
)

# Verify validation did not modify model parameters
parameters_unchanged = all(
    torch.equal(before, after)
    for before, after in zip(
        parameters_before_validation,
        diagnostic_model.parameters()
    )
)

print("GRAPHSAGE END-TO-END DIAGNOSTIC")
print("--------------------------------")
print("Training events:", training_results["events_processed"])
print("Training batches:", training_results["batches_processed"])
print("Average training loss:",
      round(training_results["average_loss"], 6))

print("\nValidation results:")
print("Validation events:", len(diagnostic_validation_ids))

for metric in [
    "loss",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "average_precision"
]:
    print(
        f"{metric}: {validation_results[metric]:.6f}"
    )

print("\nModel parameters unchanged during validation:",
      parameters_unchanged)

assert training_results["events_processed"] == 128
assert training_results["batches_processed"] == 8
assert len(validation_results["y_true"]) == 64
assert np.all(np.isfinite(validation_results["y_prob"]))
assert parameters_unchanged

print("\nEnd-to-end GraphSAGE diagnostic successful!")
print("The full training model remains untouched.")


GRAPHSAGE END-TO-END DIAGNOSTIC
--------------------------------
Training events: 128
Training batches: 8
Average training loss: 0.689263

Validation results:
Validation events: 64
loss: 0.661550
accuracy: 0.625000
precision: 0.000000
recall: 0.000000
f1: 0.000000
roc_auc: 0.571875
average_precision: 0.525775

Model parameters unchanged during validation: True

End-to-end GraphSAGE diagnostic successful!
The full training model remains untouched.


### 3.37 End-to-End GraphSAGE Training and Validation Diagnostic

An end-to-end diagnostic experiment was prepared to verify the integration of the GraphSAGE training and validation functions.

A reproducible subset of 128 borrowing events was selected from the chronological training partition, together with 64 events from the later validation partition.

A separate GraphSAGE model was initialised and trained for one diagnostic epoch using mini-batches of 16 events.

After training, the model was evaluated on the selected validation events without gradient computation or parameter updates.

The diagnostic records training loss and validation loss, accuracy, precision, recall, F1-score, ROC-AUC and average precision.

An additional parameter-consistency check verifies that model weights remain unchanged during validation.

This experiment is intended to identify implementation problems before full-scale training. The small subset metrics are not treated as final research findings or directly compared with the full Logistic Regression baseline.

The full GraphSAGE training model remains separate and will be used only after the diagnostic procedure has been verified.

In [71]:

# Step 69: Define full GraphSAGE training with validation monitoring

import numpy as np
import pandas as pd
import torch
import copy
import time

def run_graphsage_training(
    model,
    optimizer,
    train_ids,
    validation_ids,
    epochs=GNN_MAX_EPOCHS,
    batch_size=GNN_BATCH_SIZE,
    seed=42
):
    """
    Train GraphSAGE over multiple epochs.

    Tracks training loss and validation metrics.
    Saves the best model state using validation ROC-AUC.
    """

    history = []
    best_validation_auc = -np.inf
    best_model_state = None
    best_epoch = None

    for epoch in range(1, epochs + 1):

        start_time = time.perf_counter()

        # Train on chronological training partition
        train_results = train_graphsage_epoch(
            model=model,
            optimizer=optimizer,
            event_ids=train_ids,
            batch_size=batch_size,
            seed=seed + epoch
        )

        # Evaluate on chronological validation partition
        val_results = evaluate_graphsage(
            model=model,
            event_ids=validation_ids,
            batch_size=batch_size,
            threshold=0.5
        )

        epoch_seconds = time.perf_counter() - start_time

        row = {
            "epoch": epoch,
            "train_loss": train_results["average_loss"],
            "val_loss": val_results["loss"],
            "val_accuracy": val_results["accuracy"],
            "val_precision": val_results["precision"],
            "val_recall": val_results["recall"],
            "val_f1": val_results["f1"],
            "val_roc_auc": val_results["roc_auc"],
            "val_average_precision": val_results["average_precision"],
            "epoch_seconds": epoch_seconds
        }

        history.append(row)

        # Select the best epoch using validation ROC-AUC
        current_auc = val_results["roc_auc"]

        if np.isfinite(current_auc) and current_auc > best_validation_auc:
            best_validation_auc = current_auc
            best_epoch = epoch
            best_model_state = copy.deepcopy(
                model.state_dict()
            )

        print(
            f"Epoch {epoch}/{epochs} | "
            f"Train Loss: {row['train_loss']:.4f} | "
            f"Val Loss: {row['val_loss']:.4f} | "
            f"Val AUC: {row['val_roc_auc']:.4f} | "
            f"Val AP: {row['val_average_precision']:.4f} | "
            f"Time: {epoch_seconds:.1f}s",
            flush=True
        )

    history_df = pd.DataFrame(history)

    return {
        "history": history_df,
        "best_epoch": best_epoch,
        "best_validation_auc": best_validation_auc,
        "best_model_state": best_model_state
    }


print("FULL GRAPHSAGE TRAINING FUNCTION")
print("--------------------------------")
print("Function defined:", callable(run_graphsage_training))
print("Maximum epochs:", GNN_MAX_EPOCHS)
print("Batch size:", GNN_BATCH_SIZE)
print("Model selection metric: Validation ROC-AUC")
print("Best model checkpoint: In-memory state dictionary")

assert callable(run_graphsage_training)

print("\nMulti-epoch GraphSAGE training function ready!")
print("No full training has been executed.")


FULL GRAPHSAGE TRAINING FUNCTION
--------------------------------
Function defined: True
Maximum epochs: 10
Batch size: 16
Model selection metric: Validation ROC-AUC
Best model checkpoint: In-memory state dictionary

Multi-epoch GraphSAGE training function ready!
No full training has been executed.


### 3.38 Multi-Epoch GraphSAGE Training and Validation Monitoring

Following successful end-to-end diagnostic testing, a multi-epoch training procedure was defined for the heterogeneous GraphSAGE model.

The procedure performs supervised optimisation on the chronological training partition and evaluates the resulting model on the later validation partition after each epoch.

Training loss and validation metrics, including accuracy, precision, recall, F1-score, ROC-AUC and average precision, are recorded alongside the time required to complete each epoch.

Model selection is based on validation ROC-AUC. Whenever an epoch achieves a higher validation ROC-AUC than previous epochs, a copy of its model parameters is retained in memory.

This approach allows the best-performing model checkpoint to be recovered rather than automatically selecting the final training epoch.

Validation labels are used for evaluation and model selection but not for gradient-based parameter updates. The chronological test partition remains untouched.

The training procedure was defined without executing the full experiment. A runtime benchmark will be conducted before full-scale training to assess computational feasibility on the available CPU environment.

In [72]:

# Step 70: Benchmark GraphSAGE training and validation runtime

import time
import math
import numpy as np
import torch

torch.manual_seed(42)

# Reuse the diagnostic subsets from Step 68
benchmark_train_ids = diagnostic_train_ids
benchmark_validation_ids = diagnostic_validation_ids

# Create a disposable benchmark model
benchmark_model = ProsperGraphSAGE(
    hidden_channels=GNN_HIDDEN_CHANNELS
).to(gnn_device)

# Initialise lazy parameters using training data
benchmark_model.eval()

with torch.no_grad():
    _ = benchmark_model(
        mixed_train_batch.to(gnn_device)
    )

benchmark_optimizer = torch.optim.Adam(
    benchmark_model.parameters(),
    lr=GNN_LEARNING_RATE
)

# Measure one training epoch
train_start = time.perf_counter()

benchmark_train_results = train_graphsage_epoch(
    model=benchmark_model,
    optimizer=benchmark_optimizer,
    event_ids=benchmark_train_ids,
    batch_size=GNN_BATCH_SIZE,
    seed=42
)

train_seconds = time.perf_counter() - train_start

# Measure validation
validation_start = time.perf_counter()

benchmark_val_results = evaluate_graphsage(
    model=benchmark_model,
    event_ids=benchmark_validation_ids,
    batch_size=GNN_BATCH_SIZE
)

validation_seconds = time.perf_counter() - validation_start

# Estimate full-dataset runtime
train_seconds_per_event = (
    train_seconds / len(benchmark_train_ids)
)

validation_seconds_per_event = (
    validation_seconds / len(benchmark_validation_ids)
)

estimated_full_train_seconds = (
    train_seconds_per_event * len(train_event_ids)
)

estimated_full_validation_seconds = (
    validation_seconds_per_event * len(validation_event_ids)
)

estimated_epoch_seconds = (
    estimated_full_train_seconds
    + estimated_full_validation_seconds
)

estimated_total_seconds = (
    estimated_epoch_seconds * GNN_MAX_EPOCHS
)

print("GRAPHSAGE CPU RUNTIME BENCHMARK")
print("-------------------------------")
print("Training events benchmarked:", len(benchmark_train_ids))
print("Validation events benchmarked:", len(benchmark_validation_ids))
print(f"Training time: {train_seconds:.2f} seconds")
print(f"Validation time: {validation_seconds:.2f} seconds")

print("\nEstimated full training runtime:")
print(
    f"One epoch: {estimated_epoch_seconds / 60:.2f} minutes"
)
print(
    f"{GNN_MAX_EPOCHS} epochs: "
    f"{estimated_total_seconds / 60:.2f} minutes"
)

print("\nBenchmark model validation ROC-AUC:",
      round(benchmark_val_results["roc_auc"], 4))

assert np.isfinite(train_seconds_per_event)
assert np.isfinite(validation_seconds_per_event)
assert estimated_epoch_seconds > 0

print("\nGraphSAGE runtime benchmark completed!")
print("The full training model remains untouched.")


GRAPHSAGE CPU RUNTIME BENCHMARK
-------------------------------
Training events benchmarked: 128
Validation events benchmarked: 64
Training time: 1.49 seconds
Validation time: 0.75 seconds

Estimated full training runtime:
One epoch: 4.23 minutes
10 epochs: 42.34 minutes

Benchmark model validation ROC-AUC: 0.5719

GraphSAGE runtime benchmark completed!
The full training model remains untouched.


### 3.39 CPU Runtime Benchmark for GraphSAGE Training

A preliminary runtime benchmark was designed to assess the computational feasibility of training the heterogeneous GraphSAGE model on the available CPU environment.

The benchmark uses 128 training events and 64 validation events from the previously established chronological partitions.

A separate disposable GraphSAGE model is initialised to prevent the benchmark from modifying the model reserved for the full experiment.

The execution time of one diagnostic training epoch and a validation pass is measured using a high-resolution timer.

Average processing times per event are then used to estimate the duration of one full training and validation epoch, as well as the planned ten-epoch experiment.

The estimates include historical neighbourhood construction, mini-batch preparation, model computation and optimisation.

Because the benchmark uses a limited subset of events, the resulting runtime projections are approximate. Actual runtime may differ because borrowing events vary in neighbourhood size and computational complexity.

The benchmark is intended to guide subsequent decisions about batching, sampling and computational optimisation before full-scale GraphSAGE training.

In [73]:

# Step 71: Prepare persistent GraphSAGE checkpoint storage

from pathlib import Path
import json
import torch

# Locate repository root from the current notebook
current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

# Create model and results directories
checkpoint_directory = project_root / "models" / "graphsage"
results_directory = project_root / "results" / "graphsage"

checkpoint_directory.mkdir(parents=True, exist_ok=True)
results_directory.mkdir(parents=True, exist_ok=True)

# Define future output paths
best_model_path = checkpoint_directory / "best_graphsage_model.pt"
training_history_path = results_directory / "training_history.csv"
training_metadata_path = results_directory / "training_metadata.json"

# Save configuration metadata
training_metadata = {
    "model": "Heterogeneous GraphSAGE",
    "dataset": "Prosper lender-borrower network",
    "hidden_channels": GNN_HIDDEN_CHANNELS,
    "learning_rate": GNN_LEARNING_RATE,
    "batch_size": GNN_BATCH_SIZE,
    "planned_epochs": GNN_MAX_EPOCHS,
    "random_seed": RANDOM_SEED,
    "training_events": len(train_event_ids),
    "validation_events": len(validation_event_ids),
    "test_events": len(test_event_ids),
    "selection_metric": "validation_roc_auc",
    "checkpoint_status": "training_not_started"
}

with open(training_metadata_path, "w", encoding="utf-8") as file:
    json.dump(training_metadata, file, indent=4)

# Confirm metadata was written successfully
with open(training_metadata_path, "r", encoding="utf-8") as file:
    saved_metadata = json.load(file)

print("GRAPHSAGE CHECKPOINT STORAGE")
print("----------------------------")
print("Project root:", project_root)
print("Checkpoint directory:", checkpoint_directory)
print("Results directory:", results_directory)

print("\nPlanned output files:")
print("Best model:", best_model_path.name)
print("Training history:", training_history_path.name)
print("Training metadata:", training_metadata_path.name)

print("\nMetadata saved:", training_metadata_path.exists())
print("Saved model:", saved_metadata["model"])
print("Checkpoint status:", saved_metadata["checkpoint_status"])

assert checkpoint_directory.exists()
assert results_directory.exists()
assert training_metadata_path.exists()
assert saved_metadata["training_events"] == 17361

print("\nPersistent checkpoint storage prepared successfully!")
print("Full GraphSAGE training has not started.")


GRAPHSAGE CHECKPOINT STORAGE
----------------------------
Project root: c:\GroupWork\DS_research_project-maria_kisakye-
Checkpoint directory: c:\GroupWork\DS_research_project-maria_kisakye-\models\graphsage
Results directory: c:\GroupWork\DS_research_project-maria_kisakye-\results\graphsage

Planned output files:
Best model: best_graphsage_model.pt
Training history: training_history.csv
Training metadata: training_metadata.json

Metadata saved: True
Saved model: Heterogeneous GraphSAGE
Checkpoint status: training_not_started

Persistent checkpoint storage prepared successfully!
Full GraphSAGE training has not started.


### 3.40 Persistent Storage for GraphSAGE Model Checkpoints

The preliminary CPU benchmark estimated that ten epochs of GraphSAGE training would require approximately 42.34 minutes under the tested configuration.

To support reproducibility and reduce the risk of losing experimental results, persistent storage directories were prepared within the research repository.

A dedicated model directory was created for saving the best-performing GraphSAGE checkpoint, while a separate results directory was established for training histories and experiment metadata.

The metadata file records the model architecture, dataset, training configuration, chronological partition sizes, random seed and model-selection criterion.

The best model checkpoint will be selected using validation ROC-AUC. Training metrics will be recorded separately for subsequent analysis and visualisation.

Preparing these directories does not execute model training. Persistent checkpoint writing will be integrated into the training procedure before the full experiment begins.

In [74]:

# Step 72: Add persistent checkpoint saving to GraphSAGE training

import copy
import json
import time
import numpy as np
import pandas as pd
import torch

def run_graphsage_training(
    model,
    optimizer,
    train_ids,
    validation_ids,
    epochs=GNN_MAX_EPOCHS,
    batch_size=GNN_BATCH_SIZE,
    seed=42
):
    """
    Train GraphSAGE and persist results after every epoch.

    Saves:
      - Best model checkpoint by validation ROC-AUC
      - Training history CSV after each completed epoch
      - Updated experiment metadata
    """

    history = []
    best_validation_auc = -np.inf
    best_epoch = None
    best_model_state = None

    for epoch in range(1, epochs + 1):

        start_time = time.perf_counter()

        train_results = train_graphsage_epoch(
            model=model,
            optimizer=optimizer,
            event_ids=train_ids,
            batch_size=batch_size,
            seed=seed + epoch
        )

        val_results = evaluate_graphsage(
            model=model,
            event_ids=validation_ids,
            batch_size=batch_size,
            threshold=0.5
        )

        epoch_seconds = time.perf_counter() - start_time

        row = {
            "epoch": epoch,
            "train_loss": train_results["average_loss"],
            "val_loss": val_results["loss"],
            "val_accuracy": val_results["accuracy"],
            "val_precision": val_results["precision"],
            "val_recall": val_results["recall"],
            "val_f1": val_results["f1"],
            "val_roc_auc": val_results["roc_auc"],
            "val_average_precision": val_results["average_precision"],
            "epoch_seconds": epoch_seconds
        }

        history.append(row)

        # Save history after each completed epoch
        history_df = pd.DataFrame(history)
        history_df.to_csv(training_history_path, index=False)

        current_auc = val_results["roc_auc"]

        if np.isfinite(current_auc) and current_auc > best_validation_auc:

            best_validation_auc = float(current_auc)
            best_epoch = epoch

            best_model_state = copy.deepcopy(
                model.state_dict()
            )

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": best_model_state,
                    "validation_roc_auc": best_validation_auc,
                    "hidden_channels": GNN_HIDDEN_CHANNELS,
                    "random_seed": seed
                },
                best_model_path
            )

        # Update metadata after every epoch
        updated_metadata = {
            **training_metadata,
            "checkpoint_status": "in_progress",
            "completed_epochs": epoch,
            "best_epoch": best_epoch,
            "best_validation_roc_auc": (
                best_validation_auc
                if np.isfinite(best_validation_auc)
                else None
            )
        }

        with open(
            training_metadata_path,
            "w",
            encoding="utf-8"
        ) as file:
            json.dump(updated_metadata, file, indent=4)

        print(
            f"Epoch {epoch}/{epochs} | "
            f"Train Loss: {row['train_loss']:.4f} | "
            f"Val Loss: {row['val_loss']:.4f} | "
            f"Val AUC: {row['val_roc_auc']:.4f} | "
            f"Val AP: {row['val_average_precision']:.4f} | "
            f"Time: {epoch_seconds:.1f}s",
            flush=True
        )

    # Mark experiment complete
    updated_metadata["checkpoint_status"] = "completed"

    with open(
        training_metadata_path,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(updated_metadata, file, indent=4)

    return {
        "history": pd.DataFrame(history),
        "best_epoch": best_epoch,
        "best_validation_auc": best_validation_auc,
        "best_model_state": best_model_state
    }


print("PERSISTENT GRAPHSAGE TRAINING FUNCTION")
print("--------------------------------------")
print("Function defined:", callable(run_graphsage_training))
print("Best model path:", best_model_path)
print("Training history path:", training_history_path)
print("Metadata path:", training_metadata_path)

assert callable(run_graphsage_training)
assert checkpoint_directory.exists()
assert results_directory.exists()

print("\nPersistent training function ready!")
print("No training has been executed.")


PERSISTENT GRAPHSAGE TRAINING FUNCTION
--------------------------------------
Function defined: True
Best model path: c:\GroupWork\DS_research_project-maria_kisakye-\models\graphsage\best_graphsage_model.pt
Training history path: c:\GroupWork\DS_research_project-maria_kisakye-\results\graphsage\training_history.csv
Metadata path: c:\GroupWork\DS_research_project-maria_kisakye-\results\graphsage\training_metadata.json

Persistent training function ready!
No training has been executed.


### 3.41 Persistent GraphSAGE Checkpointing and Training History

The multi-epoch GraphSAGE training procedure was extended to support persistent storage of experimental results.

After each completed epoch, the procedure records training loss, validation metrics and execution time in a CSV file.

Model selection is based on validation ROC-AUC. Whenever an epoch achieves a higher validation ROC-AUC than previous epochs, its model parameters are saved as the best-performing checkpoint.

The experiment metadata is also updated after each epoch to record the number of completed epochs, the best-performing epoch and the corresponding validation ROC-AUC.

This approach reduces the risk of losing completed experimental results if the notebook session is interrupted.

The checkpoint stores the best model parameters for subsequent evaluation. It does not currently provide automatic continuation of an interrupted optimisation run.

The revised procedure was defined without executing training. Checkpoint creation and file-writing behaviour will be verified before the full GraphSAGE experiment begins.

In [75]:

# Step 73: Verify persistent GraphSAGE checkpoint saving

import torch
import json
import pandas as pd
from pathlib import Path

torch.manual_seed(42)

# Preserve the production file paths
production_model_path = best_model_path
production_history_path = training_history_path
production_metadata_path = training_metadata_path

# Temporary diagnostic paths
diagnostic_directory = results_directory / "checkpoint_test"
diagnostic_directory.mkdir(parents=True, exist_ok=True)

diagnostic_model_path = diagnostic_directory / "test_model.pt"
diagnostic_history_path = diagnostic_directory / "test_history.csv"
diagnostic_metadata_path = diagnostic_directory / "test_metadata.json"

# Prepare diagnostic metadata
diagnostic_metadata = {
    **training_metadata,
    "experiment_type": "checkpoint_diagnostic",
    "training_events": 128,
    "validation_events": 64,
    "planned_epochs": 1
}

# Temporarily redirect paths used by the training function
best_model_path = diagnostic_model_path
training_history_path = diagnostic_history_path
training_metadata_path = diagnostic_metadata_path
training_metadata = diagnostic_metadata

try:
    # Fresh disposable model
    checkpoint_test_model = ProsperGraphSAGE(
        hidden_channels=GNN_HIDDEN_CHANNELS
    ).to(gnn_device)

    checkpoint_test_model.eval()

    with torch.no_grad():
        _ = checkpoint_test_model(
            mixed_train_batch.to(gnn_device)
        )

    checkpoint_test_optimizer = torch.optim.Adam(
        checkpoint_test_model.parameters(),
        lr=GNN_LEARNING_RATE
    )

    # Run one diagnostic epoch
    checkpoint_test_results = run_graphsage_training(
        model=checkpoint_test_model,
        optimizer=checkpoint_test_optimizer,
        train_ids=diagnostic_train_ids,
        validation_ids=diagnostic_validation_ids,
        epochs=1,
        batch_size=GNN_BATCH_SIZE,
        seed=42
    )

    # Verify all output files exist
    assert diagnostic_model_path.exists()
    assert diagnostic_history_path.exists()
    assert diagnostic_metadata_path.exists()

    # Verify model checkpoint can be loaded
    saved_checkpoint = torch.load(
        diagnostic_model_path,
        map_location="cpu",
        weights_only=True
    )

    saved_history = pd.read_csv(diagnostic_history_path)

    with open(diagnostic_metadata_path, "r", encoding="utf-8") as file:
        saved_diagnostic_metadata = json.load(file)

    assert saved_checkpoint["epoch"] == 1
    assert len(saved_history) == 1
    assert saved_diagnostic_metadata["checkpoint_status"] == "completed"
    assert saved_diagnostic_metadata["completed_epochs"] == 1

    print("\nGRAPHSAGE CHECKPOINT VERIFICATION")
    print("--------------------------------")
    print("Model checkpoint saved:", diagnostic_model_path.exists())
    print("Training history saved:", diagnostic_history_path.exists())
    print("Metadata saved:", diagnostic_metadata_path.exists())

    print("\nSaved checkpoint:")
    print("Epoch:", saved_checkpoint["epoch"])
    print("Validation ROC-AUC:",
          round(saved_checkpoint["validation_roc_auc"], 6))

    print("\nSaved training history:")
    print(saved_history[[
        "epoch",
        "train_loss",
        "val_loss",
        "val_roc_auc"
    ]].to_string(index=False))

    print("\nMetadata status:",
          saved_diagnostic_metadata["checkpoint_status"])

    print("\nPersistent checkpoint verification successful!")

finally:
    # Restore production paths and metadata
    best_model_path = production_model_path
    training_history_path = production_history_path
    training_metadata_path = production_metadata_path

    training_metadata = {
        "model": "Heterogeneous GraphSAGE",
        "dataset": "Prosper lender-borrower network",
        "hidden_channels": GNN_HIDDEN_CHANNELS,
        "learning_rate": GNN_LEARNING_RATE,
        "batch_size": GNN_BATCH_SIZE,
        "planned_epochs": GNN_MAX_EPOCHS,
        "random_seed": RANDOM_SEED,
        "training_events": len(train_event_ids),
        "validation_events": len(validation_event_ids),
        "test_events": len(test_event_ids),
        "selection_metric": "validation_roc_auc",
        "checkpoint_status": "training_not_started"
    }

print("\nProduction checkpoint paths restored.")
print("Full GraphSAGE training model remains untouched.")


Epoch 1/1 | Train Loss: 0.6936 | Val Loss: 0.6671 | Val AUC: 0.5750 | Val AP: 0.4910 | Time: 3.6s

GRAPHSAGE CHECKPOINT VERIFICATION
--------------------------------
Model checkpoint saved: True
Training history saved: True
Metadata saved: True

Saved checkpoint:
Epoch: 1
Validation ROC-AUC: 0.575

Saved training history:
 epoch  train_loss  val_loss  val_roc_auc
     1    0.693608   0.66713        0.575

Metadata status: completed

Persistent checkpoint verification successful!

Production checkpoint paths restored.
Full GraphSAGE training model remains untouched.


### 3.42 Verification of Persistent GraphSAGE Checkpointing

A small-scale diagnostic experiment was designed to verify the persistent checkpointing mechanism before full GraphSAGE training.

The experiment uses 128 training events and 64 validation events, processed over one training epoch.

A separate GraphSAGE model is initialised for this diagnostic. Temporary output paths are used to prevent the test from overwriting files reserved for the full experiment.

The procedure verifies that the best model checkpoint is saved successfully, that the training history contains the completed epoch, and that the experiment metadata records the correct completion status.

The saved model checkpoint is also loaded from disk to confirm that the stored parameters and associated information can be recovered.

After the diagnostic, the original production file paths and training configuration are restored.

This verification supports reproducibility and reduces the risk of losing model results during the subsequent full-scale experiment. The diagnostic results are not treated as final model performance findings.

In [76]:

# Step 74: Full GraphSAGE training experiment

import json
import torch

# Confirm that production paths are active
assert best_model_path == production_model_path
assert training_history_path == production_history_path
assert training_metadata_path == production_metadata_path

# Prevent accidental overwriting of a previous full run
assert not best_model_path.exists(), (
    "A production model checkpoint already exists. "
    "Inspect it before starting another training run."
)

assert not training_history_path.exists(), (
    "A production training history already exists. "
    "Inspect it before starting another training run."
)

# Verify the intended chronological partitions
assert len(train_event_ids) == 17361
assert len(validation_event_ids) == 4379
assert len(test_event_ids) == 5545

assert set(train_event_ids).isdisjoint(validation_event_ids)
assert set(train_event_ids).isdisjoint(test_event_ids)
assert set(validation_event_ids).isdisjoint(test_event_ids)

print("FULL GRAPHSAGE TRAINING")
print("-----------------------")
print("Training events:", len(train_event_ids))
print("Validation events:", len(validation_event_ids))
print("Reserved test events:", len(test_event_ids))
print("Maximum epochs:", GNN_MAX_EPOCHS)
print("Batch size:", GNN_BATCH_SIZE)
print("Device:", gnn_device)

print("\nStarting full training...\n", flush=True)

full_gnn_results = run_graphsage_training(
    model=training_gnn_model,
    optimizer=gnn_optimizer,
    train_ids=train_event_ids,
    validation_ids=validation_event_ids,
    epochs=GNN_MAX_EPOCHS,
    batch_size=GNN_BATCH_SIZE,
    seed=RANDOM_SEED
)

print("\nFULL GRAPHSAGE TRAINING COMPLETED")
print("---------------------------------")
print("Best epoch:", full_gnn_results["best_epoch"])
print(
    "Best validation ROC-AUC:",
    round(full_gnn_results["best_validation_auc"], 6)
)

print("\nTraining history:")
print(
    full_gnn_results["history"][
        [
            "epoch",
            "train_loss",
            "val_loss",
            "val_roc_auc",
            "val_average_precision",
            "epoch_seconds"
        ]
    ].to_string(index=False)
)

print("\nCheckpoint saved:", best_model_path.exists())
print("History saved:", training_history_path.exists())

assert best_model_path.exists()
assert training_history_path.exists()

print("\nGraphSAGE full training successful!")


FULL GRAPHSAGE TRAINING
-----------------------
Training events: 17361
Validation events: 4379
Reserved test events: 5545
Maximum epochs: 10
Batch size: 16
Device: cpu

Starting full training...

Epoch 1/10 | Train Loss: 0.6053 | Val Loss: 0.6005 | Val AUC: 0.6666 | Val AP: 0.4801 | Time: 230.2s
Epoch 2/10 | Train Loss: 0.5895 | Val Loss: 0.5974 | Val AUC: 0.6684 | Val AP: 0.4845 | Time: 246.9s
Epoch 3/10 | Train Loss: 0.5865 | Val Loss: 0.6026 | Val AUC: 0.6717 | Val AP: 0.4873 | Time: 208.6s
Epoch 4/10 | Train Loss: 0.5858 | Val Loss: 0.5993 | Val AUC: 0.6716 | Val AP: 0.4881 | Time: 198.1s
Epoch 5/10 | Train Loss: 0.5844 | Val Loss: 0.6052 | Val AUC: 0.6745 | Val AP: 0.4881 | Time: 241.8s
Epoch 6/10 | Train Loss: 0.5837 | Val Loss: 0.5939 | Val AUC: 0.6754 | Val AP: 0.4924 | Time: 275.3s
Epoch 7/10 | Train Loss: 0.5827 | Val Loss: 0.5998 | Val AUC: 0.6762 | Val AP: 0.4922 | Time: 209.2s
Epoch 8/10 | Train Loss: 0.5817 | Val Loss: 0.5986 | Val AUC: 0.6762 | Val AP: 0.4912 | Time: 232